# Set Up 

In [ ]:
%%bash
pip install jupysql
pip install duckdb-engine


In [82]:

import duckdb
import pandas as pd
import gc
import numpy as np
import lightgbm as lgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
import optuna
from sklearn.metrics import roc_auc_score

from pathlib import Path



%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb:///

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [2]:
dataset_path = (
    "/Users/manuelmarasco/Library/CloudStorage/"
    "GoogleDrive-hola@manuelmarasco.com/"
    "Mi unidad/DMEyF/2026/notebooks/data/"
)

dataset_filename = "competencia_01_crudo.csv"

In [12]:
%%sql

CREATE OR REPLACE TABLE competencia_01_crudo AS

SELECT *
FROM read_csv_auto(
    '{{dataset_path + dataset_filename}}',
    sample_size = -1
);

,Success


In [13]:
%%sql

SELECT *
FROM competencia_01_crudo
LIMIT 5;

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo
0,12159854.0,202103,1,0,0,56,134,688.41,26701.84,98.82,...,0.0,1,0.0,-16247.77,0.0,4056,15732.34,1,0,1137.81
1,12159858.0,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN
2,12160484.0,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,0.0,1,0.0,-31103.23,0.0,1632,2860.54,2,0,19858.89
3,12160591.0,202103,1,0,0,46,275,14825.78,138050.05,1146.27,...,0.0,1,0.0,-13733.44,0.0,2122,1419.36,3,0,1231.65
4,12160747.0,202103,1,0,0,47,194,2015.61,31240.49,1791.25,...,0.0,1,0.0,0.00,0.0,5901,1286.93,1,0,82.11


# Target

In [14]:
%%sql
CREATE OR REPLACE TABLE competencia_01 AS

WITH periodos AS (
    SELECT DISTINCT foto_mes FROM competencia_01_crudo
),

clientes AS (
    SELECT DISTINCT numero_de_cliente FROM competencia_01_crudo
),

todo AS (
    SELECT numero_de_cliente, foto_mes
    FROM clientes CROSS JOIN periodos
),

presencia AS (
    SELECT
        c.* EXCLUDE (numero_de_cliente, foto_mes),
        t.numero_de_cliente,
        t.foto_mes,
        CASE WHEN c.numero_de_cliente IS NULL THEN 0 ELSE 1 END AS mes_0
    FROM todo AS t
    LEFT JOIN competencia_01_crudo AS c
        ON  c.numero_de_cliente = t.numero_de_cliente
        AND c.foto_mes          = t.foto_mes
),

etiquetado AS (
    SELECT
        *,
        LEAD(mes_0, 1) OVER w AS m1,
        LEAD(mes_0, 2) OVER w AS m2
    FROM presencia
    WINDOW w AS (PARTITION BY numero_de_cliente ORDER BY foto_mes)
)

SELECT
    * EXCLUDE (mes_0, m1, m2),
    CASE
        WHEN m1 = 0            THEN 'BAJA+1'
        WHEN m1 = 1 AND m2 = 0 THEN 'BAJA+2'
        WHEN m1 = 1 AND m2 = 1 THEN 'CONTINUA'
    END AS clase_ternaria
FROM etiquetado
WHERE mes_0 = 1
ORDER BY foto_mes, numero_de_cliente;

,Success


In [15]:
%%sql
PIVOT competencia_01
on clase_ternaria
USING count(numero_de_cliente)
GROUP BY foto_mes

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,202103,1019,960,160921
1,202104,964,1139,161181
2,202105,1143,870,161755
3,202106,874,1098,162142
4,202107,1103,0,0
5,202108,0,0,0


In [54]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+2'
    ) AS bajas_2,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+1'
    ) AS bajas_1,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'CONTINUA'
    ) AS continuas,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+2'
        )
        / COUNT(*),
        4
    ) AS porcentaje_baja_2,

    -- Máximo teórico: encontrar todas las BAJA+2
    -- sin enviar estímulos a ningún otro cliente.
    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+2'
    ) * 1072500 AS ganancia_maxima_teorica

FROM competencia_01

WHERE foto_mes BETWEEN 202103 AND 202106

GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,bajas_2,bajas_1,continuas,porcentaje_baja_2,ganancia_maxima_teorica
0,202103,162900,960,1019,160921,0.5893,1029600000
1,202104,163284,1139,964,161181,0.6976,1221577500
2,202105,163768,870,1143,161755,0.5312,933075000
3,202106,164114,1098,874,162142,0.6690,1177605000


# Macro para ceros y nulos

In [16]:
%%sql
CREATE OR REPLACE MACRO ratio_seguro(numerador, denominador) AS
    ifnull(numerador, 0)
    / nullif(ifnull(denominador, 0), 0);

,Success


# Guía de Features  

| Concepto | Construcción | Por qué importa | Columnas fuente |
|---|---|---|---|
| Presencia de payroll | `cpayroll_trx > 0 OR mpayroll > 0` | Es uno de los principales anclajes con el banco. | `cpayroll_trx`, `mpayroll` |
| Payroll de referencia | Promedio del `mpayroll` positivo actual y del mes anterior. | Reduce ruido por acreditaciones ocasionales y por el salto de junio. | `mpayroll`, su lag1 |
| Apagó payroll | Mes anterior con payroll y actual sin payroll. | En el perfil nómina, 33,3% lo apagó antes de irse. | `cpayroll_trx`, `mpayroll`, lag1 |
| Compromisos domiciliados | Suma de débitos en cuenta y tarjetas, pago de servicios y PagoMisCuentas. | Mide costos de mudarse a otro banco. Su caída representa migración. | `ccuenta_debitos_automaticos`, `ctarjeta_visa_debitos_automaticos`, `ctarjeta_master_debitos_automaticos`, `cpagodeservicios`, `cpagomiscuentas` |
| Marcas de tarjeta | Cantidad de Visa/Master tenidas y cantidad efectivamente usadas. | Diferencia “no tiene”, “tiene pero no usa” y “usa”. | `ctarjeta_visa`, `ctarjeta_master`, `ctarjeta_visa_transacciones`, `ctarjeta_master_transacciones` |
| Tarjeta sin uso | Tiene la marca, pero sus transacciones son cero. | En el EDA, perder tenencia o uso resultó más informativo que cambiar el límite. | Las cuatro anteriores |
| Canales utilizados | Cantidad de canales con operaciones: homebanking, app, ATM, caja y call center. | La pérdida de canales distingue la desvinculación avanzada. | `chomebanking_transacciones`, `cmobile_app_trx`, `catm_trx`, `catm_trx_other`, `ccajas_transacciones`, `ccallcenter_transacciones` |
| Familias activas | Cantidad de familias presentes: cuentas, tarjetas, créditos, inversiones, seguros, payroll, recurrentes y canales digitales. | Miranda encontró medianas de 4, 3, 2 y 1 familias según perfil. | Todos los `c*` relevantes y los flags anteriores |
| Actividad por producto | `ctrx_quarter / cproductos` | Distingue tener productos de utilizarlos realmente. | `ctrx_quarter`, `cproductos` |
| Saldo negativo | `mcuentas_saldo < 0` | Los saldos negativos pasan de 22% en nómina a 89% en desvinculación avanzada. | `mcuentas_saldo` |
| Descubierto utilizado | `GREATEST(-mcuenta_corriente, 0)` | Aproxima estrés de liquidez. No es “utilización del límite”. | `mcuenta_corriente` |
| Cliente anclado | Payroll + algún compromiso recurrente + alguna tarjeta. | Resume una relación difícil de trasladar. | Payroll, recurrentes y tenencia de tarjetas |
| Caída de actividad | Delta de `ctrx_quarter` y flag de caída de al menos 20%. | El multicanal todavía está activo cuando comienza a contraerse. | `ctrx_quarter`, lag1 |
| Cruce a saldo negativo | Saldo anterior no negativo y saldo actual negativo. | Captura deterioro, no solamente el nivel terminal. | `mcuentas_saldo`, lag1 |
| Apagó una familia | Valor anterior positivo y actual cero. | Es central para payroll, recurrentes, tarjeta, digital, crédito, inversión y seguros. | Columnas de cada familia y sus lag1 |

| Concepto | ¿Se opera para obtener el numerador? | Numerador resultante | Operación final |
|---|---|---|---|
| Carga de comisiones | No | `mcomisiones` | `mcomisiones / payroll` |
| Costo de mantenimiento | No | `mcomisiones_mantenimiento` | `mcomisiones_mantenimiento / payroll` |
| Deuda de tarjetas en pesos | Sí | Parte positiva de Visa + Master | `deuda_tarjetas_pesos / payroll` |
| Pago mínimo requerido | Sí | Suma segura Visa + Master | `pago_minimo_tarjetas / payroll` |
| Préstamos totales | Sí | Suma de los tres préstamos | `prestamos_totales / payroll` |
| Descubierto utilizado | Sí | Convierte saldo negativo en deuda positiva | `descubierto_utilizado / payroll` |
| Deuda observada total | Sí | Tarjetas + préstamos + descubierto | `deuda_total_pesos / payroll` |
| Posición de saldo | No | `mcuentas_saldo`, conservando su signo | `mcuentas_saldo / payroll` |

| Concepto | Fórmula o información | Importancia | Columnas |
|---|---|---|---|
| Saldo total en pesos | Suma Visa + Master, preservando all-NULL. | Exposición financiada. | `Visa_msaldopesos`, `Master_msaldopesos` |
| Saldo total en dólares | Suma Visa + Master, mantenida separada de pesos. | Exposición en moneda extranjera. | `Visa_msaldodolares`, `Master_msaldodolares` |
| Límite total de compra | Suma de límites Visa + Master. | Capacidad disponible. | `Visa_mlimitecompra`, `Master_mlimitecompra` |
| Utilización en pesos | `saldo_tarjetas_pesos / limite_compra_total` | Indicador clásico de tensión crediticia. | Saldos en pesos y límites |
| Holgura disponible | `limite_compra_total - saldo_tarjetas_pesos` | Crédito todavía disponible. | Mismas columnas |
| Consumo sobre límite | Consumos en pesos Visa+Master / límite. | Intensidad de gasto respecto de capacidad. | `Visa_mconsumospesos`, `Master_mconsumospesos`, límites |
| Ticket operativo | `mtarjeta_*_consumo / ctarjeta_*_transacciones` | Cambio en tamaño medio de las compras. | Par Visa y par Master por separado |
| Pago sobre saldo | Pagos en pesos / saldo positivo en pesos. | Aproxima cancelación versus financiación. | `Visa_mpagospesos`, `Master_mpagospesos`, saldos |
| Pago sobre mínimo | Pagos en pesos / pago mínimo. | Diferencia pago mínimo de pago más holgado. | Pagos y `*_mpagominimo` |
| Mínimo sobre saldo | Pago mínimo / saldo positivo. | Exigencia de corto plazo. | `*_mpagominimo`, `*_msaldopesos` |
| Adelantos sobre límite | Adelantos en pesos / límite. | Uso de efectivo con tarjeta, posible señal de liquidez. | `*_madelantopesos`, `*_mlimitecompra` |
| Adelantos sobre consumo | Adelantos en pesos / consumo en pesos. | Distingue consumo ordinario de necesidad de efectivo. | `*_madelantopesos`, `*_mconsumospesos` |
| Share Visa | Visa / total Visa+Master en transacciones, consumo o saldo. | Identifica marca principal y migraciones entre marcas. | Pares Visa/Master del mismo concepto |
| Uso de débitos automáticos | Cantidad/monto de débitos de cada tarjeta y su caída. | Parte del anclaje operativo. | `ctarjeta_*_debitos_automaticos`, `mttarjeta_*_debitos_automaticos` |
| Uso de descuentos | Descuentos/cantidad de compras o descuentos/consumo. | Sensibilidad a beneficios. Baja prioridad. | `ctarjeta_*_descuentos`, `mtarjeta_*_descuentos` |
| Mora consolidada | Máxima mora entre Visa y Master; flag de empeoramiento. | Fricción y riesgo crediticio. | `Visa_delinquency`, `Master_delinquency`, lag1 |
| Estado consolidado | Flags por código, no promedio ni suma. | Los estados son categorías, no magnitudes. | `Visa_status`, `Master_status` |
| Ausencia de resumen | Flags de campos de consumo/pago faltantes con tarjeta vigente. | El faltante puede tener información propia. | Límites, tenencia y campos `Visa_*`/`Master_*` nulos |
| Vida y vencimiento | Días desde alta, hasta vencimiento y desde último cierre. | Posibles eventos administrativos. | `*_fechaalta`, `*_Fvencimiento`, `*_fultimo_cierre` |

| Concepto | Construcción | Por qué importa | Columnas |
|---|---|---|---|
| Saldo promedio por cuenta | `mcuentas_saldo / (ccuenta_corriente + ccaja_ahorro)` | Profundidad monetaria por cuenta. | Saldos y cantidades de cuenta |
| Composición cuenta/ahorro | Parte positiva de cuenta corriente versus caja de ahorro, sólo si tienen misma moneda. | Forma de mantener liquidez. | `mcuenta_corriente`, `mcaja_ahorro` |
| Cantidad de créditos | Suma de préstamos personales, prendarios e hipotecarios. | El crédito puede funcionar como producto ancla. | `cprestamos_personales`, `cprestamos_prendarios`, `cprestamos_hipotecarios` |
| Saldo total de préstamos | Suma de los tres montos. | Exposición crediticia total. | Sus respectivos `mprestamos_*` |
| Saldo promedio por crédito | Préstamos totales / cantidad de créditos. | Tamaño medio de exposición. | Cantidades y montos anteriores |
| Composición del crédito | Personal, prendario e hipotecario / préstamos totales. | Distingue crédito de consumo de crédito garantizado. | `mprestamos_*` |
| Crédito sobre payroll | Préstamos totales / payroll de referencia. | Esfuerzo relativo al ingreso observado. | Préstamos y payroll |
| Concentración de deuda | Tarjetas, préstamos o descubierto / deuda total. | Identifica la fuente del endeudamiento. | Los tres componentes |
| Inversiones en pesos | Mantener juntos sólo campos confirmados en pesos. | Ahorro e inversión local. | `mplazo_fijo_pesos`, `minversion1_pesos`; revisar unidad de `minversion2` |
| Inversiones en dólares | Total separado en dólares. | Tenencia en moneda extranjera. | `mplazo_fijo_dolares`, `minversion1_dolares` |
| Composición de inversiones | Plazo fijo versus inversión, dentro de la misma moneda. | Preferencia de liquidez y plazo. | Campos anteriores |
| Diversificación | Cantidad de tipos de inversión presentes. | Profundidad de la relación inversora. | `cplazo_fijo`, `cinversion1`, `cinversion2` |
| Desinversión | Tenía inversión y pasa a cero. | Puede anticipar salida de fondos. | Cantidades/montos y lag1 |
| Cobertura de seguros | Cantidad y variedad de seguros. | Cross-selling y costo de abandonar el banco. | `cseguro_vida`, `cseguro_auto`, `cseguro_vivienda`, `cseguro_accidentes_personales` |
| Caja de seguridad | Flag de tenencia y apagado. | Otro vínculo no transaccional. | `ccaja_seguridad` |

| Concepto | Construcción | Columnas |
|---|---|---|
| Ingresos observados | Payroll + transferencias recibidas + cheques depositados. | `mpayroll`, `mtransferencias_recibidas`, `mcheques_depositados` |
| Egresos observados | Transferencias emitidas + extracciones + cheques emitidos + pagos. | `mtransferencias_emitidas`, `mextraccion_autoservicio`, `matm`, `matm_other`, `mcheques_emitidos`, `mpagomiscuentas`, `mpagodeservicios` |
| Flujo observado neto | Ingresos observados − egresos observados. | Columnas anteriores |
| Velocidad de vaciado | Egresos observados / payroll de referencia. | Egresos y payroll |
| Egresos/ingresos | Egresos observados / ingresos observados. | Agregados anteriores |
| Retención de ingresos | `mcuentas_saldo / ingresos_observados`. | Saldo e ingresos |
| Balance de transferencias | Recibidas − emitidas. | `mtransferencias_recibidas`, `mtransferencias_emitidas` |
| Salida por transferencias | Emitidas / recibidas o emitidas / ingresos observados. | Transferencias |
| Ticket de transferencias | Monto/cantidad, recibido y emitido por separado. | `c/mtransferencias_recibidas`, `c/mtransferencias_emitidas` |
| ATM ajeno | `catm_trx_other / (catm_trx + catm_trx_other)` y equivalente en montos. | `catm_trx`, `catm_trx_other`, `matm`, `matm_other` |
| Actividad digital observada | Homebanking + app. | `chomebanking_transacciones`, `cmobile_app_trx` |
| Share digital | Digital / suma de canales mensuales observados. | Digital, ATM, caja y call center |
| Share app dentro de digital | App / (app + homebanking). | `cmobile_app_trx`, `chomebanking_transacciones` |
| Share sucursal | Caja / canales mensuales observados. | `ccajas_transacciones` y demás canales |
| Operaciones por canal activo | Total mensual de canales / cantidad de canales usados. | Conteos de canales |
| Migración a ATM ajeno | Baja ATM propio y sube ATM ajeno. | Deltas de `catm_trx`, `catm_trx_other` |
| Fricción de call center | Empieza a llamar después de no usar call center. | `ccallcenter_transacciones`, lag1 |
| Tasa de cheques rechazados | Rechazados / emitidos o depositados. | `ccheques_*`, `ccheques_*_rechazados` |
| Monto rechazado relativo | Monto rechazado / monto total correspondiente. | Pares monetarios de cheques |
| Flag de rechazo | Algún rechazo en el mes. | Conteos rechazados |

| Concepto | Construcción | Lectura |
|---|---|---|
| Monto recurrente total | Débitos de cuenta y tarjetas + servicios + PagoMisCuentas. | Compromiso económico domiciliado. |
| Ticket recurrente | Monto recurrente / cantidad recurrente. | Tamaño medio de compromisos. |
| Composición recurrente | Cada tipo / total recurrente. | Qué vínculo mantiene al cliente. |
| Comisión por evento | Monto de mantenimiento u otras / cantidad correspondiente. | Precio por cargo. |
| Comisión sobre payroll | `mcomisiones / payroll_ref`. | Peso de cargos respecto del ingreso. |
| Mantenimiento por producto | `mcomisiones_mantenimiento / cproductos`. | Costo aparente de mantener el paquete. |
| Rentabilidad por producto | `mrentabilidad / cproductos`. | Valor observado por profundidad de relación. |
| Desvío del run-rate | `mrentabilidad - mrentabilidad_annual / 12`. | Mes por encima o debajo del promedio anual registrado. |
| Margen activo sobre payroll | `mactivos_margen / payroll_ref`. | Intensidad del margen asociado a activos. |
| Margen pasivo sobre payroll | `mpasivos_margen / payroll_ref`. | Intensidad del margen asociado a pasivos. |
| Descuento promedio | Monto de descuento / cantidad, sólo cuando el monto está informado. | Valor del beneficio utilizado. |
| Share de compras con descuento | Cantidad de descuentos / transacciones de la misma tarjeta. | Dependencia de promociones. |
| Forex compra/venta | Cantidad de compras / total de operaciones forex. | Dirección de la actividad cambiaria. |
| Ticket forex | `mforex_buy/cforex_buy` y venta equivalente. | Tamaño de operación. |
| Balance forex | Compra − venta, sólo si ambos montos tienen la misma unidad. | Demanda neta observada. |

In [25]:
%%sql
SELECT
    numero_de_cliente,
    foto_mes,
    COUNT(*) AS cantidad
FROM competencia_01
GROUP BY numero_de_cliente, foto_mes
HAVING COUNT(*) > 1;

,numero_de_cliente,foto_mes,cantidad


In [26]:
%%sql
-- ============================================================
-- 1. MACROS AUXILIARES
-- ============================================================

-- Ignora NULL parciales, pero devuelve NULL si todos son NULL.
CREATE OR REPLACE MACRO suma_sin_null(valores) AS
    list_sum(valores);

-- En estas features el denominador tiene que ser positivo.
CREATE OR REPLACE MACRO ratio_seguro(numerador, denominador) AS (
    CASE
        WHEN denominador > 0
        THEN numerador::DOUBLE / denominador::DOUBLE
        ELSE NULL
    END
);


-- ============================================================
-- 2. CONSTRUCCIÓN DE FEATURES
-- ============================================================

CREATE OR REPLACE TABLE competencia_02_features AS

WITH base AS (
    SELECT
        *,

        -- Convierte YYYYMM en un índice mensual consecutivo.
        (foto_mes // 100) * 12
            + (foto_mes % 100) AS _indice_mes

    FROM competencia_01
),


-- ============================================================
-- COMPONENTES Y FLAGS BÁSICOS
-- ============================================================

componentes AS (
    SELECT
        *,

        -- Presencia de payroll.
        (
            COALESCE(cpayroll_trx, 0) > 0
            OR COALESCE(mpayroll, 0) > 0
        )::TINYINT AS tiene_payroll,

        -- Definición utilizada exactamente por el EDA de Miranda.
        (
            COALESCE(cpayroll_trx, 0) > 0
        )::TINYINT AS payroll_activo_miranda,

        -- Compromisos domiciliados.
        suma_sin_null([
            ccuenta_debitos_automaticos,
            ctarjeta_visa_debitos_automaticos,
            ctarjeta_master_debitos_automaticos,
            cpagodeservicios,
            cpagomiscuentas
        ]) AS coperaciones_recurrentes_total,

        -- Componentes separados para reproducir las familias de Miranda.
        suma_sin_null([
            ccuenta_debitos_automaticos,
            ctarjeta_visa_debitos_automaticos,
            ctarjeta_master_debitos_automaticos
        ]) AS debitos_automaticos_total,

        suma_sin_null([
            cpagodeservicios,
            cpagomiscuentas
        ]) AS pagos_servicios_total,

        -- Tenencia de tarjetas.
        (
            COALESCE(ctarjeta_visa, 0) > 0
        )::TINYINT AS tiene_visa,

        (
            COALESCE(ctarjeta_master, 0) > 0
        )::TINYINT AS tiene_master,

        -- Uso de tarjetas.
        (
            COALESCE(ctarjeta_visa_transacciones, 0) > 0
        )::TINYINT AS usa_visa,

        (
            COALESCE(ctarjeta_master_transacciones, 0) > 0
        )::TINYINT AS usa_master,

        -- No confundimos NULL con cero sin dejar una marca.
        (
            ctarjeta_visa_transacciones IS NULL
        )::TINYINT AS visa_uso_faltante,

        (
            ctarjeta_master_transacciones IS NULL
        )::TINYINT AS master_uso_faltante,

        -- Canales.
        (
            COALESCE(chomebanking_transacciones, 0) > 0
        )::TINYINT AS usa_homebanking_transaccional,

        (
            COALESCE(cmobile_app_trx, 0) > 0
        )::TINYINT AS usa_mobile_app_transaccional,

        -- ATM propio y ajeno representan un único canal.
        (
            COALESCE(catm_trx, 0)
            + COALESCE(catm_trx_other, 0) > 0
        )::TINYINT AS usa_atm,

        (
            COALESCE(ccajas_transacciones, 0) > 0
        )::TINYINT AS usa_cajas,

        (
            COALESCE(ccallcenter_transacciones, 0) > 0
        )::TINYINT AS usa_callcenter,

        -- Familia cuentas.
        (
            COALESCE(ccuenta_corriente, 0)
            + COALESCE(ccaja_ahorro, 0) > 0
        )::TINYINT AS tiene_cuentas,

        -- Familia créditos.
        suma_sin_null([
            cprestamos_personales,
            cprestamos_prendarios,
            cprestamos_hipotecarios
        ]) AS cantidad_creditos,

        -- Familia inversiones.
        suma_sin_null([
            cplazo_fijo,
            cinversion1,
            cinversion2,
            cforex
        ]) AS cantidad_inversiones,

        -- Familia seguros.
        suma_sin_null([
            cseguro_vida,
            cseguro_auto,
            cseguro_vivienda,
            cseguro_accidentes_personales
        ]) AS cantidad_seguros,

        (
            COALESCE(ctransferencias_emitidas, 0) > 0
        )::TINYINT AS emite_transferencias

    FROM base
),


-- ============================================================
-- AGREGADOS DE TARJETAS Y CANALES
-- ============================================================

agregados AS (
    SELECT
        *,

        tiene_visa
            + tiene_master
            AS cantidad_marcas_tarjeta,

        usa_visa
            + usa_master
            AS cantidad_marcas_tarjeta_con_uso,

        (
            tiene_visa = 1
            AND ctarjeta_visa_transacciones = 0
        )::TINYINT AS visa_sin_uso,

        (
            tiene_master = 1
            AND ctarjeta_master_transacciones = 0
        )::TINYINT AS master_sin_uso,

        usa_homebanking_transaccional
            + usa_mobile_app_transaccional
            + usa_atm
            + usa_cajas
            + usa_callcenter
            AS cantidad_canales_usados

    FROM componentes
),


-- ============================================================
-- ESTADOS DE LAS FAMILIAS
-- ============================================================

estados AS (
    SELECT
        *,

        (
            COALESCE(coperaciones_recurrentes_total, 0) > 0
        )::TINYINT AS tiene_operaciones_recurrentes,

        (
            cantidad_marcas_tarjeta > 0
        )::TINYINT AS tiene_tarjeta,

        (
            cantidad_marcas_tarjeta_con_uso > 0
        )::TINYINT AS usa_tarjeta,

        (
            visa_sin_uso = 1
            OR master_sin_uso = 1
        )::TINYINT AS tiene_tarjeta_sin_uso,

        (
            usa_homebanking_transaccional = 1
            OR usa_mobile_app_transaccional = 1
        )::TINYINT AS actividad_digital_transaccional,

        (
            COALESCE(cantidad_creditos, 0) > 0
        )::TINYINT AS tiene_credito,

        (
            COALESCE(cantidad_inversiones, 0) > 0
        )::TINYINT AS tiene_inversion,

        (
            COALESCE(cantidad_seguros, 0) > 0
        )::TINYINT AS tiene_seguro,

        (
            COALESCE(debitos_automaticos_total, 0) > 0
        )::TINYINT AS usa_debitos_automaticos,

        (
            COALESCE(pagos_servicios_total, 0) > 0
        )::TINYINT AS paga_servicios

    FROM agregados
),


-- ============================================================
-- PROFUNDIDAD, ACTIVIDAD Y SALDOS
-- ============================================================

profundidad AS (
    SELECT
        *,

        -- Las ocho familias que definimos para el modelo.
        tiene_cuentas
            + tiene_tarjeta
            + tiene_credito
            + tiene_inversion
            + tiene_seguro
            + tiene_payroll
            + tiene_operaciones_recurrentes
            + actividad_digital_transaccional
            AS familias_activas_modelo,

        -- Las nueve familias utilizadas exactamente por Miranda.
        payroll_activo_miranda
            + usa_tarjeta
            + actividad_digital_transaccional
            + emite_transferencias
            + tiene_credito
            + tiene_inversion
            + tiene_seguro
            + usa_debitos_automaticos
            + paga_servicios
            AS familias_activas_miranda,

        ratio_seguro(
            ctrx_quarter,
            cproductos
        ) AS actividad_por_producto,

        CASE
            WHEN mcuentas_saldo IS NULL THEN NULL
            ELSE (mcuentas_saldo < 0)::TINYINT
        END AS saldo_negativo,

        CASE
            WHEN mcuenta_corriente IS NULL THEN NULL
            ELSE GREATEST(-mcuenta_corriente, 0)
        END AS descubierto_utilizado,

        (
            tiene_payroll = 1
            AND tiene_operaciones_recurrentes = 1
            AND tiene_tarjeta = 1
        )::TINYINT AS cliente_anclado

    FROM estados
),


-- ============================================================
-- LAGS CRUDOS
-- ============================================================

lags_crudos AS (
    SELECT
        *,

        LAG(_indice_mes) OVER w
            AS _indice_mes_lag1,

        LAG(mpayroll) OVER w
            AS _mpayroll_lag1_crudo,

        LAG(tiene_payroll) OVER w
            AS _tiene_payroll_lag1_crudo,

        LAG(coperaciones_recurrentes_total) OVER w
            AS _recurrentes_lag1_crudo,

        LAG(tiene_operaciones_recurrentes) OVER w
            AS _tiene_recurrentes_lag1_crudo,

        LAG(cantidad_marcas_tarjeta) OVER w
            AS _marcas_lag1_crudo,

        LAG(cantidad_marcas_tarjeta_con_uso) OVER w
            AS _marcas_uso_lag1_crudo,

        LAG(tiene_tarjeta) OVER w
            AS _tiene_tarjeta_lag1_crudo,

        LAG(usa_tarjeta) OVER w
            AS _usa_tarjeta_lag1_crudo,

        LAG(cantidad_canales_usados) OVER w
            AS _canales_lag1_crudo,

        LAG(actividad_digital_transaccional) OVER w
            AS _digital_lag1_crudo,

        LAG(familias_activas_modelo) OVER w
            AS _familias_modelo_lag1_crudo,

        LAG(tiene_credito) OVER w
            AS _credito_lag1_crudo,

        LAG(tiene_inversion) OVER w
            AS _inversion_lag1_crudo,

        LAG(tiene_seguro) OVER w
            AS _seguro_lag1_crudo,

        LAG(ctrx_quarter) OVER w
            AS _ctrx_lag1_crudo,

        LAG(mcuentas_saldo) OVER w
            AS _saldo_lag1_crudo

    FROM profundidad

    WINDOW w AS (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    )
),


-- ============================================================
-- CONTROL DE CONTINUIDAD CALENDARIO
-- ============================================================

historia AS (
    SELECT
        *,

        CASE
            WHEN _indice_mes_lag1 IS NOT NULL
             AND _indice_mes - _indice_mes_lag1 = 1
            THEN 1
            ELSE 0
        END::TINYINT AS tiene_historia_lag1

    FROM lags_crudos
),


-- Solamente aceptamos el lag si corresponde exactamente
-- al mes calendario anterior.
lags AS (
    SELECT
        *,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _mpayroll_lag1_crudo
        END AS mpayroll_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _tiene_payroll_lag1_crudo
        END AS tiene_payroll_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _recurrentes_lag1_crudo
        END AS coperaciones_recurrentes_total_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _tiene_recurrentes_lag1_crudo
        END AS tiene_operaciones_recurrentes_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _marcas_lag1_crudo
        END AS cantidad_marcas_tarjeta_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _marcas_uso_lag1_crudo
        END AS cantidad_marcas_tarjeta_con_uso_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _tiene_tarjeta_lag1_crudo
        END AS tiene_tarjeta_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _usa_tarjeta_lag1_crudo
        END AS usa_tarjeta_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _canales_lag1_crudo
        END AS cantidad_canales_usados_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _digital_lag1_crudo
        END AS actividad_digital_transaccional_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _familias_modelo_lag1_crudo
        END AS familias_activas_modelo_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _credito_lag1_crudo
        END AS tiene_credito_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _inversion_lag1_crudo
        END AS tiene_inversion_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _seguro_lag1_crudo
        END AS tiene_seguro_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _ctrx_lag1_crudo
        END AS ctrx_quarter_lag1,

        CASE WHEN tiene_historia_lag1 = 1
            THEN _saldo_lag1_crudo
        END AS mcuentas_saldo_lag1

    FROM historia
),


-- ============================================================
-- REFERENCIAS Y DELTAS
-- ============================================================

derivadas AS (
    SELECT
        *,

        -- Promedia exclusivamente los payroll positivos disponibles.
        list_avg([
            CASE WHEN mpayroll > 0
                THEN mpayroll
            END,

            CASE WHEN mpayroll_lag1 > 0
                THEN mpayroll_lag1
            END
        ]) AS payroll_referencia_2m,

        ctrx_quarter
            - ctrx_quarter_lag1
            AS ctrx_quarter_delta1,

        ratio_seguro(
            ctrx_quarter - ctrx_quarter_lag1,
            ABS(ctrx_quarter_lag1)
        ) AS ctrx_quarter_cambio_relativo_lag1,

        mcuentas_saldo
            - mcuentas_saldo_lag1
            AS mcuentas_saldo_delta1,

        ratio_seguro(
            mpayroll - mpayroll_lag1,
            ABS(mpayroll_lag1)
        ) AS payroll_cambio_relativo_lag1,

        coperaciones_recurrentes_total
            - coperaciones_recurrentes_total_lag1
            AS recurrentes_delta1,

        cantidad_marcas_tarjeta
            - cantidad_marcas_tarjeta_lag1
            AS marcas_tarjeta_delta1,

        cantidad_marcas_tarjeta_con_uso
            - cantidad_marcas_tarjeta_con_uso_lag1
            AS marcas_tarjeta_con_uso_delta1,

        cantidad_canales_usados
            - cantidad_canales_usados_lag1
            AS canales_usados_delta1,

        familias_activas_modelo
            - familias_activas_modelo_lag1
            AS familias_activas_modelo_delta1

    FROM lags
),


-- ============================================================
-- EVENTOS DE APAGADO Y DETERIORO
-- ============================================================

eventos AS (
    SELECT
        *,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN tiene_payroll_lag1 = 1
                 AND tiene_payroll = 0
            THEN 1 ELSE 0
        END AS apago_payroll_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN tiene_operaciones_recurrentes_lag1 = 1
                 AND tiene_operaciones_recurrentes = 0
            THEN 1 ELSE 0
        END AS apago_recurrentes_lag1,

        -- Pérdida de tenencia de Visa/Master.
        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN tiene_tarjeta_lag1 = 1
                 AND tiene_tarjeta = 0
            THEN 1 ELSE 0
        END AS apago_tarjeta_lag1,

        -- Conserva la tarjeta, pero deja de utilizarla.
        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN usa_tarjeta_lag1 = 1
                 AND usa_tarjeta = 0
            THEN 1 ELSE 0
        END AS apago_uso_tarjeta_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN actividad_digital_transaccional_lag1 = 1
                 AND actividad_digital_transaccional = 0
            THEN 1 ELSE 0
        END AS apago_digital_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN tiene_credito_lag1 = 1
                 AND tiene_credito = 0
            THEN 1 ELSE 0
        END AS apago_credito_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN tiene_inversion_lag1 = 1
                 AND tiene_inversion = 0
            THEN 1 ELSE 0
        END AS apago_inversion_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            WHEN tiene_seguro_lag1 = 1
                 AND tiene_seguro = 0
            THEN 1 ELSE 0
        END AS apago_seguro_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1
                 OR ctrx_quarter_lag1 IS NULL
                 OR ctrx_quarter IS NULL
            THEN NULL

            WHEN ctrx_quarter_lag1 > 0
                 AND ctrx_quarter <= 0.80 * ctrx_quarter_lag1
            THEN 1

            ELSE 0
        END AS caida_ctrx_20pct_lag1,

        CASE
            WHEN tiene_historia_lag1 <> 1
                 OR mcuentas_saldo_lag1 IS NULL
                 OR mcuentas_saldo IS NULL
            THEN NULL

            WHEN mcuentas_saldo_lag1 >= 0
                 AND mcuentas_saldo < 0
            THEN 1

            ELSE 0
        END AS cruzo_saldo_negativo_lag1

    FROM derivadas
),


-- ============================================================
-- RESUMEN DE FAMILIAS APAGADAS
-- ============================================================

resultado AS (
    SELECT
        *,

        CASE
            WHEN tiene_historia_lag1 <> 1 THEN NULL
            ELSE
                COALESCE(apago_payroll_lag1, 0)
                + COALESCE(apago_recurrentes_lag1, 0)
                + COALESCE(apago_uso_tarjeta_lag1, 0)
                + COALESCE(apago_digital_lag1, 0)
                + COALESCE(apago_credito_lag1, 0)
                + COALESCE(apago_inversion_lag1, 0)
                + COALESCE(apago_seguro_lag1, 0)
        END AS cantidad_familias_apagadas_lag1

    FROM eventos
),


resultado_final AS (
    SELECT
        *,

        CASE
            WHEN cantidad_familias_apagadas_lag1 IS NULL
            THEN NULL

            WHEN cantidad_familias_apagadas_lag1 > 0
            THEN 1

            ELSE 0
        END AS apago_alguna_familia_lag1

    FROM resultado
)


-- ============================================================
-- RESULTADO FINAL
-- Se eliminan solamente las columnas auxiliares internas.
-- ============================================================

SELECT * EXCLUDE (
    _indice_mes,
    _indice_mes_lag1,
    _mpayroll_lag1_crudo,
    _tiene_payroll_lag1_crudo,
    _recurrentes_lag1_crudo,
    _tiene_recurrentes_lag1_crudo,
    _marcas_lag1_crudo,
    _marcas_uso_lag1_crudo,
    _tiene_tarjeta_lag1_crudo,
    _usa_tarjeta_lag1_crudo,
    _canales_lag1_crudo,
    _digital_lag1_crudo,
    _familias_modelo_lag1_crudo,
    _credito_lag1_crudo,
    _inversion_lag1_crudo,
    _seguro_lag1_crudo,
    _ctrx_lag1_crudo,
    _saldo_lag1_crudo
)
FROM resultado_final;

,Success


Auditoría


In [27]:
%%sql
SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS claves_cliente_mes,

    MIN(foto_mes) AS primer_mes,
    MAX(foto_mes) AS ultimo_mes,

    COUNT(*) FILTER (
        WHERE cantidad_marcas_tarjeta NOT BETWEEN 0 AND 2
    ) AS error_rango_marcas,

    COUNT(*) FILTER (
        WHERE cantidad_marcas_tarjeta_con_uso NOT BETWEEN 0 AND 2
    ) AS error_rango_marcas_usadas,

    COUNT(*) FILTER (
        WHERE cantidad_canales_usados NOT BETWEEN 0 AND 5
    ) AS error_rango_canales,

    COUNT(*) FILTER (
        WHERE familias_activas_modelo NOT BETWEEN 0 AND 8
    ) AS error_familias_modelo,

    COUNT(*) FILTER (
        WHERE familias_activas_miranda NOT BETWEEN 0 AND 9
    ) AS error_familias_miranda,

    COUNT(*) FILTER (
        WHERE descubierto_utilizado < 0
    ) AS error_descubierto_negativo,

    COUNT(*) FILTER (
        WHERE payroll_referencia_2m <= 0
    ) AS error_payroll_referencia,

    COUNT(*) FILTER (
        WHERE cliente_anclado = 1
          AND (
              tiene_payroll <> 1
              OR tiene_operaciones_recurrentes <> 1
              OR tiene_tarjeta <> 1
          )
    ) AS error_cliente_anclado,

    COUNT(*) FILTER (
        WHERE tiene_tarjeta_sin_uso = 1
          AND tiene_tarjeta <> 1
    ) AS error_tarjeta_sin_tenencia,

    COUNT(*) FILTER (
        WHERE cantidad_familias_apagadas_lag1 NOT BETWEEN 0 AND 7
    ) AS error_familias_apagadas

FROM competencia_02_features;

,filas,claves_cliente_mes,primer_mes,ultimo_mes,error_rango_marcas,error_rango_marcas_usadas,error_rango_canales,error_familias_modelo,error_familias_miranda,error_descubierto_negativo,error_payroll_referencia,error_cliente_anclado,error_tarjeta_sin_tenencia,error_familias_apagadas
0,983061,983061,202103,202108,0,0,0,0,0,0,0,0,0,0


In [28]:
%%sql
SELECT
    tiene_historia_lag1,
    COUNT(*) AS filas,

    COUNT(mpayroll_lag1) AS payroll_lag_informado,
    COUNT(ctrx_quarter_lag1) AS actividad_lag_informada,
    COUNT(mcuentas_saldo_lag1) AS saldo_lag_informado,

    COUNT(apago_payroll_lag1) AS eventos_payroll_informados,
    COUNT(caida_ctrx_20pct_lag1) AS eventos_caida_informados,
    COUNT(cruzo_saldo_negativo_lag1) AS cruces_negativos_informados

FROM competencia_02_features
GROUP BY tiene_historia_lag1
ORDER BY tiene_historia_lag1;

,tiene_historia_lag1,filas,payroll_lag_informado,actividad_lag_informada,saldo_lag_informado,eventos_payroll_informados,eventos_caida_informados,cruces_negativos_informados
0,0,169750,0,0,0,0,0,0
1,1,813311,813311,813311,813311,813311,813311,813311


In [29]:
%%sql
SELECT
    foto_mes,
    COUNT(*) AS clientes,

    AVG(tiene_payroll) AS proporcion_payroll,
    AVG(tiene_operaciones_recurrentes) AS proporcion_recurrentes,
    AVG(tiene_tarjeta) AS proporcion_tarjeta,
    AVG(usa_tarjeta) AS proporcion_usa_tarjeta,
    AVG(actividad_digital_transaccional) AS proporcion_digital,
    AVG(saldo_negativo) AS proporcion_saldo_negativo,
    AVG(cliente_anclado) AS proporcion_anclados,

    MEDIAN(cantidad_canales_usados) AS mediana_canales,
    MEDIAN(familias_activas_modelo) AS mediana_familias_modelo,
    MEDIAN(familias_activas_miranda) AS mediana_familias_miranda,

    AVG(apago_payroll_lag1) AS proporcion_apago_payroll,
    AVG(apago_recurrentes_lag1) AS proporcion_apago_recurrentes,
    AVG(apago_uso_tarjeta_lag1) AS proporcion_apago_uso_tarjeta,
    AVG(caida_ctrx_20pct_lag1) AS proporcion_caida_actividad,
    AVG(cruzo_saldo_negativo_lag1) AS proporcion_cruzo_negativo

FROM competencia_02_features
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,proporcion_payroll,proporcion_recurrentes,proporcion_tarjeta,proporcion_usa_tarjeta,proporcion_digital,proporcion_saldo_negativo,proporcion_anclados,mediana_canales,mediana_familias_modelo,mediana_familias_miranda,proporcion_apago_payroll,proporcion_apago_recurrentes,proporcion_apago_uso_tarjeta,proporcion_caida_actividad,proporcion_cruzo_negativo
0,202103,162900,0.540454,0.842897,0.957287,0.862136,0.939650,0.204205,0.465721,2.0,5.0,5.0,NaN,NaN,NaN,NaN,NaN
1,202104,163284,0.541014,0.857990,0.955985,0.864255,0.939314,0.203670,0.472239,2.0,5.0,5.0,0.016166,0.012213,0.008611,0.048721,0.058438
2,202105,163768,0.549601,0.855973,0.955870,0.864644,0.941326,0.204142,0.478781,2.0,5.0,5.0,0.011428,0.014194,0.009142,0.052631,0.054892
3,202106,164114,0.552665,0.856161,0.956756,0.869804,0.941711,0.172344,0.481592,2.0,5.0,5.0,0.013405,0.017144,0.007022,0.053688,0.044489
4,202107,164348,0.551683,0.860996,0.955728,0.869071,0.942451,0.182302,0.483121,2.0,5.0,5.0,0.014647,0.013373,0.008080,0.043923,0.062319
5,202108,164647,0.557478,0.862554,0.953494,0.869448,0.944427,0.200854,0.487431,2.0,5.0,5.0,0.011467,0.013029,0.008343,0.032742,0.065638


In [30]:
%%sql

-- ============================================================
-- MACROS
-- ============================================================

-- Ignora NULL parciales, pero devuelve NULL si todos son NULL.
CREATE OR REPLACE MACRO suma_sin_null(valores) AS
    list_sum(valores);

-- El ratio existe solamente si el denominador es positivo.
CREATE OR REPLACE MACRO ratio_seguro(numerador, denominador) AS (
    CASE
        WHEN denominador > 0
        THEN numerador::DOUBLE / denominador::DOUBLE
        ELSE NULL
    END
);


-- ============================================================
-- BLOQUE 2: CARGA FINANCIERA SOBRE PAYROLL
-- ============================================================

CREATE OR REPLACE TABLE competencia_03_carga_financiera AS

WITH numeradores AS (
    SELECT
        *,

        -- ----------------------------------------------------
        -- Deuda de tarjetas en pesos
        -- Conservamos solamente saldos positivos.
        -- Un saldo negativo puede ser un ajuste o saldo a favor.
        -- ----------------------------------------------------

        suma_sin_null([
            CASE
                WHEN Visa_msaldopesos IS NULL THEN NULL
                ELSE GREATEST(Visa_msaldopesos, 0)
            END,

            CASE
                WHEN Master_msaldopesos IS NULL THEN NULL
                ELSE GREATEST(Master_msaldopesos, 0)
            END
        ]) AS deuda_tarjetas_pesos,

        -- ----------------------------------------------------
        -- Pago mínimo total de Visa y Master
        -- ----------------------------------------------------

        suma_sin_null([
            Visa_mpagominimo,
            Master_mpagominimo
        ]) AS pago_minimo_tarjetas,

        -- ----------------------------------------------------
        -- Préstamos totales
        -- ----------------------------------------------------

        suma_sin_null([
            mprestamos_personales,
            mprestamos_prendarios,
            mprestamos_hipotecarios
        ]) AS prestamos_totales

    FROM competencia_02_features
),


deuda_total AS (
    SELECT
        *,

        -- ----------------------------------------------------
        -- Deuda observada total
        -- ----------------------------------------------------

        suma_sin_null([
            deuda_tarjetas_pesos,
            prestamos_totales,
            descubierto_utilizado
        ]) AS deuda_total_pesos

    FROM numeradores
)


SELECT
    *,

    -- 1. Carga de comisiones
    ratio_seguro(
        mcomisiones,
        payroll_referencia_2m
    ) AS comisiones_sobre_payroll_referencia_2m,

    -- 2. Costo de mantenimiento
    ratio_seguro(
        mcomisiones_mantenimiento,
        payroll_referencia_2m
    ) AS comisiones_mantenimiento_sobre_payroll_referencia_2m,

    -- 3. Deuda de tarjetas en pesos
    ratio_seguro(
        deuda_tarjetas_pesos,
        payroll_referencia_2m
    ) AS deuda_tarjetas_pesos_sobre_payroll_referencia_2m,

    -- 4. Pago mínimo requerido
    ratio_seguro(
        pago_minimo_tarjetas,
        payroll_referencia_2m
    ) AS pago_minimo_tarjetas_sobre_payroll_referencia_2m,

    -- 5. Préstamos totales
    ratio_seguro(
        prestamos_totales,
        payroll_referencia_2m
    ) AS prestamos_totales_sobre_payroll_referencia_2m,

    -- 6. Descubierto utilizado
    ratio_seguro(
        descubierto_utilizado,
        payroll_referencia_2m
    ) AS descubierto_utilizado_sobre_payroll_referencia_2m,

    -- 7. Deuda observada total
    ratio_seguro(
        deuda_total_pesos,
        payroll_referencia_2m
    ) AS deuda_total_pesos_sobre_payroll_referencia_2m,

    -- 8. Posición de saldo
    -- Se conserva el signo de mcuentas_saldo.
    ratio_seguro(
        mcuentas_saldo,
        payroll_referencia_2m
    ) AS saldo_cuentas_sobre_payroll_referencia_2m

FROM deuda_total;

,Success


In [31]:
%%sql

SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS claves_cliente_mes,

    COUNT(*) FILTER (
        WHERE deuda_tarjetas_pesos < 0
    ) AS errores_deuda_tarjetas_negativa,

    COUNT(*) FILTER (
        WHERE pago_minimo_tarjetas < 0
    ) AS pagos_minimos_negativos,

    COUNT(*) FILTER (
        WHERE prestamos_totales < 0
    ) AS prestamos_totales_negativos,

    COUNT(*) FILTER (
        WHERE deuda_total_pesos < 0
    ) AS errores_deuda_total_negativa,

    COUNT(*) FILTER (
        WHERE payroll_referencia_2m IS NULL
          AND (
              comisiones_sobre_payroll_referencia_2m IS NOT NULL
              OR comisiones_mantenimiento_sobre_payroll_referencia_2m IS NOT NULL
              OR deuda_tarjetas_pesos_sobre_payroll_referencia_2m IS NOT NULL
              OR pago_minimo_tarjetas_sobre_payroll_referencia_2m IS NOT NULL
              OR prestamos_totales_sobre_payroll_referencia_2m IS NOT NULL
              OR descubierto_utilizado_sobre_payroll_referencia_2m IS NOT NULL
              OR deuda_total_pesos_sobre_payroll_referencia_2m IS NOT NULL
              OR saldo_cuentas_sobre_payroll_referencia_2m IS NOT NULL
          )
    ) AS ratios_informados_sin_payroll

FROM competencia_03_carga_financiera;

,filas,claves_cliente_mes,errores_deuda_tarjetas_negativa,pagos_minimos_negativos,prestamos_totales_negativos,errores_deuda_total_negativa,ratios_informados_sin_payroll
0,983061,983061,0,0,0,0,0


In [32]:
%%sql

WITH base AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY foto_mes
        ) AS clientes_mes
    FROM competencia_03_carga_financiera
),

ratios_largos AS (
    UNPIVOT base
    ON
        comisiones_sobre_payroll_referencia_2m,
        comisiones_mantenimiento_sobre_payroll_referencia_2m,
        deuda_tarjetas_pesos_sobre_payroll_referencia_2m,
        pago_minimo_tarjetas_sobre_payroll_referencia_2m,
        prestamos_totales_sobre_payroll_referencia_2m,
        descubierto_utilizado_sobre_payroll_referencia_2m,
        deuda_total_pesos_sobre_payroll_referencia_2m,
        saldo_cuentas_sobre_payroll_referencia_2m
    INTO
        NAME feature
        VALUE valor
)

SELECT
    foto_mes,
    feature,

    MAX(clientes_mes) AS clientes,
    COUNT(*) AS valores_informados,

    ROUND(
        100.0 * COUNT(*) / MAX(clientes_mes),
        2
    ) AS cobertura_porcentaje,

    MEDIAN(valor) AS mediana,
    QUANTILE_CONT(valor, 0.95) AS percentil_95,
    QUANTILE_CONT(valor, 0.99) AS percentil_99,
    MIN(valor) AS minimo,
    MAX(valor) AS maximo

FROM ratios_largos
GROUP BY foto_mes, feature
ORDER BY feature, foto_mes;

,foto_mes,feature,clientes,valores_informados,cobertura_porcentaje,mediana,percentil_95,percentil_99,minimo,maximo
0,202103,comisiones_mantenimiento_sobre_payroll_referen...,162900,87077,53.45,0.000000,0.000000,0.058198,-5.180357e+02,5.011900e+03
1,202104,comisiones_mantenimiento_sobre_payroll_referen...,163284,89984,55.11,0.000000,0.000000,0.069063,-1.505976e+01,5.011900e+03
2,202105,comisiones_mantenimiento_sobre_payroll_referen...,163768,90886,55.50,0.000000,0.000000,0.058654,-4.034669e+01,1.252975e+05
3,202106,comisiones_mantenimiento_sobre_payroll_referen...,164114,91918,56.01,0.000000,0.000000,0.058805,-5.683715e+01,1.252975e+05
4,202107,comisiones_mantenimiento_sobre_payroll_referen...,164348,92096,56.04,0.000000,0.000000,0.058313,-5.819829e+01,1.216486e+03
5,202108,comisiones_mantenimiento_sobre_payroll_referen...,164647,92695,56.30,0.000000,0.000000,0.063909,-9.772418e+00,4.188217e+02
6,202103,comisiones_sobre_payroll_referencia_2m,162900,87077,53.45,0.005223,0.047410,0.258271,-9.213000e+03,1.180810e+05
7,202104,comisiones_sobre_payroll_referencia_2m,163284,89984,55.11,0.005585,0.035358,0.173998,-5.742190e+00,1.236560e+05
8,202105,comisiones_sobre_payroll_referencia_2m,163768,90886,55.50,0.005360,0.034865,0.169390,-3.514492e+01,1.884550e+05
9,202106,comisiones_sobre_payroll_referencia_2m,164114,91918,56.01,0.004503,0.028585,0.156317,-2.344160e+01,1.905220e+05


El bloque tiene buena cobertura y estabilidad central, pero presenta colas extremas que debemos revisar antes de modelar.

Lo que está bien
- Cobertura general: sube suavemente de 53,45% a 56,30%.
- Ratios de tarjetas: cobertura algo menor, 51,80%–54,55%, por los NULL estructurales de Visa/Master. Es esperable.
- Medianas, percentiles 95 y 99 son razonablemente estables entre meses.
- No se observa un salto mensual que sugiera un error de construcción.

Lecturas principales:
- Comisiones: mediana de 0,45%–0,60% del payroll.
- Deuda de tarjetas: mediana de 21%–26%; percentil 99 alrededor de 7–8,5 payrolls.
- Pago mínimo: mediana de 2,2%–2,7%.
- Deuda total: mediana de 29%–36%; percentil 99 alrededor de 19–21 payrolls.
- Saldo: mediana de 0,51–0,65 payrolls.
- Préstamos y descubierto tienen mediana cero, coherente con su baja penetración.

Lo que debemos revisar
Los máximos son desproporcionados:
- Comisiones/payroll: hasta 311.465.
- Descubierto/payroll: hasta 5 millones.
- Deuda total/payroll: hasta 69 millones.
- Saldo/payroll: hasta 131 millones.
Como los percentiles 99 son muchísimo menores, probablemente son muy pocos clientes con payroll_referencia_2m positivo pero extremadamente pequeño.

In [33]:
%%sql

SELECT
    foto_mes,

    COUNT(*) FILTER (
        WHERE payroll_referencia_2m > 0
    ) AS payroll_informado,

    MIN(payroll_referencia_2m) AS minimo,

    QUANTILE_CONT(
        payroll_referencia_2m,
        0.001
    ) FILTER (
        WHERE payroll_referencia_2m > 0
    ) AS percentil_001,

    QUANTILE_CONT(
        payroll_referencia_2m,
        0.01
    ) FILTER (
        WHERE payroll_referencia_2m > 0
    ) AS percentil_01,

    QUANTILE_CONT(
        payroll_referencia_2m,
        0.05
    ) FILTER (
        WHERE payroll_referencia_2m > 0
    ) AS percentil_05,

    MEDIAN(
        payroll_referencia_2m
    ) FILTER (
        WHERE payroll_referencia_2m > 0
    ) AS mediana,

    COUNT(*) FILTER (
        WHERE payroll_referencia_2m > 0
          AND payroll_referencia_2m < 1
    ) AS payroll_menor_1,

    COUNT(*) FILTER (
        WHERE payroll_referencia_2m >= 1
          AND payroll_referencia_2m < 100
    ) AS payroll_entre_1_y_100,

    COUNT(*) FILTER (
        WHERE payroll_referencia_2m >= 100
          AND payroll_referencia_2m < 1000
    ) AS payroll_entre_100_y_1000

FROM competencia_03_carga_financiera
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,payroll_informado,minimo,percentil_001,percentil_01,percentil_05,mediana,payroll_menor_1,payroll_entre_1_y_100,payroll_entre_100_y_1000
0,202103,87077,0.01,258.82012,5327.76,21870.71600,98080.3200,24,35,125
1,202104,89984,0.01,453.06241,5865.00,22824.13800,100091.2050,21,26,113
2,202105,90886,0.01,555.02800,5865.00,23517.33000,103991.1475,8,43,85
3,202106,91918,0.01,384.71069,5865.00,26859.27825,130983.3350,7,44,121
4,202107,92096,0.01,574.59925,5865.00,28127.29000,136683.6475,5,25,112
5,202108,92695,0.01,684.11822,5865.00,25481.43500,118678.2750,7,33,92


Quedó confirmado: los máximos enormes provienen de payrolls positivos diminutos, incluso de `0.01`.

Pero afectan a muy pocos registros:

| Mes | Payroll entre 0 y 1.000 | % de payroll informado |
|---|---:|---:|
| 202103 | 184 | 0,21% |
| 202104 | 160 | 0,18% |
| 202105 | 136 | 0,15% |
| 202106 | 172 | 0,19% |
| 202107 | 142 | 0,15% |
| 202108 | 132 | 0,14% |

Conclusión:

- No es un problema generalizado.
- La cobertura es estable.
- Los percentiles 95 y 99 son razonables.
- Los máximos extremos quedan explicados por menos del 0,22% de los casos.
- No hay que reconstruir la tabla.

Para mantener coherencia con el bloque que había mejorado la validación, **dejaría los ratios originales sin recortar**. LightGBM puede separar esos valores extremos en un nodo/bin y manejar los `NULL` nativamente.

No agregaría ahora un límite arbitrario de $1.000 ni reemplazaría esos casos por cero. Si posteriormente hacemos una prueba específica de robustez, podemos comparar contra versiones recortadas o rankeadas por mes, pero como challenger separado.

Resultado final: **el segundo bloque está correctamente construido y su salud queda aprobada**. El siguiente paso es medir si estas ocho variables efectivamente aumentan la tasa de detección de `BAJA+2` y luego compararlas mediante validación temporal.

In [34]:
%%sql

-- ============================================================
-- BLOQUE 3: INFORMACIÓN DE TARJETAS
-- ============================================================
-- Tabla challenger: no modifica competencia_03_carga_financiera.
--
-- IMPORTANTE:
-- suma_sin_null ignora NULL parciales, pero devuelve NULL
-- cuando Visa y Master son ambas NULL.
-- ============================================================

CREATE OR REPLACE TABLE competencia_04_tarjetas_challenger AS

WITH totales_tarjetas AS (
    SELECT
        *,

        -- ----------------------------------------------------
        -- ACTIVIDAD OPERATIVA DE TARJETAS
        -- ----------------------------------------------------

        suma_sin_null([
            ctarjeta_visa_transacciones,
            ctarjeta_master_transacciones
        ]) AS ctarjetas_credito_transacciones_total,

        suma_sin_null([
            mtarjeta_visa_consumo,
            mtarjeta_master_consumo
        ]) AS mtarjetas_credito_consumo_total,

        -- ----------------------------------------------------
        -- SALDOS DE RESUMEN
        -- Pesos y dólares se mantienen separados.
        -- En pesos conservamos el signo original.
        -- ----------------------------------------------------

        suma_sin_null([
            Visa_msaldopesos,
            Master_msaldopesos
        ]) AS tarjetas_saldo_pesos_total,

        suma_sin_null([
            Visa_msaldodolares,
            Master_msaldodolares
        ]) AS tarjetas_saldo_dolares_total,

        -- ----------------------------------------------------
        -- LÍMITES
        -- ----------------------------------------------------

        suma_sin_null([
            Visa_mlimitecompra,
            Master_mlimitecompra
        ]) AS tarjetas_limite_compra_total,

        suma_sin_null([
            Visa_mfinanciacion_limite,
            Master_mfinanciacion_limite
        ]) AS tarjetas_limite_financiacion_total,

        -- ----------------------------------------------------
        -- CONSUMOS DEL RESUMEN
        -- ----------------------------------------------------

        suma_sin_null([
            Visa_mconsumospesos,
            Master_mconsumospesos
        ]) AS tarjetas_consumo_pesos_total,

        suma_sin_null([
            Visa_mconsumosdolares,
            Master_mconsumosdolares
        ]) AS tarjetas_consumo_dolares_total,

        suma_sin_null([
            Visa_cconsumos,
            Master_cconsumos
        ]) AS tarjetas_cconsumos_total

        -- No reconstruimos pago_minimo_tarjetas:
        -- ya existe en competencia_03_carga_financiera.

    FROM competencia_03_carga_financiera
),


ratios_tarjetas AS (
    SELECT
        *,

        -- Saldo en pesos respecto del límite disponible.
        ratio_seguro(
            tarjetas_saldo_pesos_total,
            tarjetas_limite_compra_total
        ) AS tarjetas_utilizacion_limite,

        -- Consumo operativo respecto del límite.
        ratio_seguro(
            mtarjetas_credito_consumo_total,
            tarjetas_limite_compra_total
        ) AS tarjetas_consumo_sobre_limite,

        -- Relación entre límite de financiación y de compra.
        ratio_seguro(
            tarjetas_limite_financiacion_total,
            tarjetas_limite_compra_total
        ) AS tarjetas_financiacion_sobre_limite,

        -- Crédito todavía disponible.
        CASE
            WHEN tarjetas_limite_compra_total IS NOT NULL
             AND tarjetas_saldo_pesos_total IS NOT NULL
            THEN tarjetas_limite_compra_total
                 - tarjetas_saldo_pesos_total
            ELSE NULL
        END AS tarjetas_holgura_compra,

        -- Consumo promedio por transacción.
        ratio_seguro(
            mtarjetas_credito_consumo_total,
            ctarjetas_credito_transacciones_total
        ) AS tarjetas_ticket_promedio,

        -- Participación de Visa en las transacciones.
        ratio_seguro(
            COALESCE(ctarjeta_visa_transacciones, 0),
            ctarjetas_credito_transacciones_total
        ) AS share_visa_transacciones,

        -- Participación de Visa en el consumo.
        ratio_seguro(
            COALESCE(mtarjeta_visa_consumo, 0),
            mtarjetas_credito_consumo_total
        ) AS share_visa_consumo

    FROM totales_tarjetas
)


SELECT *
FROM ratios_tarjetas;

,Success


In [35]:
%%sql

SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS claves_cliente_mes,

    COUNT(*) FILTER (
        WHERE share_visa_transacciones < 0
           OR share_visa_transacciones > 1
    ) AS errores_share_transacciones,

    COUNT(*) FILTER (
        WHERE share_visa_consumo < 0
           OR share_visa_consumo > 1
    ) AS errores_share_consumo,

    COUNT(*) FILTER (
        WHERE (
            tarjetas_limite_compra_total IS NULL
            OR tarjetas_limite_compra_total <= 0
        )
        AND (
            tarjetas_utilizacion_limite IS NOT NULL
            OR tarjetas_consumo_sobre_limite IS NOT NULL
            OR tarjetas_financiacion_sobre_limite IS NOT NULL
        )
    ) AS ratios_informados_sin_limite_valido,

    COUNT(*) FILTER (
        WHERE (
            ctarjetas_credito_transacciones_total IS NULL
            OR ctarjetas_credito_transacciones_total <= 0
        )
        AND tarjetas_ticket_promedio IS NOT NULL
    ) AS tickets_informados_sin_transacciones

FROM competencia_04_tarjetas_challenger;

,filas,claves_cliente_mes,errores_share_transacciones,errores_share_consumo,ratios_informados_sin_limite_valido,tickets_informados_sin_transacciones
0,983061,983061,0,0,0,0


In [36]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(tarjetas_limite_compra_total) / COUNT(*),
        2
    ) AS cobertura_limite,

    ROUND(
        100.0 * COUNT(tarjetas_utilizacion_limite) / COUNT(*),
        2
    ) AS cobertura_utilizacion,

    ROUND(
        100.0 * COUNT(tarjetas_ticket_promedio) / COUNT(*),
        2
    ) AS cobertura_ticket,

    ROUND(
        100.0 * COUNT(share_visa_transacciones) / COUNT(*),
        2
    ) AS cobertura_share_transacciones,

    MEDIAN(tarjetas_utilizacion_limite)
        AS mediana_utilizacion,

    QUANTILE_CONT(
        tarjetas_utilizacion_limite,
        0.99
    ) AS p99_utilizacion,

    MEDIAN(tarjetas_ticket_promedio)
        AS mediana_ticket,

    QUANTILE_CONT(
        tarjetas_ticket_promedio,
        0.99
    ) AS p99_ticket

FROM competencia_04_tarjetas_challenger
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,cobertura_limite,cobertura_utilizacion,cobertura_ticket,cobertura_share_transacciones,mediana_utilizacion,p99_utilizacion,mediana_ticket,p99_ticket
0,202103,162900,95.97,95.97,86.21,86.21,0.057047,0.891472,2127.296720,13281.334767
1,202104,163284,95.93,95.93,86.43,86.43,0.060368,0.897842,2211.511778,13304.442124
2,202105,163768,95.93,95.93,86.46,86.46,0.062112,0.928398,2311.874444,14339.203333
3,202106,164114,95.92,95.92,86.98,86.98,0.069407,0.952627,2411.283750,14988.004467
4,202107,164348,96.38,96.38,86.91,86.91,0.066010,0.944173,2467.334731,14782.237990
5,202108,164647,95.90,95.90,86.94,86.94,0.070287,1.033101,2548.780910,15068.279300


In [37]:
%%sql

-- ============================================================
-- BLOQUE 4: CUENTAS, CRÉDITOS, INVERSIONES Y SEGUROS
-- ============================================================

CREATE OR REPLACE TABLE
    competencia_05_productos_financieros_challenger AS

WITH componentes AS (
    SELECT
        *,

        -- ====================================================
        -- CUENTAS
        -- ====================================================

        suma_sin_null([
            ccuenta_corriente,
            ccaja_ahorro
        ]) AS cantidad_cuentas_basicas,

        CASE
            WHEN mcuenta_corriente IS NULL THEN NULL
            ELSE GREATEST(mcuenta_corriente, 0)
        END AS _cuenta_corriente_positiva,

        CASE
            WHEN mcaja_ahorro IS NULL THEN NULL
            ELSE GREATEST(mcaja_ahorro, 0)
        END AS _caja_ahorro_positiva,

        suma_sin_null([
            CASE
                WHEN mcuenta_corriente IS NULL THEN NULL
                ELSE GREATEST(mcuenta_corriente, 0)
            END,
            CASE
                WHEN mcaja_ahorro IS NULL THEN NULL
                ELSE GREATEST(mcaja_ahorro, 0)
            END
        ]) AS liquidez_cuentas_positiva,

        -- ====================================================
        -- INVERSIONES
        -- Sólo sumamos montos con moneda confirmada.
        -- ====================================================

        CASE
            WHEN mplazo_fijo_pesos IS NULL THEN NULL
            ELSE GREATEST(mplazo_fijo_pesos, 0)
        END AS _plazo_fijo_pesos_positivo,

        CASE
            WHEN minversion1_pesos IS NULL THEN NULL
            ELSE GREATEST(minversion1_pesos, 0)
        END AS _inversion1_pesos_positiva,

        CASE
            WHEN mplazo_fijo_dolares IS NULL THEN NULL
            ELSE GREATEST(mplazo_fijo_dolares, 0)
        END AS _plazo_fijo_dolares_positivo,

        CASE
            WHEN minversion1_dolares IS NULL THEN NULL
            ELSE GREATEST(minversion1_dolares, 0)
        END AS _inversion1_dolares_positiva,

        suma_sin_null([
            CASE
                WHEN mplazo_fijo_pesos IS NULL THEN NULL
                ELSE GREATEST(mplazo_fijo_pesos, 0)
            END,
            CASE
                WHEN minversion1_pesos IS NULL THEN NULL
                ELSE GREATEST(minversion1_pesos, 0)
            END
        ]) AS inversiones_pesos_confirmadas,

        suma_sin_null([
            CASE
                WHEN mplazo_fijo_dolares IS NULL THEN NULL
                ELSE GREATEST(mplazo_fijo_dolares, 0)
            END,
            CASE
                WHEN minversion1_dolares IS NULL THEN NULL
                ELSE GREATEST(minversion1_dolares, 0)
            END
        ]) AS inversiones_dolares_confirmadas,

        -- Número de tipos, no cantidad de operaciones.
        (
            (COALESCE(cplazo_fijo, 0) > 0)::TINYINT
            + (COALESCE(cinversion1, 0) > 0)::TINYINT
            + (COALESCE(cinversion2, 0) > 0)::TINYINT
        ) AS diversificacion_tipos_inversion,

        -- ====================================================
        -- SEGUROS Y CAJA DE SEGURIDAD
        -- ====================================================

        (
            (COALESCE(cseguro_vida, 0) > 0)::TINYINT
            + (COALESCE(cseguro_auto, 0) > 0)::TINYINT
            + (COALESCE(cseguro_vivienda, 0) > 0)::TINYINT
            + (
                COALESCE(cseguro_accidentes_personales, 0) > 0
              )::TINYINT
        ) AS diversificacion_tipos_seguro,

        (
            COALESCE(ccaja_seguridad, 0) > 0
        )::TINYINT AS tiene_caja_seguridad

    FROM competencia_04_tarjetas_challenger
),


derivadas AS (
    SELECT
        *,

        -- ====================================================
        -- PROFUNDIDAD Y COMPOSICIÓN DE CUENTAS
        -- ====================================================

        ratio_seguro(
            mcuentas_saldo,
            cantidad_cuentas_basicas
        ) AS saldo_promedio_por_cuenta,

        ratio_seguro(
            COALESCE(_cuenta_corriente_positiva, 0),
            liquidez_cuentas_positiva
        ) AS share_cuenta_corriente_liquidez,

        -- ====================================================
        -- CRÉDITOS
        -- ====================================================

        ratio_seguro(
            prestamos_totales,
            cantidad_creditos
        ) AS prestamos_promedio_por_credito,

        ratio_seguro(
            COALESCE(mprestamos_personales, 0),
            prestamos_totales
        ) AS share_prestamos_personales,

        ratio_seguro(
            COALESCE(mprestamos_prendarios, 0),
            prestamos_totales
        ) AS share_prestamos_prendarios,

        ratio_seguro(
            COALESCE(mprestamos_hipotecarios, 0),
            prestamos_totales
        ) AS share_prestamos_hipotecarios,

        -- Composición de la deuda observada.
        ratio_seguro(
            COALESCE(deuda_tarjetas_pesos, 0),
            deuda_total_pesos
        ) AS share_deuda_tarjetas,

        ratio_seguro(
            COALESCE(prestamos_totales, 0),
            deuda_total_pesos
        ) AS share_deuda_prestamos,

        ratio_seguro(
            COALESCE(descubierto_utilizado, 0),
            deuda_total_pesos
        ) AS share_deuda_descubierto,

        -- ====================================================
        -- INVERSIONES
        -- ====================================================

        ratio_seguro(
            COALESCE(_plazo_fijo_pesos_positivo, 0),
            inversiones_pesos_confirmadas
        ) AS share_plazo_fijo_inversion_pesos,

        ratio_seguro(
            COALESCE(_plazo_fijo_dolares_positivo, 0),
            inversiones_dolares_confirmadas
        ) AS share_plazo_fijo_inversion_dolares,

        CASE
            WHEN inversiones_pesos_confirmadas IS NULL THEN NULL
            ELSE (
                inversiones_pesos_confirmadas > 0
            )::TINYINT
        END AS tiene_inversion_pesos,

        CASE
            WHEN inversiones_dolares_confirmadas IS NULL THEN NULL
            ELSE (
                inversiones_dolares_confirmadas > 0
            )::TINYINT
        END AS tiene_inversion_dolares

    FROM componentes
),


lag_caja_seguridad AS (
    SELECT
        *,

        LAG(tiene_caja_seguridad) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS _tiene_caja_seguridad_lag1

    FROM derivadas
)


SELECT
    * EXCLUDE (
        _cuenta_corriente_positiva,
        _caja_ahorro_positiva,
        _plazo_fijo_pesos_positivo,
        _inversion1_pesos_positiva,
        _plazo_fijo_dolares_positivo,
        _inversion1_dolares_positiva,
        _tiene_caja_seguridad_lag1
    ),

    CASE
        WHEN tiene_historia_lag1 <> 1 THEN NULL

        WHEN _tiene_caja_seguridad_lag1 = 1
         AND tiene_caja_seguridad = 0
        THEN 1

        ELSE 0
    END AS apago_caja_seguridad_lag1

FROM lag_caja_seguridad;

,Success


In [38]:
%%sql

SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS claves_cliente_mes,

    COUNT(*) FILTER (
        WHERE share_cuenta_corriente_liquidez < 0
           OR share_cuenta_corriente_liquidez > 1

           OR share_prestamos_personales < 0
           OR share_prestamos_personales > 1

           OR share_prestamos_prendarios < 0
           OR share_prestamos_prendarios > 1

           OR share_prestamos_hipotecarios < 0
           OR share_prestamos_hipotecarios > 1

           OR share_deuda_tarjetas < 0
           OR share_deuda_tarjetas > 1

           OR share_deuda_prestamos < 0
           OR share_deuda_prestamos > 1

           OR share_deuda_descubierto < 0
           OR share_deuda_descubierto > 1

           OR share_plazo_fijo_inversion_pesos < 0
           OR share_plazo_fijo_inversion_pesos > 1

           OR share_plazo_fijo_inversion_dolares < 0
           OR share_plazo_fijo_inversion_dolares > 1
    ) AS shares_fuera_de_rango,

    COUNT(*) FILTER (
        WHERE prestamos_totales > 0
          AND ABS(
              share_prestamos_personales
              + share_prestamos_prendarios
              + share_prestamos_hipotecarios
              - 1
          ) > 0.000001
    ) AS composiciones_credito_invalidas,

    COUNT(*) FILTER (
        WHERE deuda_total_pesos > 0
          AND ABS(
              share_deuda_tarjetas
              + share_deuda_prestamos
              + share_deuda_descubierto
              - 1
          ) > 0.000001
    ) AS composiciones_deuda_invalidas,

    COUNT(*) FILTER (
        WHERE tiene_historia_lag1 <> 1
          AND apago_caja_seguridad_lag1 IS NOT NULL
    ) AS eventos_caja_sin_historia,

    COUNT(*) FILTER (
        WHERE mplazo_fijo_pesos < 0
           OR minversion1_pesos < 0
           OR mplazo_fijo_dolares < 0
           OR minversion1_dolares < 0
    ) AS filas_inversiones_negativas_originales

FROM competencia_05_productos_financieros_challenger;

,filas,claves_cliente_mes,shares_fuera_de_rango,composiciones_credito_invalidas,composiciones_deuda_invalidas,eventos_caja_sin_historia,filas_inversiones_negativas_originales
0,983061,983061,0,0,0,0,0


In [39]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(saldo_promedio_por_cuenta) / COUNT(*),
        2
    ) AS cobertura_saldo_promedio_cuenta,

    ROUND(
        100.0 * COUNT(prestamos_promedio_por_credito) / COUNT(*),
        2
    ) AS cobertura_credito_promedio,

    ROUND(
        100.0 * COUNT(inversiones_pesos_confirmadas) / COUNT(*),
        2
    ) AS cobertura_inversiones_pesos,

    ROUND(
        100.0 * COUNT(inversiones_dolares_confirmadas) / COUNT(*),
        2
    ) AS cobertura_inversiones_dolares,

    ROUND(
        100.0 * AVG(tiene_caja_seguridad),
        2
    ) AS porcentaje_caja_seguridad,

    MEDIAN(saldo_promedio_por_cuenta)
        AS mediana_saldo_por_cuenta,

    MEDIAN(prestamos_promedio_por_credito)
        AS mediana_prestamo_por_credito,

    QUANTILE_CONT(
        prestamos_promedio_por_credito,
        0.99
    ) AS p99_prestamo_por_credito,

    MEDIAN(inversiones_pesos_confirmadas)
        AS mediana_inversiones_pesos,

    QUANTILE_CONT(
        inversiones_pesos_confirmadas,
        0.99
    ) AS p99_inversiones_pesos

FROM competencia_05_productos_financieros_challenger
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,cobertura_saldo_promedio_cuenta,cobertura_credito_promedio,cobertura_inversiones_pesos,cobertura_inversiones_dolares,porcentaje_caja_seguridad,mediana_saldo_por_cuenta,mediana_prestamo_por_credito,p99_prestamo_por_credito,mediana_inversiones_pesos,p99_inversiones_pesos
0,202103,162900,100.0,23.45,100.0,100.0,6.74,10816.394583,39768.515000,3.018658e+06,0.0,352487.3798
1,202104,163284,100.0,23.60,100.0,100.0,6.76,10491.581667,40741.515000,9.209241e+05,0.0,375015.3785
2,202105,163768,100.0,23.41,100.0,100.0,6.76,10412.798333,42224.426250,9.176073e+05,0.0,424956.5494
3,202106,164114,100.0,23.16,100.0,100.0,6.78,15378.540000,43480.423750,9.590951e+05,0.0,495516.9711
4,202107,164348,100.0,22.77,100.0,100.0,6.79,15567.806667,44616.696667,1.005876e+06,0.0,551250.1318
5,202108,164647,100.0,22.13,100.0,100.0,6.80,12200.113333,48978.770000,1.064602e+06,0.0,582868.2928


El 100% informado como “cobertura de inversiones” no significa que todos tengan inversiones: significa que las columnas contienen 0 en lugar de NULL.

In [41]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(saldo_promedio_por_cuenta) / COUNT(*),
        2
    ) AS cobertura_saldo_promedio_cuenta,

    ROUND(
        100.0 * COUNT(prestamos_promedio_por_credito) / COUNT(*),
        2
    ) AS cobertura_credito_promedio,

    ROUND(
        100.0 * AVG(
            CAST(
                inversiones_pesos_confirmadas > 0
                AS INTEGER
            )
        ),
        2
    ) AS porcentaje_con_inversion_pesos,

    ROUND(
        100.0 * AVG(
            CAST(
                inversiones_dolares_confirmadas > 0
                AS INTEGER
            )
        ),
        2
    ) AS porcentaje_con_inversion_dolares,

    ROUND(
        100.0 * AVG(tiene_caja_seguridad),
        2
    ) AS porcentaje_caja_seguridad,

    MEDIAN(saldo_promedio_por_cuenta)
        AS mediana_saldo_por_cuenta,

    MEDIAN(prestamos_promedio_por_credito)
        AS mediana_prestamo_por_credito,

    QUANTILE_CONT(
        prestamos_promedio_por_credito,
        0.99
    ) AS p99_prestamo_por_credito,

    MEDIAN(inversiones_pesos_confirmadas)
        AS mediana_inversiones_pesos,

    QUANTILE_CONT(
        inversiones_pesos_confirmadas,
        0.99
    ) AS p99_inversiones_pesos

FROM competencia_05_productos_financieros_challenger

GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,cobertura_saldo_promedio_cuenta,cobertura_credito_promedio,porcentaje_con_inversion_pesos,porcentaje_con_inversion_dolares,porcentaje_caja_seguridad,mediana_saldo_por_cuenta,mediana_prestamo_por_credito,p99_prestamo_por_credito,mediana_inversiones_pesos,p99_inversiones_pesos
0,202103,162900,100.0,23.45,3.57,12.25,6.74,10816.394583,39768.515000,3.018658e+06,0.0,352487.3798
1,202104,163284,100.0,23.60,3.73,12.65,6.76,10491.581667,40741.515000,9.209241e+05,0.0,375015.3785
2,202105,163768,100.0,23.41,3.99,12.78,6.76,10412.798333,42224.426250,9.176073e+05,0.0,424956.5494
3,202106,164114,100.0,23.16,4.28,13.20,6.78,15378.540000,43480.423750,9.590951e+05,0.0,495516.9711
4,202107,164348,100.0,22.77,4.55,13.80,6.79,15567.806667,44616.696667,1.005876e+06,0.0,551250.1318
5,202108,164647,100.0,22.13,4.70,13.45,6.80,12200.113333,48978.770000,1.064602e+06,0.0,582868.2928


In [42]:
%%sql

-- ============================================================
-- BLOQUE 5: FLUJOS, TRANSFERENCIAS, CANALES Y CHEQUES
-- ============================================================

CREATE OR REPLACE TABLE
    competencia_06_flujos_canales_challenger AS

WITH componentes AS (
    SELECT
        *,

        -- ====================================================
        -- INGRESOS Y EGRESOS OBSERVADOS
        -- ====================================================

        suma_sin_null([
            mpayroll,
            mtransferencias_recibidas,
            mcheques_depositados
        ]) AS ingresos_observados,

        suma_sin_null([
            mtransferencias_emitidas,
            mextraccion_autoservicio,
            matm,
            matm_other,
            mcheques_emitidos,
            mpagomiscuentas,
            mpagodeservicios
        ]) AS egresos_observados,

        -- ====================================================
        -- TRANSFERENCIAS
        -- ====================================================

        suma_sin_null([
            mtransferencias_recibidas,
            -mtransferencias_emitidas
        ]) AS balance_transferencias,

        -- ====================================================
        -- CANALES
        -- ====================================================

        suma_sin_null([
            chomebanking_transacciones,
            cmobile_app_trx
        ]) AS transacciones_digitales_total,

        suma_sin_null([
            catm_trx,
            catm_trx_other
        ]) AS transacciones_atm_total,

        suma_sin_null([
            chomebanking_transacciones,
            cmobile_app_trx,
            catm_trx,
            catm_trx_other,
            ccajas_transacciones,
            ccallcenter_transacciones
        ]) AS transacciones_canales_total,

        suma_sin_null([
            matm,
            matm_other
        ]) AS monto_atm_total,

        -- ====================================================
        -- CHEQUES
        -- ====================================================

        suma_sin_null([
            ccheques_depositados,
            ccheques_emitidos
        ]) AS cheques_cantidad_total,

        suma_sin_null([
            mcheques_depositados,
            mcheques_emitidos
        ]) AS cheques_monto_total,

        suma_sin_null([
            ccheques_depositados_rechazados,
            ccheques_emitidos_rechazados
        ]) AS cheques_rechazados_cantidad_total,

        suma_sin_null([
            mcheques_depositados_rechazados,
            mcheques_emitidos_rechazados
        ]) AS cheques_rechazados_monto_total

    FROM competencia_05_productos_financieros_challenger
),


indicadores AS (
    SELECT
        *,

        -- ====================================================
        -- FLUJO
        -- ====================================================

        CASE
            WHEN ingresos_observados IS NULL
              OR egresos_observados IS NULL
            THEN NULL
            ELSE ingresos_observados - egresos_observados
        END AS flujo_observado_neto,

        ratio_seguro(
            egresos_observados,
            payroll_referencia_2m
        ) AS velocidad_vaciado_sobre_payroll,

        ratio_seguro(
            egresos_observados,
            ingresos_observados
        ) AS egresos_sobre_ingresos,

        ratio_seguro(
            mcuentas_saldo,
            ingresos_observados
        ) AS saldo_cuentas_sobre_ingresos,

        -- ====================================================
        -- TRANSFERENCIAS
        -- ====================================================

        ratio_seguro(
            mtransferencias_emitidas,
            mtransferencias_recibidas
        ) AS transferencias_emitidas_sobre_recibidas,

        ratio_seguro(
            mtransferencias_emitidas,
            ingresos_observados
        ) AS transferencias_emitidas_sobre_ingresos,

        ratio_seguro(
            mtransferencias_recibidas,
            ctransferencias_recibidas
        ) AS ticket_transferencias_recibidas,

        ratio_seguro(
            mtransferencias_emitidas,
            ctransferencias_emitidas
        ) AS ticket_transferencias_emitidas,

        -- ====================================================
        -- CANALES
        -- ====================================================

        ratio_seguro(
            catm_trx_other,
            transacciones_atm_total
        ) AS share_atm_ajeno_cantidad,

        ratio_seguro(
            matm_other,
            monto_atm_total
        ) AS share_atm_ajeno_monto,

        ratio_seguro(
            transacciones_digitales_total,
            transacciones_canales_total
        ) AS share_digital_transacciones,

        ratio_seguro(
            cmobile_app_trx,
            transacciones_digitales_total
        ) AS share_app_dentro_digital,

        ratio_seguro(
            ccajas_transacciones,
            transacciones_canales_total
        ) AS share_sucursal_transacciones,

        ratio_seguro(
            transacciones_canales_total,
            cantidad_canales_usados
        ) AS operaciones_por_canal_activo,

        -- ====================================================
        -- CHEQUES RECHAZADOS
        -- ====================================================

        ratio_seguro(
            ccheques_depositados_rechazados,
            ccheques_depositados
        ) AS tasa_cheques_depositados_rechazados,

        ratio_seguro(
            ccheques_emitidos_rechazados,
            ccheques_emitidos
        ) AS tasa_cheques_emitidos_rechazados,

        ratio_seguro(
            cheques_rechazados_cantidad_total,
            cheques_cantidad_total
        ) AS tasa_cheques_rechazados_total,

        ratio_seguro(
            mcheques_depositados_rechazados,
            mcheques_depositados
        ) AS monto_cheques_depositados_rechazado_relativo,

        ratio_seguro(
            mcheques_emitidos_rechazados,
            mcheques_emitidos
        ) AS monto_cheques_emitidos_rechazado_relativo,

        ratio_seguro(
            cheques_rechazados_monto_total,
            cheques_monto_total
        ) AS monto_cheques_rechazado_relativo_total,

        CASE
            WHEN cheques_rechazados_cantidad_total IS NULL
            THEN NULL

            ELSE CAST(
                cheques_rechazados_cantidad_total > 0
                AS TINYINT
            )
        END AS tuvo_cheques_rechazados

    FROM componentes
),


lags_eventos AS (
    SELECT
        *,

        LAG(catm_trx) OVER w
            AS _catm_trx_lag1,

        LAG(catm_trx_other) OVER w
            AS _catm_trx_other_lag1,

        LAG(ccallcenter_transacciones) OVER w
            AS _callcenter_lag1

    FROM indicadores

    WINDOW w AS (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    )
)


SELECT
    * EXCLUDE (
        _catm_trx_lag1,
        _catm_trx_other_lag1,
        _callcenter_lag1
    ),

    -- Baja el uso de ATM propio y aumenta el ATM ajeno.
    CASE
        WHEN tiene_historia_lag1 <> 1 THEN NULL

        WHEN catm_trx IS NULL
          OR catm_trx_other IS NULL
          OR _catm_trx_lag1 IS NULL
          OR _catm_trx_other_lag1 IS NULL
        THEN NULL

        WHEN catm_trx < _catm_trx_lag1
         AND catm_trx_other > _catm_trx_other_lag1
        THEN 1

        ELSE 0
    END AS migracion_atm_ajeno_lag1,

    -- Comienza a llamar al call center.
    CASE
        WHEN tiene_historia_lag1 <> 1 THEN NULL

        WHEN ccallcenter_transacciones IS NULL
          OR _callcenter_lag1 IS NULL
        THEN NULL

        WHEN _callcenter_lag1 = 0
         AND ccallcenter_transacciones > 0
        THEN 1

        ELSE 0
    END AS empezo_callcenter_lag1

FROM lags_eventos;

,Success


In [43]:
%%sql

SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS claves_cliente_mes,

    COUNT(*) FILTER (
        WHERE share_atm_ajeno_cantidad < 0
           OR share_atm_ajeno_cantidad > 1

           OR share_atm_ajeno_monto < 0
           OR share_atm_ajeno_monto > 1

           OR share_digital_transacciones < 0
           OR share_digital_transacciones > 1

           OR share_app_dentro_digital < 0
           OR share_app_dentro_digital > 1

           OR share_sucursal_transacciones < 0
           OR share_sucursal_transacciones > 1

           OR tasa_cheques_depositados_rechazados < 0
           OR tasa_cheques_depositados_rechazados > 1

           OR tasa_cheques_emitidos_rechazados < 0
           OR tasa_cheques_emitidos_rechazados > 1

           OR tasa_cheques_rechazados_total < 0
           OR tasa_cheques_rechazados_total > 1

           OR monto_cheques_depositados_rechazado_relativo < 0
           OR monto_cheques_depositados_rechazado_relativo > 1

           OR monto_cheques_emitidos_rechazado_relativo < 0
           OR monto_cheques_emitidos_rechazado_relativo > 1

           OR monto_cheques_rechazado_relativo_total < 0
           OR monto_cheques_rechazado_relativo_total > 1
    ) AS proporciones_fuera_de_rango,

    COUNT(*) FILTER (
        WHERE flujo_observado_neto IS NOT NULL
          AND ABS(
              flujo_observado_neto
              - (ingresos_observados - egresos_observados)
          ) > 0.01
    ) AS errores_flujo_neto,

    COUNT(*) FILTER (
        WHERE tiene_historia_lag1 <> 1
          AND (
              migracion_atm_ajeno_lag1 IS NOT NULL
              OR empezo_callcenter_lag1 IS NOT NULL
          )
    ) AS eventos_informados_sin_historia,

    COUNT(*) FILTER (
        WHERE ingresos_observados < 0
           OR egresos_observados < 0
    ) AS flujos_brutos_negativos

FROM competencia_06_flujos_canales_challenger;

,filas,claves_cliente_mes,proporciones_fuera_de_rango,errores_flujo_neto,eventos_informados_sin_historia,flujos_brutos_negativos
0,983061,983061,160,0,0,0


In [44]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * AVG(
            CAST(ingresos_observados > 0 AS INTEGER)
        ),
        2
    ) AS porcentaje_con_ingresos,

    ROUND(
        100.0 * AVG(
            CAST(egresos_observados > 0 AS INTEGER)
        ),
        2
    ) AS porcentaje_con_egresos,

    ROUND(
        100.0 * AVG(
            CAST(transacciones_digitales_total > 0 AS INTEGER)
        ),
        2
    ) AS porcentaje_actividad_digital,

    ROUND(
        100.0 * AVG(
            CAST(transacciones_atm_total > 0 AS INTEGER)
        ),
        2
    ) AS porcentaje_actividad_atm,

    ROUND(
        100.0 * AVG(
            CAST(tuvo_cheques_rechazados = 1 AS INTEGER)
        ),
        3
    ) AS porcentaje_cheques_rechazados,

    MEDIAN(ingresos_observados)
        AS mediana_ingresos,

    MEDIAN(egresos_observados)
        AS mediana_egresos,

    MEDIAN(flujo_observado_neto)
        AS mediana_flujo_neto,

    ROUND(
        100.0 * AVG(
            CAST(flujo_observado_neto < 0 AS INTEGER)
        ),
        2
    ) AS porcentaje_flujo_neto_negativo,

    MEDIAN(egresos_sobre_ingresos)
        AS mediana_egresos_sobre_ingresos,

    QUANTILE_CONT(
        egresos_sobre_ingresos,
        0.99
    ) AS p99_egresos_sobre_ingresos,

    MEDIAN(share_digital_transacciones)
        AS mediana_share_digital

FROM competencia_06_flujos_canales_challenger
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,porcentaje_con_ingresos,porcentaje_con_egresos,porcentaje_actividad_digital,porcentaje_actividad_atm,porcentaje_cheques_rechazados,mediana_ingresos,mediana_egresos,mediana_flujo_neto,porcentaje_flujo_neto_negativo,mediana_egresos_sobre_ingresos,p99_egresos_sobre_ingresos,mediana_share_digital
0,202103,162900,86.50,80.94,93.97,51.79,0.362,110079.005,57975.565,29325.000,25.21,0.498878,20.000000,0.972973
1,202104,163284,85.71,80.73,93.93,50.83,0.355,107194.995,55463.225,27827.305,25.68,0.499397,22.552742,0.974359
2,202105,163768,85.85,80.49,94.13,49.22,0.330,111171.025,55074.985,29325.000,25.31,0.489796,22.148443,0.978495
3,202106,164114,87.13,80.85,94.17,49.65,0.338,156526.875,61372.905,59003.355,19.64,0.391784,17.329347,0.976190
4,202107,164348,87.28,81.19,94.25,51.04,0.316,134363.625,71091.820,32036.915,26.61,0.525081,19.606744,0.973214
5,202108,164647,87.50,81.78,94.44,50.46,0.333,135207.680,68731.720,37536.000,24.78,0.496398,20.574575,0.976190


In [45]:
%%sql

-- ============================================================
-- BLOQUE 6: RECURRENTES, COMISIONES, RENTABILIDAD,
--           DESCUENTOS Y FOREX
-- ============================================================

CREATE OR REPLACE TABLE
    competencia_07_recurrentes_rentabilidad_challenger AS

WITH componentes AS (
    SELECT
        *,

        -- ====================================================
        -- COMPROMISOS RECURRENTES
        -- ====================================================

        suma_sin_null([
            mttarjeta_visa_debitos_automaticos,
            mttarjeta_master_debitos_automaticos
        ]) AS monto_debitos_automaticos_tarjetas,

        suma_sin_null([
            mpagodeservicios,
            mpagomiscuentas
        ]) AS monto_pagos_recurrentes_servicios,

        suma_sin_null([
            mcuenta_debitos_automaticos,
            mttarjeta_visa_debitos_automaticos,
            mttarjeta_master_debitos_automaticos,
            mpagodeservicios,
            mpagomiscuentas
        ]) AS monto_recurrente_total,

        -- ====================================================
        -- COMISIONES OBSERVADAS
        -- ====================================================

        suma_sin_null([
            ccomisiones_mantenimiento,
            ccomisiones_otras
        ]) AS comisiones_cantidad_observada,

        suma_sin_null([
            mcomisiones_mantenimiento,
            mcomisiones_otras
        ]) AS comisiones_monto_observado,

        -- ====================================================
        -- DESCUENTOS
        -- ====================================================

        suma_sin_null([
            ctarjeta_visa_descuentos,
            ctarjeta_master_descuentos
        ]) AS descuentos_tarjetas_cantidad_total,

        suma_sin_null([
            ctarjeta_visa_descuentos,
            ctarjeta_master_descuentos,
            ccajeros_propios_descuentos
        ]) AS descuentos_cantidad_total,

        suma_sin_null([
            mtarjeta_visa_descuentos,
            mtarjeta_master_descuentos,
            mcajeros_propios_descuentos
        ]) AS descuentos_monto_total,

        -- ====================================================
        -- FOREX
        -- ====================================================

        suma_sin_null([
            mforex_buy,
            mforex_sell
        ]) AS forex_monto_total

    FROM competencia_06_flujos_canales_challenger
),


indicadores AS (
    SELECT
        *,

        -- ====================================================
        -- RECURRENTES
        -- ====================================================

        ratio_seguro(
            monto_recurrente_total,
            coperaciones_recurrentes_total
        ) AS ticket_recurrente,

        ratio_seguro(
            mcuenta_debitos_automaticos,
            monto_recurrente_total
        ) AS share_recurrente_debitos_cuenta,

        ratio_seguro(
            monto_debitos_automaticos_tarjetas,
            monto_recurrente_total
        ) AS share_recurrente_debitos_tarjetas,

        ratio_seguro(
            monto_pagos_recurrentes_servicios,
            monto_recurrente_total
        ) AS share_recurrente_pagos_servicios,

        ratio_seguro(
            monto_recurrente_total,
            payroll_referencia_2m
        ) AS monto_recurrente_sobre_payroll,

        -- ====================================================
        -- COMISIONES Y RENTABILIDAD
        -- ====================================================

        ratio_seguro(
            mcomisiones_mantenimiento,
            ccomisiones_mantenimiento
        ) AS comision_mantenimiento_por_evento,

        ratio_seguro(
            mcomisiones_otras,
            ccomisiones_otras
        ) AS comision_otra_por_evento,

        ratio_seguro(
            comisiones_monto_observado,
            comisiones_cantidad_observada
        ) AS comision_observada_por_evento,

        ratio_seguro(
            mcomisiones_mantenimiento,
            cproductos
        ) AS mantenimiento_por_producto,

        ratio_seguro(
            mrentabilidad,
            cproductos
        ) AS rentabilidad_por_producto,

        mrentabilidad
            - mrentabilidad_annual / 12.0
            AS desvio_rentabilidad_run_rate,

        ratio_seguro(
            mactivos_margen,
            payroll_referencia_2m
        ) AS margen_activo_sobre_payroll,

        ratio_seguro(
            mpasivos_margen,
            payroll_referencia_2m
        ) AS margen_pasivo_sobre_payroll,

        -- ====================================================
        -- DESCUENTOS
        -- ====================================================

        ratio_seguro(
            descuentos_monto_total,
            descuentos_cantidad_total
        ) AS descuento_promedio,

        -- No se llaman shares porque pueden superar 1:
        -- una transacción puede tener más de un descuento.
        ratio_seguro(
            ctarjeta_visa_descuentos,
            ctarjeta_visa_transacciones
        ) AS descuentos_visa_por_transaccion,

        ratio_seguro(
            ctarjeta_master_descuentos,
            ctarjeta_master_transacciones
        ) AS descuentos_master_por_transaccion,

        ratio_seguro(
            descuentos_tarjetas_cantidad_total,
            ctarjetas_credito_transacciones_total
        ) AS descuentos_tarjetas_por_transaccion,

        CASE
            WHEN descuentos_cantidad_total IS NULL
            THEN NULL

            ELSE CAST(
                descuentos_cantidad_total > 0
                AS TINYINT
            )
        END AS utilizo_descuentos,

        -- ====================================================
        -- FOREX
        -- ====================================================

        ratio_seguro(
            cforex_buy,
            cforex
        ) AS share_forex_compra_cantidad,

        ratio_seguro(
            cforex_sell,
            cforex
        ) AS share_forex_venta_cantidad,

        ratio_seguro(
            mforex_buy,
            cforex_buy
        ) AS ticket_forex_compra,

        ratio_seguro(
            mforex_sell,
            cforex_sell
        ) AS ticket_forex_venta,

        CASE
            WHEN mforex_buy IS NULL
              OR mforex_sell IS NULL
            THEN NULL

            ELSE mforex_buy - mforex_sell
        END AS balance_forex,

        CASE
            WHEN cforex IS NULL THEN NULL

            ELSE CAST(
                cforex > 0
                AS TINYINT
            )
        END AS opero_forex

    FROM componentes
)


SELECT *
FROM indicadores;

,Success


In [46]:
%%sql

SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS claves_cliente_mes,

    COUNT(*) FILTER (
        WHERE share_recurrente_debitos_cuenta < 0
           OR share_recurrente_debitos_cuenta > 1

           OR share_recurrente_debitos_tarjetas < 0
           OR share_recurrente_debitos_tarjetas > 1

           OR share_recurrente_pagos_servicios < 0
           OR share_recurrente_pagos_servicios > 1

           OR share_forex_compra_cantidad < 0
           OR share_forex_compra_cantidad > 1

           OR share_forex_venta_cantidad < 0
           OR share_forex_venta_cantidad > 1
    ) AS shares_fuera_de_rango,

    COUNT(*) FILTER (
        WHERE monto_recurrente_total > 0
          AND ABS(
              share_recurrente_debitos_cuenta
              + share_recurrente_debitos_tarjetas
              + share_recurrente_pagos_servicios
              - 1
          ) > 0.000001
    ) AS composiciones_recurrentes_invalidas,

    COUNT(*) FILTER (
        WHERE cforex > 0
          AND ABS(
              share_forex_compra_cantidad
              + share_forex_venta_cantidad
              - 1
          ) > 0.000001
    ) AS composiciones_forex_invalidas,

    COUNT(*) FILTER (
        WHERE balance_forex IS NOT NULL
          AND ABS(
              balance_forex
              - (mforex_buy - mforex_sell)
          ) > 0.01
    ) AS errores_balance_forex,

    -- Informativo: puede ser mayor que cero.
    COUNT(*) FILTER (
        WHERE descuentos_visa_por_transaccion > 1
           OR descuentos_master_por_transaccion > 1
           OR descuentos_tarjetas_por_transaccion > 1
    ) AS descuentos_multiples_por_transaccion,

    -- Informativo: pueden existir devoluciones o ajustes.
    COUNT(*) FILTER (
        WHERE mcomisiones < 0
           OR mcomisiones_mantenimiento < 0
           OR mcomisiones_otras < 0
    ) AS filas_comisiones_negativas

FROM competencia_07_recurrentes_rentabilidad_challenger;

,filas,claves_cliente_mes,shares_fuera_de_rango,composiciones_recurrentes_invalidas,composiciones_forex_invalidas,errores_balance_forex,descuentos_multiples_por_transaccion,filas_comisiones_negativas
0,983061,983061,0,0,0,0,216,24769


In [47]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * AVG(
            CAST(monto_recurrente_total > 0 AS INTEGER)
        ),
        2
    ) AS porcentaje_con_monto_recurrente,

    ROUND(
        100.0 * AVG(
            CAST(utilizo_descuentos = 1 AS INTEGER)
        ),
        2
    ) AS porcentaje_con_descuentos,

    ROUND(
        100.0 * AVG(
            CAST(opero_forex = 1 AS INTEGER)
        ),
        2
    ) AS porcentaje_con_forex,

    MEDIAN(monto_recurrente_total)
        AS mediana_monto_recurrente,

    MEDIAN(ticket_recurrente)
        AS mediana_ticket_recurrente,

    MEDIAN(monto_recurrente_sobre_payroll)
        AS mediana_recurrente_sobre_payroll,

    MEDIAN(comision_observada_por_evento)
        AS mediana_comision_por_evento,

    MEDIAN(rentabilidad_por_producto)
        AS mediana_rentabilidad_por_producto,

    MEDIAN(desvio_rentabilidad_run_rate)
        AS mediana_desvio_run_rate,

    MEDIAN(descuento_promedio)
        AS mediana_descuento_promedio,

    MEDIAN(share_forex_compra_cantidad)
        AS mediana_share_forex_compra,

    QUANTILE_CONT(
        ticket_forex_compra,
        0.99
    ) AS p99_ticket_forex_compra

FROM competencia_07_recurrentes_rentabilidad_challenger
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,porcentaje_con_monto_recurrente,porcentaje_con_descuentos,porcentaje_con_forex,mediana_monto_recurrente,mediana_ticket_recurrente,mediana_recurrente_sobre_payroll,mediana_comision_por_evento,mediana_rentabilidad_por_producto,mediana_desvio_run_rate,mediana_descuento_promedio,mediana_share_forex_compra,p99_ticket_forex_compra
0,202103,162900,84.29,15.64,4.67,8886.945,2345.271111,0.088103,120.716190,168.937302,125.237500,83.570000,0.0,1759.5000
1,202104,163284,85.80,11.65,4.01,10405.105,2345.817000,0.096464,121.499000,151.473214,14.948333,133.383333,0.0,2645.1150
2,202105,163768,85.60,11.40,3.93,10804.425,2462.361765,0.100802,116.913289,164.567500,49.094167,80.448333,0.0,1312.4650
3,202106,164114,85.62,12.89,5.49,10388.995,2335.913667,0.078822,130.728333,182.884500,143.295417,253.800000,0.0,2346.1904
4,202107,164348,86.10,13.45,8.07,11935.000,2599.241364,0.086469,138.668239,229.255208,443.867500,175.950000,0.0,1627.3849
5,202108,164647,86.26,11.99,8.04,12328.850,2677.286667,0.101379,145.423333,249.305000,504.315833,109.080000,0.0,1876.8000


# Features 

In [49]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

SEMILLAS = [
    100129,
    100183,
    100151,
    500041,
    900007,
]

FOLDS_TEMPORALES = [
    {
        "entrenamiento": [202103],
        "validacion": 202105,
    },
    {
        "entrenamiento": [202103, 202104],
        "validacion": 202106,
    },
]

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500

BREAK_EVEN = (
    COSTO_ESTIMULO
    / (GANANCIA_ACIERTO + COSTO_ESTIMULO)
)


# ============================================================
# TABLAS DE CADA ETAPA
# ============================================================

secuencia_bloques = [
    (
        "B1_ANCLAJE_DINAMICA",
        "competencia_01",
        "competencia_02_features",
    ),
    (
        "B2_CARGA_FINANCIERA",
        "competencia_02_features",
        "competencia_03_carga_financiera",
    ),
    (
        "B3_TARJETAS",
        "competencia_03_carga_financiera",
        "competencia_04_tarjetas_challenger",
    ),
    (
        "B4_PRODUCTOS_FINANCIEROS",
        "competencia_04_tarjetas_challenger",
        "competencia_05_productos_financieros_challenger",
    ),
    (
        "B5_FLUJOS_CANALES",
        "competencia_05_productos_financieros_challenger",
        "competencia_06_flujos_canales_challenger",
    ),
    (
        "B6_RECURRENTES_RENTABILIDAD",
        "competencia_06_flujos_canales_challenger",
        "competencia_07_recurrentes_rentabilidad_challenger",
    ),
]


# ============================================================
# LEER ESQUEMAS
# ============================================================

esquema = %sql SELECT table_name, column_name, ordinal_position FROM information_schema.columns ORDER BY table_name, ordinal_position


tablas_necesarias = {
    "competencia_01",
    *[
        tabla
        for _, tabla_anterior, tabla_actual
        in secuencia_bloques
        for tabla in [tabla_anterior, tabla_actual]
    ],
}

tablas_disponibles = set(
    esquema["table_name"].tolist()
)

tablas_faltantes = (
    tablas_necesarias
    - tablas_disponibles
)

assert not tablas_faltantes, (
    f"Faltan estas tablas: {sorted(tablas_faltantes)}"
)


columnas_por_tabla = {}

for tabla in tablas_necesarias:

    columnas_por_tabla[tabla] = (
        esquema.loc[
            esquema["table_name"] == tabla,
            ["column_name", "ordinal_position"]
        ]
        .sort_values("ordinal_position")
        ["column_name"]
        .tolist()
    )


# ============================================================
# COLUMNAS EXCLUIDAS
# ============================================================

columnas_excluidas = {
    "numero_de_cliente",
    "foto_mes",
    "clase_ternaria",
    "target_baja2",

    # Fechas todavía no transformadas.
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_fultimo_cierre",
    "Master_fechaalta",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_fultimo_cierre",
    "Visa_fechaalta",

    # Si existieran indicadores técnicos de corrección.
    "Master_limite_financiacion_fue_corregido",
    "Visa_limite_financiacion_fue_corregido",
}


# ============================================================
# FEATURES ORIGINALES Y FEATURES NUEVAS POR BLOQUE
# ============================================================

features_originales = [
    columna
    for columna in columnas_por_tabla["competencia_01"]
    if columna not in columnas_excluidas
]

features_por_bloque = {}

for (
    nombre_bloque,
    tabla_anterior,
    tabla_actual,
) in secuencia_bloques:

    columnas_anteriores = set(
        columnas_por_tabla[tabla_anterior]
    )

    features_nuevas = [
        columna
        for columna in columnas_por_tabla[tabla_actual]
        if columna not in columnas_anteriores
        and columna not in columnas_excluidas
    ]

    assert features_nuevas, (
        f"No se detectaron features para {nombre_bloque}"
    )

    features_por_bloque[nombre_bloque] = (
        features_nuevas
    )


# Cada bloque se compara individualmente contra el baseline.
features_experimentos = {
    "BASE": features_originales,
}

for nombre_bloque, features_nuevas in (
    features_por_bloque.items()
):
    features_experimentos[nombre_bloque] = (
        features_originales
        + features_nuevas
    )


# ============================================================
# LEER LOS CUATRO MESES CON TARGET MADURO
# ============================================================

base_modelo = %sql SELECT * FROM competencia_07_recurrentes_rentabilidad_challenger WHERE foto_mes BETWEEN 202103 AND 202106 AND clase_ternaria IS NOT NULL ORDER BY foto_mes, numero_de_cliente


base_modelo["target_baja2"] = (
    base_modelo["clase_ternaria"]
    .eq("BAJA+2")
    .astype("int8")
)


# Todas las columnas que utilizará al menos un experimento.
features_utilizadas = list(
    dict.fromkeys(
        feature
        for lista_features in features_experimentos.values()
        for feature in lista_features
    )
)


# LightGBM admite NULL, pero no infinitos.
base_modelo[features_utilizadas] = (
    base_modelo[features_utilizadas]
    .replace([np.inf, -np.inf], np.nan)
)


columnas_no_numericas = (
    base_modelo[features_utilizadas]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

assert columnas_no_numericas == [], (
    "Hay columnas no numéricas: "
    f"{columnas_no_numericas}"
)


print("Filas para evaluación:", len(base_modelo))
print("Features originales:", len(features_originales))

for nombre, features_nuevas in features_por_bloque.items():
    print(
        nombre,
        "| features nuevas:",
        len(features_nuevas),
    )

Filas para evaluación: 654066
Features originales: 144
B1_ANCLAJE_DINAMICA | features nuevas: 81
B2_CARGA_FINANCIERA | features nuevas: 12
B3_TARJETAS | features nuevas: 16
B4_PRODUCTOS_FINANCIEROS | features nuevas: 21
B5_FLUJOS_CANALES | features nuevas: 34
B6_RECURRENTES_RENTABILIDAD | features nuevas: 33


In [50]:
# ============================================================
# HIPERPARÁMETROS BASE CONGELADOS
# ============================================================

PARAMETROS_BASE = {
    "objective": "binary",
    "metric": "None",
    "boosting_type": "gbdt",
    "boost_from_average": True,
    "feature_pre_filter": False,
    "first_metric_only": True,
    "verbosity": -1,

    "max_depth": 8,
    "num_leaves": 48,
    "learning_rate": 0.011968361180129191,
    "min_data_in_leaf": 405,
    "feature_fraction": 0.30152141273199784,
    "bagging_fraction": 0.7006936995574339,
    "bagging_freq": 1,
    "lambda_l1": 8.320444953703719e-05,
    "lambda_l2": 0.0006696837335733366,
    "min_gain_to_split": 1.5614099380945257,
    "max_bin": 255,
}


# ============================================================
# FUNCIONES DE GANANCIA
# ============================================================

def ganancia_por_fila(clase_ternaria):

    return np.where(
        np.asarray(clase_ternaria) == "BAJA+2",
        GANANCIA_ACIERTO,
        -COSTO_ESTIMULO,
    )


def calcular_ganancia(
    probabilidades,
    ganancias_fila,
):

    probabilidades = np.asarray(
        probabilidades
    )

    ganancias_fila = np.asarray(
        ganancias_fila
    )

    seleccionados = (
        probabilidades >= BREAK_EVEN
    )

    return float(
        ganancias_fila[seleccionados].sum()
    )


def crear_metrica_ganancia(
    ganancias_validacion
):

    ganancias_validacion = np.asarray(
        ganancias_validacion
    )

    def metrica(
        probabilidades,
        dataset_lgb,
    ):

        assert (
            dataset_lgb.num_data()
            == len(ganancias_validacion)
        )

        valor = calcular_ganancia(
            probabilidades,
            ganancias_validacion,
        )

        return "ganancia", valor, True

    return metrica


# ============================================================
# FUNCIÓN DE EVALUACIÓN
# ============================================================

def evaluar_experimento(
    datos,
    features,
    nombre_experimento,
    cantidad_features_nuevas,
):

    resultados = []

    for semilla in SEMILLAS:

        parametros = {
            **PARAMETROS_BASE,

            "seed": semilla,
            "feature_fraction_seed": semilla,
            "bagging_seed": semilla,
            "data_random_seed": semilla,
        }

        for fold in FOLDS_TEMPORALES:

            mascara_train = (
                datos["foto_mes"]
                .isin(fold["entrenamiento"])
            )

            mascara_validacion = (
                datos["foto_mes"]
                == fold["validacion"]
            )

            X_train = datos.loc[
                mascara_train,
                features,
            ]

            y_train = datos.loc[
                mascara_train,
                "target_baja2",
            ]

            X_validacion = datos.loc[
                mascara_validacion,
                features,
            ]

            y_validacion = datos.loc[
                mascara_validacion,
                "target_baja2",
            ]

            clases_validacion = datos.loc[
                mascara_validacion,
                "clase_ternaria",
            ]

            ganancias_validacion = (
                ganancia_por_fila(
                    clases_validacion
                )
            )

            dataset_train = lgb.Dataset(
                X_train,
                label=y_train,
            )

            dataset_validacion = lgb.Dataset(
                X_validacion,
                label=y_validacion,
                reference=dataset_train,
            )

            modelo = lgb.train(
                params=parametros,
                train_set=dataset_train,
                num_boost_round=3000,
                valid_sets=[dataset_validacion],
                valid_names=["validacion"],
                feval=crear_metrica_ganancia(
                    ganancias_validacion
                ),
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        verbose=False,
                    ),
                    lgb.log_evaluation(period=0),
                ],
            )

            probabilidades = modelo.predict(
                X_validacion,
                num_iteration=modelo.best_iteration,
            )

            ganancia = calcular_ganancia(
                probabilidades,
                ganancias_validacion,
            )

            auc = roc_auc_score(
                y_validacion,
                probabilidades,
            )

            resultados.append({
                "experimento":
                    nombre_experimento,

                "semilla":
                    semilla,

                "validacion":
                    fold["validacion"],

                "ganancia":
                    ganancia,

                "auc":
                    float(auc),

                "iteraciones":
                    int(modelo.best_iteration),

                "cantidad_features":
                    len(features),

                "features_nuevas":
                    cantidad_features_nuevas,
            })

            print(
                nombre_experimento,
                "| semilla",
                semilla,
                "| validación",
                fold["validacion"],
                "| ganancia",
                f"{ganancia:,.0f}",
            )

            del modelo
            del dataset_train
            del dataset_validacion
            del X_train
            del X_validacion

            gc.collect()

    return pd.DataFrame(resultados)


# ============================================================
# EJECUTAR TODOS LOS EXPERIMENTOS
# ============================================================

resultados_experimentos = []

for (
    nombre_experimento,
    features_experimento,
) in features_experimentos.items():

    if nombre_experimento == "BASE":
        cantidad_nuevas = 0
    else:
        cantidad_nuevas = len(
            features_por_bloque[
                nombre_experimento
            ]
        )

    resultado = evaluar_experimento(
        datos=base_modelo,
        features=features_experimento,
        nombre_experimento=nombre_experimento,
        cantidad_features_nuevas=cantidad_nuevas,
    )

    resultados_experimentos.append(
        resultado
    )


resultados_experimentos = pd.concat(
    resultados_experimentos,
    ignore_index=True,
)

resultados_experimentos.to_csv(
    "resultados_ablacion_competencia_v4.csv",
    index=False,
)

print("Evaluación finalizada.")

BASE | semilla 100129 | validación 202105 | ganancia 253,550,000
BASE | semilla 100129 | validación 202106 | ganancia 351,340,000
BASE | semilla 100183 | validación 202105 | ganancia 249,150,000
BASE | semilla 100183 | validación 202106 | ganancia 353,622,500
BASE | semilla 100151 | validación 202105 | ganancia 250,965,000
BASE | semilla 100151 | validación 202106 | ganancia 355,740,000
BASE | semilla 500041 | validación 202105 | ganancia 245,052,500
BASE | semilla 500041 | validación 202106 | ganancia 357,720,000
BASE | semilla 900007 | validación 202105 | ganancia 247,170,000
BASE | semilla 900007 | validación 202106 | ganancia 348,645,000
B1_ANCLAJE_DINAMICA | semilla 100129 | validación 202105 | ganancia 256,795,000
B1_ANCLAJE_DINAMICA | semilla 100129 | validación 202106 | ganancia 364,897,500
B1_ANCLAJE_DINAMICA | semilla 100183 | validación 202105 | ganancia 258,280,000
B1_ANCLAJE_DINAMICA | semilla 100183 | validación 202106 | ganancia 359,012,500
B1_ANCLAJE_DINAMICA | semilla 

In [51]:
# ============================================================
# COMPARACIÓN EMPAREJADA CONTRA BASELINE
# ============================================================

referencia_base = (
    resultados_experimentos.loc[
        resultados_experimentos[
            "experimento"
        ] == "BASE",
        [
            "semilla",
            "validacion",
            "ganancia",
            "auc",
        ],
    ]
    .rename(columns={
        "ganancia": "ganancia_base",
        "auc": "auc_base",
    })
)


resultados_bloques = (
    resultados_experimentos.loc[
        resultados_experimentos[
            "experimento"
        ] != "BASE"
    ]
    .merge(
        referencia_base,
        on=["semilla", "validacion"],
        how="inner",
    )
)


resultados_bloques["delta_ganancia"] = (
    resultados_bloques["ganancia"]
    - resultados_bloques["ganancia_base"]
)

resultados_bloques["delta_auc"] = (
    resultados_bloques["auc"]
    - resultados_bloques["auc_base"]
)


# ============================================================
# RESULTADO POR MES
# ============================================================

resumen_por_mes = (
    resultados_bloques
    .groupby([
        "experimento",
        "validacion",
    ])
    .agg(
        ganancia_base=(
            "ganancia_base",
            "mean",
        ),
        ganancia_bloque=(
            "ganancia",
            "mean",
        ),
        delta_ganancia_media=(
            "delta_ganancia",
            "mean",
        ),
        delta_ganancia_minima=(
            "delta_ganancia",
            "min",
        ),
        delta_ganancia_maxima=(
            "delta_ganancia",
            "max",
        ),
        delta_auc_medio=(
            "delta_auc",
            "mean",
        ),
        semillas_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum()),
        ),
    )
    .reset_index()
)


# ============================================================
# RESULTADO GENERAL
# ============================================================

resumen_general = (
    resultados_bloques
    .groupby("experimento")
    .agg(
        features_nuevas=(
            "features_nuevas",
            "first",
        ),
        ganancia_base=(
            "ganancia_base",
            "mean",
        ),
        ganancia_bloque=(
            "ganancia",
            "mean",
        ),
        delta_ganancia_media=(
            "delta_ganancia",
            "mean",
        ),
        delta_ganancia_minima=(
            "delta_ganancia",
            "min",
        ),
        delta_ganancia_maxima=(
            "delta_ganancia",
            "max",
        ),
        delta_auc_medio=(
            "delta_auc",
            "mean",
        ),
        combinaciones_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum()),
        ),
    )
    .reset_index()
)


meses_favorables = (
    resumen_por_mes
    .assign(
        mes_favorable=lambda df:
            df["delta_ganancia_media"] > 0
    )
    .groupby("experimento")
    ["mes_favorable"]
    .sum()
    .astype(int)
    .rename("meses_favorables")
    .reset_index()
)


resumen_general = (
    resumen_general
    .merge(
        meses_favorables,
        on="experimento",
        how="left",
    )
)


resumen_general["delta_porcentaje"] = (
    100.0
    * resumen_general["delta_ganancia_media"]
    / resumen_general["ganancia_base"]
)


resumen_general["lectura_inicial"] = np.select(
    [
        (
            resumen_general[
                "delta_ganancia_media"
            ] > 0
        )
        & (
            resumen_general[
                "combinaciones_favorables"
            ] >= 6
        )
        & (
            resumen_general[
                "meses_favorables"
            ] == 2
        ),

        resumen_general[
            "delta_ganancia_media"
        ] > 0,
    ],
    [
        "CANDIDATO_FUERTE",
        "CANDIDATO_INESTABLE",
    ],
    default="NO_APORTA",
)


resumen_general = (
    resumen_general
    .sort_values(
        "delta_ganancia_media",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("RESULTADO GENERAL")
display(
    resumen_general.round(4)
)

print("RESULTADO POR MES")
display(
    resumen_por_mes.round(4)
)


resultados_bloques.to_csv(
    "comparacion_ablacion_competencia_v4.csv",
    index=False,
)

RESULTADO GENERAL


,experimento,features_nuevas,ganancia_base,ganancia_bloque,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,delta_auc_medio,combinaciones_favorables,meses_favorables,delta_porcentaje,lectura_inicial
0,B6_RECURRENTES_RENTABILIDAD,33,301295500.0,309196250.0,7900750.0,-5115000.0,27967500.0,0.0011,7,2,2.6223,CANDIDATO_FUERTE
1,B1_ANCLAJE_DINAMICA,81,301295500.0,308588500.0,7293000.0,-1897500.0,14520000.0,0.0016,9,2,2.4205,CANDIDATO_FUERTE
2,B4_PRODUCTOS_FINANCIEROS,21,301295500.0,307551750.0,6256250.0,1017500.0,10202500.0,0.0001,10,2,2.0764,CANDIDATO_FUERTE
3,B5_FLUJOS_CANALES,34,301295500.0,307122750.0,5827250.0,-6655000.0,22165000.0,0.0015,7,1,1.9341,CANDIDATO_INESTABLE
4,B3_TARJETAS,16,301295500.0,298091750.0,-3203750.0,-17985000.0,9185000.0,0.0009,4,1,-1.0633,NO_APORTA
5,B2_CARGA_FINANCIERA,12,301295500.0,297319000.0,-3976500.0,-16390000.0,4125000.0,-0.0005,4,1,-1.3198,NO_APORTA


RESULTADO POR MES


,experimento,validacion,ganancia_base,ganancia_bloque,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,delta_auc_medio,semillas_favorables
0,B1_ANCLAJE_DINAMICA,202105,249177500.0,257807000.0,8629500.0,3245000.0,14520000.0,0.0027,5
1,B1_ANCLAJE_DINAMICA,202106,353413500.0,359370000.0,5956500.0,-1897500.0,13557500.0,0.0004,4
2,B2_CARGA_FINANCIERA,202105,249177500.0,250431500.0,1254000.0,-4757500.0,4125000.0,0.0002,4
3,B2_CARGA_FINANCIERA,202106,353413500.0,344206500.0,-9207000.0,-16390000.0,-2062500.0,-0.0012,0
4,B3_TARJETAS,202105,249177500.0,249936500.0,759000.0,-14245000.0,9185000.0,0.0016,3
5,B3_TARJETAS,202106,353413500.0,346247000.0,-7166500.0,-17985000.0,2365000.0,0.0001,1
6,B4_PRODUCTOS_FINANCIEROS,202105,249177500.0,255519000.0,6341500.0,1127500.0,10202500.0,0.0014,5
7,B4_PRODUCTOS_FINANCIEROS,202106,353413500.0,359584500.0,6171000.0,1017500.0,9075000.0,-0.0011,5
8,B5_FLUJOS_CANALES,202105,249177500.0,248589000.0,-588500.0,-6655000.0,3547500.0,0.0007,3
9,B5_FLUJOS_CANALES,202106,353413500.0,365656500.0,12243000.0,-1540000.0,22165000.0,0.0023,4


In [52]:
# ============================================================
# COMBINACIONES DE LOS BLOQUES GANADORES
# ============================================================

features_b1_b4 = list(dict.fromkeys(
    features_originales
    + features_por_bloque["B1_ANCLAJE_DINAMICA"]
    + features_por_bloque["B4_PRODUCTOS_FINANCIEROS"]
))

features_b1_b4_b6 = list(dict.fromkeys(
    features_b1_b4
    + features_por_bloque[
        "B6_RECURRENTES_RENTABILIDAD"
    ]
))

features_b1_b4_b6_b5 = list(dict.fromkeys(
    features_b1_b4_b6
    + features_por_bloque["B5_FLUJOS_CANALES"]
))


combinaciones = {
    "COMBO_B1_B4":
        features_b1_b4,

    "COMBO_B1_B4_B6":
        features_b1_b4_b6,

    "COMBO_B1_B4_B6_B5":
        features_b1_b4_b6_b5,
}


resultados_combinaciones = []

for nombre, features in combinaciones.items():

    resultado = evaluar_experimento(
        datos=base_modelo,
        features=features,
        nombre_experimento=nombre,
        cantidad_features_nuevas=(
            len(features)
            - len(features_originales)
        ),
    )

    resultados_combinaciones.append(resultado)


resultados_combinaciones = pd.concat(
    resultados_combinaciones,
    ignore_index=True,
)

COMBO_B1_B4 | semilla 100129 | validación 202105 | ganancia 255,942,500
COMBO_B1_B4 | semilla 100129 | validación 202106 | ganancia 373,642,500
COMBO_B1_B4 | semilla 100183 | validación 202105 | ganancia 255,832,500
COMBO_B1_B4 | semilla 100183 | validación 202106 | ganancia 350,322,500
COMBO_B1_B4 | semilla 100151 | validación 202105 | ganancia 257,922,500
COMBO_B1_B4 | semilla 100151 | validación 202106 | ganancia 365,447,500
COMBO_B1_B4 | semilla 500041 | validación 202105 | ganancia 258,197,500
COMBO_B1_B4 | semilla 500041 | validación 202106 | ganancia 375,650,000
COMBO_B1_B4 | semilla 900007 | validación 202105 | ganancia 259,875,000
COMBO_B1_B4 | semilla 900007 | validación 202106 | ganancia 369,655,000
COMBO_B1_B4_B6 | semilla 100129 | validación 202105 | ganancia 264,165,000
COMBO_B1_B4_B6 | semilla 100129 | validación 202106 | ganancia 361,762,500
COMBO_B1_B4_B6 | semilla 100183 | validación 202105 | ganancia 264,907,500
COMBO_B1_B4_B6 | semilla 100183 | validación 202106 | g

In [53]:
# ============================================================
# COMPARACIÓN INCREMENTAL
# ============================================================

todos_resultados = pd.concat(
    [
        resultados_experimentos,
        resultados_combinaciones,
    ],
    ignore_index=True,
)


comparaciones_incrementales = [
    (
        "COMBO_B1_B4",
        "B1_ANCLAJE_DINAMICA",
    ),
    (
        "COMBO_B1_B4_B6",
        "COMBO_B1_B4",
    ),
    (
        "COMBO_B1_B4_B6_B5",
        "COMBO_B1_B4_B6",
    ),
]


filas_comparacion = []
filas_comparacion_mes = []

for candidato, referencia in comparaciones_incrementales:

    resultado_candidato = (
        todos_resultados.loc[
            todos_resultados["experimento"]
            == candidato,
            [
                "semilla",
                "validacion",
                "ganancia",
                "auc",
            ],
        ]
        .rename(columns={
            "ganancia": "ganancia_candidato",
            "auc": "auc_candidato",
        })
    )

    resultado_referencia = (
        todos_resultados.loc[
            todos_resultados["experimento"]
            == referencia,
            [
                "semilla",
                "validacion",
                "ganancia",
                "auc",
            ],
        ]
        .rename(columns={
            "ganancia": "ganancia_referencia",
            "auc": "auc_referencia",
        })
    )

    comparacion = resultado_candidato.merge(
        resultado_referencia,
        on=["semilla", "validacion"],
        how="inner",
    )

    comparacion["delta_ganancia"] = (
        comparacion["ganancia_candidato"]
        - comparacion["ganancia_referencia"]
    )

    comparacion["delta_auc"] = (
        comparacion["auc_candidato"]
        - comparacion["auc_referencia"]
    )

    filas_comparacion.append({
        "candidato": candidato,
        "referencia": referencia,

        "ganancia_referencia":
            comparacion[
                "ganancia_referencia"
            ].mean(),

        "ganancia_candidato":
            comparacion[
                "ganancia_candidato"
            ].mean(),

        "delta_ganancia_media":
            comparacion[
                "delta_ganancia"
            ].mean(),

        "delta_ganancia_minima":
            comparacion[
                "delta_ganancia"
            ].min(),

        "delta_ganancia_maxima":
            comparacion[
                "delta_ganancia"
            ].max(),

        "combinaciones_favorables":
            int(
                (
                    comparacion[
                        "delta_ganancia"
                    ] > 0
                ).sum()
            ),

        "delta_auc_medio":
            comparacion[
                "delta_auc"
            ].mean(),
    })

    for mes, grupo in comparacion.groupby(
        "validacion"
    ):

        filas_comparacion_mes.append({
            "candidato": candidato,
            "referencia": referencia,
            "validacion": mes,

            "delta_ganancia_media":
                grupo[
                    "delta_ganancia"
                ].mean(),

            "semillas_favorables":
                int(
                    (
                        grupo[
                            "delta_ganancia"
                        ] > 0
                    ).sum()
                ),
        })


resumen_combinaciones = pd.DataFrame(
    filas_comparacion
)

resumen_combinaciones_mes = pd.DataFrame(
    filas_comparacion_mes
)


display(
    resumen_combinaciones.round(4)
)

display(
    resumen_combinaciones_mes.round(4)
)

,candidato,referencia,ganancia_referencia,ganancia_candidato,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,combinaciones_favorables,delta_auc_medio
0,COMBO_B1_B4,B1_ANCLAJE_DINAMICA,308588500.0,312248750.0,3660250.0,-8690000.0,19827500.0,6,-0.0009
1,COMBO_B1_B4_B6,COMBO_B1_B4,312248750.0,315026250.0,2777500.0,-11880000.0,19332500.0,7,0.0013
2,COMBO_B1_B4_B6_B5,COMBO_B1_B4_B6,315026250.0,315865000.0,838750.0,-10752500.0,15070000.0,5,0.0000


,candidato,referencia,validacion,delta_ganancia_media,semillas_favorables
0,COMBO_B1_B4,B1_ANCLAJE_DINAMICA,202105,-253000.0,2
1,COMBO_B1_B4,B1_ANCLAJE_DINAMICA,202106,7573500.0,4
2,COMBO_B1_B4_B6,COMBO_B1_B4,202105,3586000.0,4
3,COMBO_B1_B4_B6,COMBO_B1_B4,202106,1969000.0,3
4,COMBO_B1_B4_B6_B5,COMBO_B1_B4_B6,202105,-4994000.0,0
5,COMBO_B1_B4_B6_B5,COMBO_B1_B4_B6,202106,6671500.0,5


In [57]:
# ============================================================
# RECONSTRUIR LAGS/DELTAS Y SUS LISTAS
# ============================================================

variables_temporales_b4 = [
    "saldo_promedio_por_cuenta",
    "prestamos_promedio_por_credito",
    "inversiones_pesos_confirmadas",
    "inversiones_dolares_confirmadas",
    "diversificacion_tipos_inversion",
    "diversificacion_tipos_seguro",
    "share_deuda_tarjetas",
    "share_deuda_prestamos",
]

variables_temporales_b6 = [
    "monto_recurrente_total",
    "monto_recurrente_sobre_payroll",
    "comision_observada_por_evento",
    "rentabilidad_por_producto",
    "desvio_rentabilidad_run_rate",
    "margen_activo_sobre_payroll",
    "margen_pasivo_sobre_payroll",
    "descuento_promedio",
]


base_modelo = (
    base_modelo
    .sort_values([
        "numero_de_cliente",
        "foto_mes",
    ])
    .reset_index(drop=True)
)

historia_valida = (
    base_modelo["tiene_historia_lag1"] == 1
)


features_temporales_b4 = []
features_temporales_b6 = []


for grupo, variables in [
    ("B4", variables_temporales_b4),
    ("B6", variables_temporales_b6),
]:

    for variable in variables:

        lag = (
            base_modelo
            .groupby(
                "numero_de_cliente",
                sort=False,
            )[variable]
            .shift(1)
            .where(historia_valida)
        )

        nombre_lag = (
            f"{variable}_lag1_extra"
        )

        nombre_delta = (
            f"{variable}_delta1_extra"
        )

        base_modelo[nombre_lag] = lag

        base_modelo[nombre_delta] = (
            base_modelo[variable] - lag
        )

        if grupo == "B4":
            features_temporales_b4.extend([
                nombre_lag,
                nombre_delta,
            ])
        else:
            features_temporales_b6.extend([
                nombre_lag,
                nombre_delta,
            ])


features_temporales = (
    features_temporales_b4
    + features_temporales_b6
)


base_modelo[features_temporales] = (
    base_modelo[features_temporales]
    .replace([np.inf, -np.inf], np.nan)
)


print(
    "Features temporales B4:",
    len(features_temporales_b4),
)

print(
    "Features temporales B6:",
    len(features_temporales_b6),
)

print(
    "Total:",
    len(features_temporales),
)

Features temporales B4: 16
Features temporales B6: 16
Total: 32


In [58]:
# ============================================================
# CANDIDATO CONGELADO: BASE + B1 + B4 + B6
# ============================================================

features_candidato_principal = list(
    dict.fromkeys(
        features_originales
        + features_por_bloque[
            "B1_ANCLAJE_DINAMICA"
        ]
        + features_por_bloque[
            "B4_PRODUCTOS_FINANCIEROS"
        ]
        + features_por_bloque[
            "B6_RECURRENTES_RENTABILIDAD"
        ]
    )
)


experimentos_temporales = {
    "CANDIDATO_MAS_TEMPORALES_B4":
        list(dict.fromkeys(
            features_candidato_principal
            + features_temporales_b4
        )),

    "CANDIDATO_MAS_TEMPORALES_B6":
        list(dict.fromkeys(
            features_candidato_principal
            + features_temporales_b6
        )),
}


resultados_temporales = []

for nombre, features in (
    experimentos_temporales.items()
):

    resultado = evaluar_experimento(
        datos=base_modelo,
        features=features,
        nombre_experimento=nombre,
        cantidad_features_nuevas=(
            len(features)
            - len(features_candidato_principal)
        ),
    )

    resultados_temporales.append(
        resultado
    )


resultados_temporales = pd.concat(
    resultados_temporales,
    ignore_index=True,
)

CANDIDATO_MAS_TEMPORALES_B4 | semilla 100129 | validación 202105 | ganancia 265,347,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100129 | validación 202106 | ganancia 385,412,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100183 | validación 202105 | ganancia 260,425,000
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100183 | validación 202106 | ganancia 378,950,000
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100151 | validación 202105 | ganancia 260,507,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100151 | validación 202106 | ganancia 380,572,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 500041 | validación 202105 | ganancia 255,227,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 500041 | validación 202106 | ganancia 376,062,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 900007 | validación 202105 | ganancia 262,432,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 900007 | validación 202106 | ganancia 377,822,500
CANDIDATO_MAS_TEMPORALES_B6 | semilla 100129 | validación 202105 | ganancia 265,347,500
CANDIDATO_MAS_TEMPORALES_B6 | se

In [63]:
# ============================================================
# TEMPORALES B4 Y B6: ENTRENAMIENTO + COMPARACIÓN CORRECTA
# ============================================================

# ------------------------------------------------------------
# 1. Candidato principal: BASE + B1 + B4 + B6
# ------------------------------------------------------------

features_candidato_principal = list(
    dict.fromkeys(
        features_originales
        + features_por_bloque["B1_ANCLAJE_DINAMICA"]
        + features_por_bloque["B4_PRODUCTOS_FINANCIEROS"]
        + features_por_bloque["B6_RECURRENTES_RENTABILIDAD"]
    )
)


# ------------------------------------------------------------
# 2. Experimentos temporales
# ------------------------------------------------------------

experimentos_temporales = {
    "CANDIDATO_MAS_TEMPORALES_B4": list(
        dict.fromkeys(
            features_candidato_principal
            + features_temporales_b4
        )
    ),

    "CANDIDATO_MAS_TEMPORALES_B6": list(
        dict.fromkeys(
            features_candidato_principal
            + features_temporales_b6
        )
    ),
}


# ------------------------------------------------------------
# 3. Entrenamiento
# evaluar_experimento devuelve un DataFrame
# ------------------------------------------------------------

resultados_temporales_lista = []

for nombre, features in experimentos_temporales.items():

    resultado = evaluar_experimento(
        datos=base_modelo,
        features=features,
        nombre_experimento=nombre,
        cantidad_features_nuevas=(
            len(features)
            - len(features_candidato_principal)
        ),
    )

    # Cada resultado es un DataFrame de 10 filas
    resultados_temporales_lista.append(resultado)


# Unir los dos DataFrames: 10 + 10 = 20 filas
resultados_temporales = pd.concat(
    resultados_temporales_lista,
    ignore_index=True,
)


# ------------------------------------------------------------
# 4. Control y guardado
# ------------------------------------------------------------

columnas_esperadas = {
    "experimento",
    "semilla",
    "validacion",
    "ganancia",
    "auc",
}

faltantes = (
    columnas_esperadas
    - set(resultados_temporales.columns)
)

assert not faltantes, (
    f"Faltan columnas: {faltantes}"
)

assert len(resultados_temporales) == 20, (
    "Se esperaban 20 resultados y se obtuvieron "
    f"{len(resultados_temporales)}"
)

resultados_temporales.to_csv(
    "resultados_temporales_b4_b6.csv",
    index=False,
)

print(
    "Resultados temporales guardados:",
    resultados_temporales.shape,
)

display(resultados_temporales.head())


# ------------------------------------------------------------
# 5. Candidato principal usado como referencia
# ------------------------------------------------------------

referencia_principal = (
    resultados_combinaciones.loc[
        resultados_combinaciones["experimento"]
        == "COMBO_B1_B4_B6",
        [
            "semilla",
            "validacion",
            "ganancia",
            "auc",
        ],
    ]
    .rename(
        columns={
            "ganancia": "ganancia_referencia",
            "auc": "auc_referencia",
        }
    )
)

assert len(referencia_principal) == 10, (
    "La referencia COMBO_B1_B4_B6 debería "
    "tener 10 resultados."
)


# ------------------------------------------------------------
# 6. Comparación por misma semilla y validación
# ------------------------------------------------------------

comparacion_temporales = (
    resultados_temporales
    .rename(
        columns={
            "ganancia": "ganancia_temporal",
            "auc": "auc_temporal",
        }
    )
    .merge(
        referencia_principal,
        on=["semilla", "validacion"],
        how="inner",
        validate="many_to_one",
    )
)

comparacion_temporales["delta_ganancia"] = (
    comparacion_temporales["ganancia_temporal"]
    - comparacion_temporales["ganancia_referencia"]
)

comparacion_temporales["delta_auc"] = (
    comparacion_temporales["auc_temporal"]
    - comparacion_temporales["auc_referencia"]
)


# ------------------------------------------------------------
# 7. Resumen general
# ------------------------------------------------------------

resumen_temporales = (
    comparacion_temporales
    .groupby(
        "experimento",
        as_index=False,
    )
    .agg(
        ganancia_referencia=(
            "ganancia_referencia",
            "mean",
        ),
        ganancia_candidato=(
            "ganancia_temporal",
            "mean",
        ),
        delta_ganancia_media=(
            "delta_ganancia",
            "mean",
        ),
        delta_ganancia_minima=(
            "delta_ganancia",
            "min",
        ),
        delta_ganancia_maxima=(
            "delta_ganancia",
            "max",
        ),
        combinaciones_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum()),
        ),
        delta_auc_medio=(
            "delta_auc",
            "mean",
        ),
    )
    .sort_values(
        "delta_ganancia_media",
        ascending=False,
    )
)

display(resumen_temporales.round(4))


# ------------------------------------------------------------
# 8. Resumen por mes
# ------------------------------------------------------------

resumen_temporales_por_mes = (
    comparacion_temporales
    .groupby(
        ["experimento", "validacion"],
        as_index=False,
    )
    .agg(
        ganancia_referencia=(
            "ganancia_referencia",
            "mean",
        ),
        ganancia_candidato=(
            "ganancia_temporal",
            "mean",
        ),
        delta_ganancia_media=(
            "delta_ganancia",
            "mean",
        ),
        delta_ganancia_minima=(
            "delta_ganancia",
            "min",
        ),
        delta_ganancia_maxima=(
            "delta_ganancia",
            "max",
        ),
        semillas_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum()),
        ),
        delta_auc_medio=(
            "delta_auc",
            "mean",
        ),
    )
    .sort_values(
        ["experimento", "validacion"]
    )
)

display(resumen_temporales_por_mes.round(4))

CANDIDATO_MAS_TEMPORALES_B4 | semilla 100129 | validación 202105 | ganancia 265,347,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100129 | validación 202106 | ganancia 385,412,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100183 | validación 202105 | ganancia 260,425,000
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100183 | validación 202106 | ganancia 378,950,000
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100151 | validación 202105 | ganancia 260,507,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 100151 | validación 202106 | ganancia 380,572,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 500041 | validación 202105 | ganancia 255,227,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 500041 | validación 202106 | ganancia 376,062,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 900007 | validación 202105 | ganancia 262,432,500
CANDIDATO_MAS_TEMPORALES_B4 | semilla 900007 | validación 202106 | ganancia 377,822,500
CANDIDATO_MAS_TEMPORALES_B6 | semilla 100129 | validación 202105 | ganancia 265,347,500
CANDIDATO_MAS_TEMPORALES_B6 | se

,experimento,semilla,validacion,ganancia,auc,iteraciones,cantidad_features,features_nuevas
0,CANDIDATO_MAS_TEMPORALES_B4,100129,202105,265347500.0,0.890235,403,295,16
1,CANDIDATO_MAS_TEMPORALES_B4,100129,202106,385412500.0,0.899963,607,295,16
2,CANDIDATO_MAS_TEMPORALES_B4,100183,202105,260425000.0,0.890137,416,295,16
3,CANDIDATO_MAS_TEMPORALES_B4,100183,202106,378950000.0,0.899167,520,295,16
4,CANDIDATO_MAS_TEMPORALES_B4,100151,202105,260507500.0,0.890070,350,295,16


,experimento,ganancia_referencia,ganancia_candidato,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,combinaciones_favorables,delta_auc_medio
1,CANDIDATO_MAS_TEMPORALES_B6,315026250.0,320812250.0,5786000.0,-4482500.0,26125000.0,7,-0.0001
0,CANDIDATO_MAS_TEMPORALES_B4,315026250.0,320276000.0,5249750.0,-4482500.0,23650000.0,7,-0.0003


,experimento,validacion,ganancia_referencia,ganancia_candidato,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,semillas_favorables,delta_auc_medio
0,CANDIDATO_MAS_TEMPORALES_B4,202105,261140000.0,260788000.0,-352000.0,-4482500.0,2502500.0,3,-0.0002
1,CANDIDATO_MAS_TEMPORALES_B4,202106,368912500.0,379764000.0,10851500.0,-1842500.0,23650000.0,4,-0.0004
2,CANDIDATO_MAS_TEMPORALES_B6,202105,261140000.0,260788000.0,-352000.0,-4482500.0,2502500.0,3,-0.0002
3,CANDIDATO_MAS_TEMPORALES_B6,202106,368912500.0,380836500.0,11924000.0,-275000.0,26125000.0,4,0.0001


In [64]:
# ============================================================
# ÚLTIMA VALIDACIÓN DE FEATURES
# TEMPORALES B4 + B6 vs TEMPORALES B6
# Solamente validación 202106
# ============================================================

# ------------------------------------------------------------
# 1. Construir el candidato combinado
# El orden conserva primero el candidato B6 de referencia
# ------------------------------------------------------------

features_temporales_b6_candidato = list(
    dict.fromkeys(
        features_candidato_principal
        + features_temporales_b6
    )
)

features_temporales_b4_b6_candidato = list(
    dict.fromkeys(
        features_candidato_principal
        + features_temporales_b6
        + features_temporales_b4
    )
)

print(
    "Features candidato B6:",
    len(features_temporales_b6_candidato),
)

print(
    "Features candidato B4+B6:",
    len(features_temporales_b4_b6_candidato),
)


# ------------------------------------------------------------
# 2. Ejecutar solamente el fold de junio
# Se restauran los folds originales aunque ocurra un error
# ------------------------------------------------------------

folds_temporales_originales = FOLDS_TEMPORALES.copy()

try:

    FOLDS_TEMPORALES = [
        fold
        for fold in folds_temporales_originales
        if fold["validacion"] == 202106
    ]

    assert len(FOLDS_TEMPORALES) == 1, (
        "No se encontró exactamente un fold "
        "para validación 202106."
    )

    resultado_temporales_b4_b6 = (
        evaluar_experimento(
            datos=base_modelo,
            features=(
                features_temporales_b4_b6_candidato
            ),
            nombre_experimento=(
                "CANDIDATO_TEMPORALES_B4_B6"
            ),
            cantidad_features_nuevas=(
                len(
                    features_temporales_b4_b6_candidato
                )
                - len(features_candidato_principal)
            ),
        )
    )

finally:

    # Dejar la configuración original intacta
    FOLDS_TEMPORALES = (
        folds_temporales_originales
    )


# ------------------------------------------------------------
# 3. Control y guardado
# ------------------------------------------------------------

assert len(resultado_temporales_b4_b6) == 5, (
    "Se esperaban 5 resultados."
)

resultado_temporales_b4_b6.to_csv(
    "resultado_temporales_b4_b6_junio.csv",
    index=False,
)

display(resultado_temporales_b4_b6)


# ------------------------------------------------------------
# 4. Referencia: candidato con temporales B6
# ------------------------------------------------------------

referencia_temporales_b6 = (
    resultados_temporales.loc[
        (
            resultados_temporales["experimento"]
            == "CANDIDATO_MAS_TEMPORALES_B6"
        )
        & (
            resultados_temporales["validacion"]
            == 202106
        ),
        [
            "semilla",
            "validacion",
            "ganancia",
            "auc",
        ],
    ]
    .rename(
        columns={
            "ganancia": "ganancia_b6",
            "auc": "auc_b6",
        }
    )
)

assert len(referencia_temporales_b6) == 5, (
    "La referencia temporal B6 debería "
    "tener 5 resultados de junio."
)


# ------------------------------------------------------------
# 5. Comparación emparejada por semilla
# ------------------------------------------------------------

comparacion_final_features = (
    resultado_temporales_b4_b6
    .rename(
        columns={
            "ganancia": "ganancia_b4_b6",
            "auc": "auc_b4_b6",
        }
    )
    .merge(
        referencia_temporales_b6,
        on=["semilla", "validacion"],
        how="inner",
        validate="one_to_one",
    )
)

comparacion_final_features["delta_ganancia"] = (
    comparacion_final_features["ganancia_b4_b6"]
    - comparacion_final_features["ganancia_b6"]
)

comparacion_final_features["delta_auc"] = (
    comparacion_final_features["auc_b4_b6"]
    - comparacion_final_features["auc_b6"]
)


# Resultado por semilla
display(
    comparacion_final_features[
        [
            "semilla",
            "validacion",
            "ganancia_b6",
            "ganancia_b4_b6",
            "delta_ganancia",
            "delta_auc",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 6. Resumen y decisión automática
# ------------------------------------------------------------

delta_media = (
    comparacion_final_features[
        "delta_ganancia"
    ].mean()
)

delta_minimo = (
    comparacion_final_features[
        "delta_ganancia"
    ].min()
)

delta_maximo = (
    comparacion_final_features[
        "delta_ganancia"
    ].max()
)

semillas_favorables = int(
    (
        comparacion_final_features[
            "delta_ganancia"
        ] > 0
    ).sum()
)

delta_auc_medio = (
    comparacion_final_features[
        "delta_auc"
    ].mean()
)

aceptar_temporales_b4 = (
    delta_media > 0
    and semillas_favorables >= 4
)

resumen_final_features = pd.DataFrame({
    "comparacion": [
        "B4+B6 temporales vs B6 temporales"
    ],
    "delta_ganancia_media": [
        delta_media
    ],
    "delta_ganancia_minima": [
        delta_minimo
    ],
    "delta_ganancia_maxima": [
        delta_maximo
    ],
    "semillas_favorables": [
        semillas_favorables
    ],
    "delta_auc_medio": [
        delta_auc_medio
    ],
    "decision": [
        (
            "ACEPTAR_B4_TEMPORAL"
            if aceptar_temporales_b4
            else "CONSERVAR_SOLO_B6_TEMPORAL"
        )
    ],
})

display(resumen_final_features.round(6))

Features candidato B6: 295
Features candidato B4+B6: 311
CANDIDATO_TEMPORALES_B4_B6 | semilla 100129 | validación 202106 | ganancia 383,707,500
CANDIDATO_TEMPORALES_B4_B6 | semilla 100183 | validación 202106 | ganancia 384,010,000
CANDIDATO_TEMPORALES_B4_B6 | semilla 100151 | validación 202106 | ganancia 391,517,500
CANDIDATO_TEMPORALES_B4_B6 | semilla 500041 | validación 202106 | ganancia 387,145,000
CANDIDATO_TEMPORALES_B4_B6 | semilla 900007 | validación 202106 | ganancia 382,772,500


,experimento,semilla,validacion,ganancia,auc,iteraciones,cantidad_features,features_nuevas
0,CANDIDATO_TEMPORALES_B4_B6,100129,202106,383707500.0,0.900076,520,311,32
1,CANDIDATO_TEMPORALES_B4_B6,100183,202106,384010000.0,0.899148,475,311,32
2,CANDIDATO_TEMPORALES_B4_B6,100151,202106,391517500.0,0.900684,764,311,32
3,CANDIDATO_TEMPORALES_B4_B6,500041,202106,387145000.0,0.900723,720,311,32
4,CANDIDATO_TEMPORALES_B4_B6,900007,202106,382772500.0,0.901330,547,311,32


,semilla,validacion,ganancia_b6,ganancia_b4_b6,delta_ganancia,delta_auc
0,100129,202106,387887500.0,383707500.0,-4180000.0,-0.000801
1,100183,202106,371195000.0,384010000.0,12815000.0,-0.000964
2,100151,202106,380462500.0,391517500.0,11055000.0,0.001946
3,500041,202106,377630000.0,387145000.0,9515000.0,0.000128
4,900007,202106,387007500.0,382772500.0,-4235000.0,0.000920


,comparacion,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,semillas_favorables,delta_auc_medio,decision
0,B4+B6 temporales vs B6 temporales,4994000.0,-4235000.0,12815000.0,3,0.000246,CONSERVAR_SOLO_B6_TEMPORAL


In [66]:
# ============================================================
# SEGMENTACIÓN DE CLIENTES BAJA+2 CON RANDOM FOREST
# Uso descriptivo: NO agregar el segmento como feature
# ============================================================


# ------------------------------------------------------------
# 1. Configuración
# ------------------------------------------------------------

ID_COL = "numero_de_cliente"
MES_COL = "foto_mes"
TARGET_COL = "clase_ternaria"

MESES_MADUROS = [
    202103,
    202104,
    202105,
    202106,
]

SEMILLA_SEGMENTACION = SEMILLAS[0]

N_ARBOLES_RF = 300
MIN_CLIENTES_HOJA = 50
CANTIDAD_SEGMENTOS = 5
TOP_VARIABLES_POR_SEGMENTO = 5


# Las 295 features congeladas
features_segmentacion = [
    columna
    for columna in features_temporales_b6_candidato
    if (
        columna in base_modelo.columns
        and pd.api.types.is_numeric_dtype(
            base_modelo[columna]
        )
    )
]

print(
    "Features utilizadas:",
    len(features_segmentacion),
)


# ------------------------------------------------------------
# 2. Trabajar solamente con meses de target maduro
# ------------------------------------------------------------

datos_maduros = (
    base_modelo.loc[
        base_modelo[MES_COL].isin(MESES_MADUROS),
        [
            ID_COL,
            MES_COL,
            TARGET_COL,
        ]
        + features_segmentacion,
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Identificar clientes BAJA+2
# ------------------------------------------------------------

clientes_baja2 = (
    datos_maduros.loc[
        datos_maduros[TARGET_COL] == "BAJA+2",
        ID_COL,
    ]
    .drop_duplicates()
)

print(
    "Clientes BAJA+2:",
    len(clientes_baja2),
)


# ------------------------------------------------------------
# 4. Identificar clientes fieles
#
# Deben:
# - aparecer en todos los meses maduros;
# - no tener BAJA+1 ni BAJA+2.
# ------------------------------------------------------------

resumen_clientes = (
    datos_maduros
    .groupby(ID_COL)
    .agg(
        meses_presentes=(
            MES_COL,
            "nunique",
        ),
        cantidad_bajas=(
            TARGET_COL,
            lambda x: int(
                x.isin(
                    ["BAJA+1", "BAJA+2"]
                ).sum()
            ),
        ),
    )
    .reset_index()
)

clientes_fieles_disponibles = (
    resumen_clientes.loc[
        (
            resumen_clientes["meses_presentes"]
            == len(MESES_MADUROS)
        )
        & (
            resumen_clientes["cantidad_bajas"]
            == 0
        ),
        ID_COL,
    ]
)


# Muestra balanceada por cantidad de clientes
cantidad_fieles = min(
    len(clientes_baja2),
    len(clientes_fieles_disponibles),
)

clientes_fieles = (
    clientes_fieles_disponibles
    .sample(
        n=cantidad_fieles,
        random_state=SEMILLA_SEGMENTACION,
        replace=False,
    )
)

print(
    "Clientes fieles seleccionados:",
    len(clientes_fieles),
)


# ------------------------------------------------------------
# 5. Construir la muestra cliente-mes
# ------------------------------------------------------------

muestra_segmentacion = (
    datos_maduros.loc[
        datos_maduros[ID_COL].isin(
            set(clientes_baja2)
            | set(clientes_fieles)
        )
    ]
    .copy()
)

muestra_segmentacion["grupo_baja2"] = (
    muestra_segmentacion[ID_COL]
    .isin(clientes_baja2)
    .astype("int8")
)

control_grupos = (
    muestra_segmentacion
    .groupby("grupo_baja2")
    .agg(
        clientes=(ID_COL, "nunique"),
        filas=(ID_COL, "size"),
    )
)

display(control_grupos)


# ------------------------------------------------------------
# 6. Imputación para Random Forest
#
# Las medianas se calculan dentro de esta muestra.
# Las columnas completamente vacías se excluyen.
# ------------------------------------------------------------

X_df = (
    muestra_segmentacion[
        features_segmentacion
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

medianas = X_df.median(
    axis=0,
    numeric_only=True,
)

columnas_utiles = (
    medianas.loc[
        medianas.notna()
    ]
    .index
    .tolist()
)

columnas_excluidas = sorted(
    set(features_segmentacion)
    - set(columnas_utiles)
)

if columnas_excluidas:
    print(
        "Columnas completamente vacías excluidas:",
        len(columnas_excluidas),
    )

X = (
    X_df[columnas_utiles]
    .fillna(medianas[columnas_utiles])
    .astype("float32")
    .to_numpy()
)

y = (
    muestra_segmentacion["grupo_baja2"]
    .to_numpy()
)

ids_filas = (
    muestra_segmentacion[ID_COL]
    .to_numpy()
)


# ------------------------------------------------------------
# 7. Entrenar Random Forest descriptivo
# ------------------------------------------------------------

rf_segmentacion = RandomForestClassifier(
    n_estimators=N_ARBOLES_RF,
    min_samples_leaf=MIN_CLIENTES_HOJA,
    max_features="sqrt",
    max_samples=0.70,
    class_weight="balanced_subsample",
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=SEMILLA_SEGMENTACION,
)

rf_segmentacion.fit(X, y)

print(
    "OOB descriptivo:",
    round(rf_segmentacion.oob_score_, 4),
)

# El OOB no reemplaza nuestra validación temporal.
# Aquí se utiliza solamente como control del bosque.


# ------------------------------------------------------------
# 8. Obtener el atributo del último corte de cada hoja
# ------------------------------------------------------------

def atributo_ultimo_corte(arbol):

    estructura = arbol.tree_

    atributo_por_nodo = np.full(
        estructura.node_count,
        -1,
        dtype=np.int32,
    )

    nodos_internos = np.flatnonzero(
        estructura.children_left != -1
    )

    atributo_por_nodo[
        estructura.children_left[nodos_internos]
    ] = estructura.feature[nodos_internos]

    atributo_por_nodo[
        estructura.children_right[nodos_internos]
    ] = estructura.feature[nodos_internos]

    return atributo_por_nodo


hojas_por_fila = rf_segmentacion.apply(X)

cantidad_features_rf = len(columnas_utiles)

codigos_atributos = np.empty_like(
    hojas_por_fila,
    dtype=np.int32,
)

for numero_arbol, arbol in enumerate(
    rf_segmentacion.estimators_
):

    mapa_atributos = atributo_ultimo_corte(
        arbol
    )

    codigos_atributos[:, numero_arbol] = (
        mapa_atributos[
            hojas_por_fila[:, numero_arbol]
        ]
    )


# Código adicional para árboles sin corte
codigos_atributos[
    codigos_atributos < 0
] = cantidad_features_rf


# ------------------------------------------------------------
# 9. Crear perfil de hojas por cliente
# ------------------------------------------------------------

codigos_clientes, ids_unicos = pd.factorize(
    ids_filas,
    sort=False,
)

conteos_cliente_atributo = np.zeros(
    (
        len(ids_unicos),
        cantidad_features_rf + 1,
    ),
    dtype=np.int32,
)

np.add.at(
    conteos_cliente_atributo,
    (
        np.repeat(
            codigos_clientes,
            hojas_por_fila.shape[1],
        ),
        codigos_atributos.ravel(),
    ),
    1,
)

columnas_perfil = (
    columnas_utiles
    + ["_sin_corte"]
)

perfil_clientes = pd.DataFrame(
    conteos_cliente_atributo,
    index=pd.Index(
        ids_unicos,
        name=ID_COL,
    ),
    columns=columnas_perfil,
)

perfil_clientes = (
    perfil_clientes
    .drop(
        columns="_sin_corte",
        errors="ignore",
    )
)

# Excluir atributos que nunca definieron una hoja
perfil_clientes = perfil_clientes.loc[
    :,
    perfil_clientes.sum(axis=0) > 0,
]

# Convertir conteos en proporciones por cliente
perfil_clientes = perfil_clientes.div(
    perfil_clientes.sum(axis=1),
    axis=0,
)


# ------------------------------------------------------------
# 10. Clusterizar únicamente los clientes BAJA+2
# ------------------------------------------------------------

ids_baja2_presentes = pd.Index(
    clientes_baja2
).intersection(
    perfil_clientes.index
)

perfil_baja2 = perfil_clientes.loc[
    ids_baja2_presentes
]

kmeans_segmentacion = KMeans(
    n_clusters=CANTIDAD_SEGMENTOS,
    n_init=20,
    random_state=SEMILLA_SEGMENTACION,
)

cluster_original = (
    kmeans_segmentacion
    .fit_predict(perfil_baja2)
)


# Ordenar segmentos por tamaño:
# segmento 1 = el más grande
conteo_original = pd.Series(
    cluster_original
).value_counts()

orden_segmentos = {
    cluster: numero + 1
    for numero, cluster in enumerate(
        conteo_original.index
    )
}

segmentos_numericos = np.array([
    orden_segmentos[cluster]
    for cluster in cluster_original
])

segmentos_baja2 = pd.DataFrame({
    ID_COL: perfil_baja2.index,
    "segmento_rf": segmentos_numericos,
})


# ------------------------------------------------------------
# 11. Tamaño de los segmentos
# ------------------------------------------------------------

tamanio_segmentos = (
    segmentos_baja2["segmento_rf"]
    .value_counts()
    .sort_index()
    .rename_axis("segmento_rf")
    .reset_index(name="clientes")
)

tamanio_segmentos["proporcion"] = (
    tamanio_segmentos["clientes"]
    / tamanio_segmentos["clientes"].sum()
)

display(tamanio_segmentos)


# ------------------------------------------------------------
# 12. Variables definitorias de cada segmento
#
# Lift:
# participación de la variable en el segmento
# dividida por su participación global.
# ------------------------------------------------------------

perfil_baja2_con_segmento = (
    perfil_baja2
    .copy()
)

perfil_baja2_con_segmento[
    "segmento_rf"
] = segmentos_numericos

share_global = (
    perfil_baja2.mean(axis=0)
)

filas_atributos = []

for segmento, grupo in (
    perfil_baja2_con_segmento
    .groupby("segmento_rf")
):

    perfil_grupo = grupo.drop(
        columns="segmento_rf"
    )

    share_segmento = (
        perfil_grupo.mean(axis=0)
    )

    tabla_segmento = pd.DataFrame({
        "share_segmento": share_segmento,
        "share_global": share_global,
    })

    tabla_segmento["lift"] = (
        tabla_segmento["share_segmento"]
        / tabla_segmento[
            "share_global"
        ].replace(0, np.nan)
    )

    tabla_segmento = (
        tabla_segmento
        .sort_values(
            "lift",
            ascending=False,
        )
        .head(
            TOP_VARIABLES_POR_SEGMENTO
        )
    )

    for ranking, (
        variable,
        fila,
    ) in enumerate(
        tabla_segmento.iterrows(),
        start=1,
    ):

        filas_atributos.append({
            "segmento_rf": segmento,
            "ranking": ranking,
            "variable": variable,
            "share_segmento":
                fila["share_segmento"],
            "share_global":
                fila["share_global"],
            "lift":
                fila["lift"],
        })


atributos_definitorios = pd.DataFrame(
    filas_atributos
)

display(
    atributos_definitorios.round(4)
)


# ------------------------------------------------------------
# 13. Guardar resultados
# ------------------------------------------------------------

segmentos_baja2.to_csv(
    "segmentos_baja2_random_forest.csv",
    index=False,
)

atributos_definitorios.to_csv(
    "atributos_segmentos_baja2.csv",
    index=False,
)

print(
    "Segmentación terminada.",
    "\nClientes segmentados:",
    len(segmentos_baja2),
    "\nArchivos:",
    "\n- segmentos_baja2_random_forest.csv",
    "\n- atributos_segmentos_baja2.csv",
)


# Liberar objetos grandes que ya no se necesitan
del hojas_por_fila
del codigos_atributos
del conteos_cliente_atributo
gc.collect()

Features utilizadas: 295
Clientes BAJA+2: 4067
Clientes fieles seleccionados: 4067


,clientes,filas
grupo_baja2,,
0,4067,16268
1,4067,13184


OOB descriptivo: 0.8287


/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value enco

,segmento_rf,clientes,proporcion
0,1,1651,0.405950
1,2,917,0.225473
2,3,786,0.193263
3,4,390,0.095894
4,5,323,0.079420


,segmento_rf,ranking,variable,share_segmento,share_global,lift
0,1,1,ccheques_emitidos,0.0000,0.0000,2.4634
1,1,2,mcheques_emitidos,0.0000,0.0000,2.4634
2,1,3,mcheques_depositados,0.0000,0.0000,2.4634
3,1,4,Master_mpagado,0.0000,0.0000,2.3446
4,1,5,mttarjeta_master_debitos_automaticos,0.0005,0.0002,2.1528
5,2,1,tcuentas,0.0016,0.0005,3.2197
6,2,2,cruzo_saldo_negativo_lag1,0.0006,0.0002,2.7155
7,2,3,marcas_tarjeta_con_uso_delta1,0.0001,0.0000,2.1354
8,2,4,usa_homebanking_transaccional,0.0046,0.0023,1.9984
9,2,5,ccaja_seguridad,0.0002,0.0001,1.8442


Segmentación terminada. 
Clientes segmentados: 4067 
Archivos: 
- segmentos_baja2_random_forest.csv 
- atributos_segmentos_baja2.csv


54

In [67]:
# ============================================================
# CORRECCIÓN DEL CLUSTERING Y DE VARIABLES DEFINITORIAS
# No vuelve a entrenar Random Forest
# ============================================================

# ------------------------------------------------------------
# 1. Limpiar y normalizar la matriz
# ------------------------------------------------------------

perfil_cluster = (
    perfil_baja2
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .astype("float64")
)

sumas_filas = perfil_cluster.sum(axis=1)

perfil_cluster = (
    perfil_cluster.loc[sumas_filas > 0]
    .div(
        sumas_filas.loc[sumas_filas > 0],
        axis=0,
    )
)

X_cluster = np.ascontiguousarray(
    perfil_cluster.to_numpy(
        dtype=np.float64
    )
)

assert np.isfinite(X_cluster).all()


# ------------------------------------------------------------
# 2. Rehacer solamente KMeans
# ------------------------------------------------------------

kmeans_segmentacion = KMeans(
    n_clusters=CANTIDAD_SEGMENTOS,
    n_init=20,
    algorithm="lloyd",
    random_state=SEMILLA_SEGMENTACION,
)

cluster_original = (
    kmeans_segmentacion
    .fit_predict(X_cluster)
)


# Segmento 1 = grupo más grande
conteo_original = (
    pd.Series(cluster_original)
    .value_counts()
)

orden_segmentos = {
    cluster: posicion + 1
    for posicion, cluster
    in enumerate(conteo_original.index)
}

segmentos_numericos = np.array([
    orden_segmentos[cluster]
    for cluster in cluster_original
])

segmentos_baja2 = pd.DataFrame({
    ID_COL: perfil_cluster.index,
    "segmento_rf": segmentos_numericos,
})


# ------------------------------------------------------------
# 3. Tamaños
# ------------------------------------------------------------

tamanio_segmentos = (
    segmentos_baja2["segmento_rf"]
    .value_counts()
    .sort_index()
    .rename_axis("segmento_rf")
    .reset_index(name="clientes")
)

tamanio_segmentos["proporcion"] = (
    tamanio_segmentos["clientes"]
    / tamanio_segmentos["clientes"].sum()
)

display(tamanio_segmentos)


# ------------------------------------------------------------
# 4. Variables definitorias con filtro mínimo
# ------------------------------------------------------------

perfil_con_segmento = perfil_cluster.copy()

perfil_con_segmento["segmento_rf"] = (
    segmentos_numericos
)

share_global = perfil_cluster.mean(axis=0)

# Filtro incluido en el script original
share_minimo = (
    0.5 / perfil_cluster.shape[1]
)

filas_atributos = []

for segmento, grupo in (
    perfil_con_segmento
    .groupby("segmento_rf")
):

    share_segmento = (
        grupo
        .drop(columns="segmento_rf")
        .mean(axis=0)
    )

    tabla = pd.DataFrame({
        "share_segmento": share_segmento,
        "share_global": share_global,
    })

    tabla["lift"] = (
        tabla["share_segmento"]
        / tabla["share_global"].replace(
            0,
            np.nan,
        )
    )

    tabla = (
        tabla.loc[
            tabla["share_segmento"]
            >= share_minimo
        ]
        .sort_values(
            "lift",
            ascending=False,
        )
        .head(TOP_VARIABLES_POR_SEGMENTO)
    )

    for ranking, (variable, fila) in enumerate(
        tabla.iterrows(),
        start=1,
    ):
        filas_atributos.append({
            "segmento_rf": segmento,
            "ranking": ranking,
            "variable": variable,
            "share_segmento":
                fila["share_segmento"],
            "share_global":
                fila["share_global"],
            "lift": fila["lift"],
        })

atributos_definitorios = pd.DataFrame(
    filas_atributos
)

display(atributos_definitorios.round(6))


# ------------------------------------------------------------
# 5. Sobrescribir resultados corregidos
# ------------------------------------------------------------

segmentos_baja2.to_csv(
    "segmentos_baja2_random_forest.csv",
    index=False,
)

atributos_definitorios.to_csv(
    "atributos_segmentos_baja2.csv",
    index=False,
)

print("Segmentación corregida y guardada.")

/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: invalid value enco

,segmento_rf,clientes,proporcion
0,1,1651,0.405950
1,2,917,0.225473
2,3,786,0.193263
3,4,390,0.095894
4,5,323,0.079420


,segmento_rf,ranking,variable,share_segmento,share_global,lift
0,1,1,ctarjeta_master_transacciones,0.002986,0.001417,2.107089
1,1,2,mtarjeta_master_consumo,0.004692,0.002273,2.063724
2,1,3,monto_debitos_automaticos_tarjetas,0.012329,0.006111,2.017583
3,1,4,mttarjeta_visa_debitos_automaticos,0.010447,0.005333,1.958791
4,1,5,ctarjeta_visa_debitos_automaticos,0.004987,0.002560,1.947843
5,2,1,usa_homebanking_transaccional,0.004642,0.002323,1.998357
6,2,2,actividad_digital_transaccional,0.003006,0.001640,1.833264
7,2,3,tiene_operaciones_recurrentes_lag1,0.003697,0.002080,1.777366
8,2,4,thomebanking,0.005183,0.002977,1.740858
9,2,5,usa_tarjeta,0.002227,0.001286,1.731690


Segmentación corregida y guardada.


In [68]:
control_segmentacion = {
    "matriz_finita":
        bool(np.isfinite(X_cluster).all()),

    "minimo":
        float(X_cluster.min()),

    "maximo":
        float(X_cluster.max()),

    "filas_suman_uno":
        bool(
            np.allclose(
                X_cluster.sum(axis=1),
                1.0,
                atol=1e-8,
            )
        ),

    "inercia_finita":
        bool(
            np.isfinite(
                kmeans_segmentacion.inertia_
            )
        ),

    "centros_finitos":
        bool(
            np.isfinite(
                kmeans_segmentacion.cluster_centers_
            ).all()
        ),

    "iteraciones":
        int(kmeans_segmentacion.n_iter_),
}

control_segmentacion

{'matriz_finita': True,
 'minimo': 0.0,
 'maximo': 0.17333333333333337,
 'filas_suman_uno': True,
 'inercia_finita': True,
 'centros_finitos': True,
 'iteraciones': 28}

In [ ]:
# ============================================================
# COMPARACIÓN DE ESQUEMAS DE CLASES - z601
#
# A: BAJA+2 vs resto
# B: BAJA+1 + BAJA+2 vs CONTINUA
#
# El top-N se elige en mayo y se evalúa en junio.
# ============================================================



# ------------------------------------------------------------
# 1. Configuración
# ------------------------------------------------------------

FEATURES_CLASES = list(
    features_temporales_b6_candidato
)

assert len(FEATURES_CLASES) == 295

ESQUEMAS_CLASES = {
    "BAJA2_VS_RESTO": "baja2",
    "BAJAS12_VS_CONTINUA": "bajas12",
}

# Rango de cantidades de clientes a estimular
CORTES_ENVIO = np.arange(
    2_000,
    30_001,
    250,
)

resultados_cortes_clases = []


# ------------------------------------------------------------
# 2. Función para construir cada target
# ------------------------------------------------------------

def construir_target_clases(
    clase_ternaria,
    tipo_target,
):

    if tipo_target == "baja2":

        return (
            clase_ternaria
            .eq("BAJA+2")
            .astype("int8")
        )

    if tipo_target == "bajas12":

        return (
            clase_ternaria
            .isin(["BAJA+1", "BAJA+2"])
            .astype("int8")
        )

    raise ValueError(
        f"Target desconocido: {tipo_target}"
    )


# ------------------------------------------------------------
# 3. Entrenamiento de ambos esquemas
#
# Early stopping usa AUC porque 0.025 no representa
# lo mismo en los dos targets.
# ------------------------------------------------------------

for esquema, tipo_target in ESQUEMAS_CLASES.items():

    for semilla in SEMILLAS:

        parametros = {
            **PARAMETROS_BASE,

            # Reemplazamos la métrica anterior
            "metric": "auc",
            "first_metric_only": True,

            "seed": semilla,
            "feature_fraction_seed": semilla,
            "bagging_seed": semilla,
            "data_random_seed": SEMILLAS_ENSEMBLE_OPTUNA[0],
        }

        for fold in FOLDS_TEMPORALES:

            mascara_train = (
                base_modelo["foto_mes"]
                .isin(fold["entrenamiento"])
            )

            mascara_validacion = (
                base_modelo["foto_mes"]
                == fold["validacion"]
            )

            X_train = base_modelo.loc[
                mascara_train,
                FEATURES_CLASES,
            ]

            X_validacion = base_modelo.loc[
                mascara_validacion,
                FEATURES_CLASES,
            ]

            clases_train = base_modelo.loc[
                mascara_train,
                "clase_ternaria",
            ]

            clases_validacion = base_modelo.loc[
                mascara_validacion,
                "clase_ternaria",
            ]

            y_train = construir_target_clases(
                clases_train,
                tipo_target,
            )

            y_validacion = construir_target_clases(
                clases_validacion,
                tipo_target,
            )

            # Target real de negocio:
            # solamente BAJA+2 genera premio
            y_baja2_validacion = (
                clases_validacion
                .eq("BAJA+2")
                .astype("int8")
                .to_numpy()
            )

            ganancias_reales = np.where(
                y_baja2_validacion == 1,
                GANANCIA_ACIERTO,
                -COSTO_ESTIMULO,
            )

            dataset_train = lgb.Dataset(
                X_train,
                label=y_train,
            )

            dataset_validacion = lgb.Dataset(
                X_validacion,
                label=y_validacion,
                reference=dataset_train,
            )

            modelo = lgb.train(
                params=parametros,
                train_set=dataset_train,
                num_boost_round=3000,
                valid_sets=[dataset_validacion],
                valid_names=["validacion"],
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        verbose=False,
                    ),
                    lgb.log_evaluation(period=0),
                ],
            )

            probabilidades = modelo.predict(
                X_validacion,
                num_iteration=modelo.best_iteration,
            )

            auc_target = roc_auc_score(
                y_validacion,
                probabilidades,
            )

            auc_baja2 = roc_auc_score(
                y_baja2_validacion,
                probabilidades,
            )


            # ------------------------------------------------
            # Ordenar clientes y construir ganancia acumulada
            # ------------------------------------------------

            orden = np.argsort(
                probabilidades,
                kind="mergesort",
            )[::-1]

            ganancias_ordenadas = (
                ganancias_reales[orden]
            )

            bajas2_ordenadas = (
                y_baja2_validacion[orden]
            )

            ganancia_acumulada = np.cumsum(
                ganancias_ordenadas
            )

            bajas2_acumuladas = np.cumsum(
                bajas2_ordenadas
            )

            total_bajas2 = int(
                y_baja2_validacion.sum()
            )

            cortes_validos = CORTES_ENVIO[
                CORTES_ENVIO
                <= len(probabilidades)
            ]

            for corte in cortes_validos:

                ganancia_corte = float(
                    ganancia_acumulada[
                        corte - 1
                    ]
                )

                bajas2_capturadas = int(
                    bajas2_acumuladas[
                        corte - 1
                    ]
                )

                resultados_cortes_clases.append({
                    "esquema": esquema,
                    "semilla": semilla,
                    "validacion":
                        fold["validacion"],
                    "corte": int(corte),
                    "ganancia":
                        ganancia_corte,
                    "bajas2_capturadas":
                        bajas2_capturadas,
                    "total_bajas2":
                        total_bajas2,
                    "captura_baja2":
                        (
                            bajas2_capturadas
                            / total_bajas2
                        ),
                    "precision_baja2":
                        (
                            bajas2_capturadas
                            / corte
                        ),
                    "auc_target":
                        float(auc_target),
                    "auc_baja2":
                        float(auc_baja2),
                    "iteraciones":
                        int(
                            modelo.best_iteration
                        ),
                })

            print(
                esquema,
                "| semilla",
                semilla,
                "| validación",
                fold["validacion"],
                "| AUC BAJA+2",
                f"{auc_baja2:.5f}",
            )

            del modelo
            del dataset_train
            del dataset_validacion
            del X_train
            del X_validacion

            gc.collect()


# ------------------------------------------------------------
# 4. Unificar y guardar
# ------------------------------------------------------------

resultados_cortes_clases = pd.DataFrame(
    resultados_cortes_clases
)

resultados_cortes_clases.to_csv(
    "comparacion_esquemas_clases.csv",
    index=False,
)

print(
    "Resultados generados:",
    resultados_cortes_clases.shape,
)


# ------------------------------------------------------------
# 5. Elegir el mejor top-N usando solamente mayo
# ------------------------------------------------------------

curvas_mayo = (
    resultados_cortes_clases.loc[
        resultados_cortes_clases[
            "validacion"
        ] == 202105
    ]
    .groupby(
        ["esquema", "corte"],
        as_index=False,
    )
    .agg(
        ganancia_media=(
            "ganancia",
            "mean",
        ),
        ganancia_std=(
            "ganancia",
            "std",
        ),
        captura_media=(
            "captura_baja2",
            "mean",
        ),
        precision_media=(
            "precision_baja2",
            "mean",
        ),
    )
)

indices_mejores = (
    curvas_mayo
    .groupby("esquema")[
        "ganancia_media"
    ]
    .idxmax()
)

mejores_cortes_mayo = (
    curvas_mayo.loc[
        indices_mejores
    ]
    .sort_values("esquema")
    .reset_index(drop=True)
)

print(
    "\nCortes elegidos exclusivamente en mayo:"
)

display(
    mejores_cortes_mayo.round(6)
)


# Avisar si el mejor corte quedó en un extremo
for fila in mejores_cortes_mayo.itertuples():

    if fila.corte in (
        CORTES_ENVIO.min(),
        CORTES_ENVIO.max(),
    ):
        print(
            "ATENCIÓN:",
            fila.esquema,
            "eligió un corte extremo:",
            fila.corte,
        )


# ------------------------------------------------------------
# 6. Evaluar esos cortes en junio
# ------------------------------------------------------------

cortes_elegidos = (
    mejores_cortes_mayo[
        ["esquema", "corte"]
    ]
)

junio_con_corte_mayo = (
    resultados_cortes_clases.loc[
        resultados_cortes_clases[
            "validacion"
        ] == 202106
    ]
    .merge(
        cortes_elegidos,
        on=["esquema", "corte"],
        how="inner",
        validate="many_to_one",
    )
)

resumen_junio_clases = (
    junio_con_corte_mayo
    .groupby(
        "esquema",
        as_index=False,
    )
    .agg(
        corte_elegido=(
            "corte",
            "first",
        ),
        ganancia_media=(
            "ganancia",
            "mean",
        ),
        ganancia_std=(
            "ganancia",
            "std",
        ),
        ganancia_minima=(
            "ganancia",
            "min",
        ),
        ganancia_maxima=(
            "ganancia",
            "max",
        ),
        captura_baja2_media=(
            "captura_baja2",
            "mean",
        ),
        precision_baja2_media=(
            "precision_baja2",
            "mean",
        ),
        auc_baja2_medio=(
            "auc_baja2",
            "mean",
        ),
        iteraciones_mediana=(
            "iteraciones",
            "median",
        ),
    )
    .sort_values(
        "ganancia_media",
        ascending=False,
    )
)

print(
    "\nEvaluación fuera de tiempo en junio:"
)

display(
    resumen_junio_clases.round(6)
)


# ------------------------------------------------------------
# 7. Comparación emparejada por semilla
# ------------------------------------------------------------

comparacion_pareada_clases = (
    junio_con_corte_mayo
    .pivot(
        index="semilla",
        columns="esquema",
        values="ganancia",
    )
    .reset_index()
)

comparacion_pareada_clases[
    "delta_bajas12_menos_baja2"
] = (
    comparacion_pareada_clases[
        "BAJAS12_VS_CONTINUA"
    ]
    - comparacion_pareada_clases[
        "BAJA2_VS_RESTO"
    ]
)

display(comparacion_pareada_clases)


# ------------------------------------------------------------
# 8. Resumen de la comparación
# ------------------------------------------------------------

delta_clases = (
    comparacion_pareada_clases[
        "delta_bajas12_menos_baja2"
    ]
)

resumen_comparacion_clases = pd.DataFrame({
    "comparacion": [
        "BAJAS12 vs BAJA2"
    ],
    "delta_ganancia_media": [
        delta_clases.mean()
    ],
    "delta_ganancia_minima": [
        delta_clases.min()
    ],
    "delta_ganancia_maxima": [
        delta_clases.max()
    ],
    "semillas_favorables_bajas12": [
        int((delta_clases > 0).sum())
    ],
})

display(
    resumen_comparacion_clases.round(4)
)

BAJA2_VS_RESTO | semilla 100129 | validación 202105 | AUC BAJA+2 0.89094
BAJA2_VS_RESTO | semilla 100129 | validación 202106 | AUC BAJA+2 0.90105
BAJA2_VS_RESTO | semilla 100183 | validación 202105 | AUC BAJA+2 0.89048
BAJA2_VS_RESTO | semilla 100183 | validación 202106 | AUC BAJA+2 0.90047
BAJA2_VS_RESTO | semilla 100151 | validación 202105 | AUC BAJA+2 0.89015
BAJA2_VS_RESTO | semilla 100151 | validación 202106 | AUC BAJA+2 0.89903
BAJA2_VS_RESTO | semilla 500041 | validación 202105 | AUC BAJA+2 0.89016
BAJA2_VS_RESTO | semilla 500041 | validación 202106 | AUC BAJA+2 0.90084
BAJA2_VS_RESTO | semilla 900007 | validación 202105 | AUC BAJA+2 0.89086
BAJA2_VS_RESTO | semilla 900007 | validación 202106 | AUC BAJA+2 0.90122
BAJAS12_VS_CONTINUA | semilla 100129 | validación 202105 | AUC BAJA+2 0.89723
BAJAS12_VS_CONTINUA | semilla 100129 | validación 202106 | AUC BAJA+2 0.90263
BAJAS12_VS_CONTINUA | semilla 100183 | validación 202105 | AUC BAJA+2 0.89695
BAJAS12_VS_CONTINUA | semilla 100183

,esquema,corte,ganancia_media,ganancia_std,captura_media,precision_media
0,BAJA2_VS_RESTO,7750,255915000.0,4.956107e+06,0.490115,0.055019
1,BAJAS12_VS_CONTINUA,8750,280335000.0,5.194516e+06,0.544368,0.054126



Evaluación fuera de tiempo en junio:


,esquema,corte_elegido,ganancia_media,ganancia_std,ganancia_minima,ganancia_maxima,captura_baja2_media,precision_baja2_media,auc_baja2_medio,iteraciones_mediana
1,BAJAS12_VS_CONTINUA,8750,368335000.0,5.641188e+06,364375000.0,377575000.0,0.504189,0.063269,0.902953,953.0
0,BAJA2_VS_RESTO,7750,361735000.0,6.104752e+06,353375000.0,367675000.0,0.475956,0.067432,0.900522,542.0


esquema,semilla,BAJA2_VS_RESTO,BAJAS12_VS_CONTINUA,delta_bajas12_menos_baja2
0,100129,367675000.0,377575000.0,9900000.0
1,100151,353375000.0,364375000.0,11000000.0
2,100183,361075000.0,369875000.0,8800000.0
3,500041,358875000.0,364375000.0,5500000.0
4,900007,367675000.0,365475000.0,-2200000.0


,comparacion,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,semillas_favorables_bajas12
0,BAJAS12 vs BAJA2,6600000.0,-2200000.0,11000000.0,4


# Optimización de hiperparámetros


In [71]:
#-------------------------------------------------------------
# 1. Configuración congelada
# ------------------------------------------------------------

FEATURES_OPTIMIZACION = list(
    features_temporales_b6_candidato
)

assert len(FEATURES_OPTIMIZACION) == 295

CONFIGURACION_ESQUEMAS = {
    "BAJA2_VS_RESTO": {
        "tipo_target": "baja2",
        "corte": 7750,
    },

    "BAJAS12_VS_CONTINUA": {
        "tipo_target": "bajas12",
        "corte": 8750,
    },
}

TRIALS_TOTALES_POR_ESQUEMA = 30

SEMILLA_OPTUNA = SEMILLAS[0]
SEMILLA_MODELO_OPTUNA = SEMILLAS[0]

STORAGE_OPTUNA = (
    "sqlite:///optuna_competencia_v4_clases.db"
)



In [72]:

# ------------------------------------------------------------
# 2. Construcción de targets
# ------------------------------------------------------------

def construir_target_optuna(
    clase_ternaria,
    tipo_target,
):

    if tipo_target == "baja2":

        return (
            clase_ternaria
            .eq("BAJA+2")
            .astype("int8")
        )

    if tipo_target == "bajas12":

        return (
            clase_ternaria
            .isin(["BAJA+1", "BAJA+2"])
            .astype("int8")
        )

    raise ValueError(
        f"Target desconocido: {tipo_target}"
    )




In [73]:
# ------------------------------------------------------------
# 3. Ganancia para un top-N fijo
# ------------------------------------------------------------

def calcular_ganancia_top_n(
    probabilidades,
    ganancias_reales,
    corte,
):

    probabilidades = np.asarray(
        probabilidades
    )

    ganancias_reales = np.asarray(
        ganancias_reales
    )

    corte_real = min(
        int(corte),
        len(probabilidades),
    )

    orden = np.argsort(
        probabilidades,
        kind="mergesort",
    )[::-1]

    seleccionados = orden[:corte_real]

    return float(
        ganancias_reales[
            seleccionados
        ].sum()
    )


def crear_metrica_top_n(
    ganancias_validacion,
    corte,
):

    ganancias_validacion = np.asarray(
        ganancias_validacion
    )

    def metrica_top_n(
        probabilidades,
        dataset_lgb,
    ):

        assert (
            dataset_lgb.num_data()
            == len(ganancias_validacion)
        )

        ganancia = calcular_ganancia_top_n(
            probabilidades,
            ganancias_validacion,
            corte,
        )

        return (
            "ganancia_top_n",
            ganancia,
            True,
        )

    return metrica_top_n




In [75]:
# ------------------------------------------------------------
# 4. Objetivo de Optuna
# ------------------------------------------------------------

def objetivo_hiperparametros(
    trial,
    esquema,
):

    configuracion = (
        CONFIGURACION_ESQUEMAS[
            esquema
        ]
    )

    tipo_target = configuracion[
        "tipo_target"
    ]

    corte = configuracion["corte"]


    # --------------------------------------------------------
    # Hiperparámetros a explorar
    # --------------------------------------------------------

    max_depth = trial.suggest_int(
        "max_depth",
        5,
        12,
    )

    num_leaves = trial.suggest_int(
        "num_leaves",
        16,
        128,
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        0.005,
        0.05,
        log=True,
    )

    min_data_in_leaf = trial.suggest_int(
        "min_data_in_leaf",
        100,
        1500,
        log=True,
    )

    feature_fraction = trial.suggest_float(
        "feature_fraction",
        0.20,
        0.80,
    )

    bagging_fraction = trial.suggest_float(
        "bagging_fraction",
        0.60,
        1.00,
    )

    lambda_l1 = trial.suggest_float(
        "lambda_l1",
        1e-5,
        10.0,
        log=True,
    )

    lambda_l2 = trial.suggest_float(
        "lambda_l2",
        1e-5,
        10.0,
        log=True,
    )

    min_gain_to_split = trial.suggest_float(
        "min_gain_to_split",
        0.0,
        5.0,
    )

    max_bin = trial.suggest_categorical(
        "max_bin",
        [31, 63, 127, 255],
    )


    parametros = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,

        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "learning_rate": learning_rate,
        "min_data_in_leaf":
            min_data_in_leaf,
        "feature_fraction":
            feature_fraction,
        "bagging_fraction":
            bagging_fraction,
        "bagging_freq": 1,
        "lambda_l1": lambda_l1,
        "lambda_l2": lambda_l2,
        "min_gain_to_split":
            min_gain_to_split,
        "max_bin": max_bin,

        "seed":
            SEMILLA_MODELO_OPTUNA,
        "feature_fraction_seed":
            SEMILLA_MODELO_OPTUNA,
        "bagging_seed":
            SEMILLA_MODELO_OPTUNA,
        "data_random_seed":
            SEMILLA_MODELO_OPTUNA,
    }


    ganancias_folds = []
    auc_baja2_folds = []
    auc_target_folds = []
    iteraciones_folds = []


    # --------------------------------------------------------
    # Evaluación temporal
    # --------------------------------------------------------

    for fold in FOLDS_TEMPORALES:

        mascara_train = (
            base_modelo["foto_mes"]
            .isin(fold["entrenamiento"])
        )

        mascara_validacion = (
            base_modelo["foto_mes"]
            == fold["validacion"]
        )

        X_train = base_modelo.loc[
            mascara_train,
            FEATURES_OPTIMIZACION,
        ]

        X_validacion = base_modelo.loc[
            mascara_validacion,
            FEATURES_OPTIMIZACION,
        ]

        clases_train = base_modelo.loc[
            mascara_train,
            "clase_ternaria",
        ]

        clases_validacion = base_modelo.loc[
            mascara_validacion,
            "clase_ternaria",
        ]

        y_train = construir_target_optuna(
            clases_train,
            tipo_target,
        )

        y_validacion = construir_target_optuna(
            clases_validacion,
            tipo_target,
        )

        y_baja2_validacion = (
            clases_validacion
            .eq("BAJA+2")
            .astype("int8")
            .to_numpy()
        )

        ganancias_validacion = np.where(
            y_baja2_validacion == 1,
            GANANCIA_ACIERTO,
            -COSTO_ESTIMULO,
        )

        dataset_train = lgb.Dataset(
            X_train,
            label=y_train,
        )

        dataset_validacion = lgb.Dataset(
            X_validacion,
            label=y_validacion,
            reference=dataset_train,
        )

        modelo = lgb.train(
            params=parametros,
            train_set=dataset_train,
            num_boost_round=4000,
            valid_sets=[dataset_validacion],
            valid_names=["validacion"],
            feval=crear_metrica_top_n(
                ganancias_validacion,
                corte,
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False,
                ),
                lgb.log_evaluation(period=0),
            ],
        )

        probabilidades = modelo.predict(
            X_validacion,
            num_iteration=modelo.best_iteration,
        )

        ganancia_fold = (
            calcular_ganancia_top_n(
                probabilidades,
                ganancias_validacion,
                corte,
            )
        )

        auc_target = roc_auc_score(
            y_validacion,
            probabilidades,
        )

        auc_baja2 = roc_auc_score(
            y_baja2_validacion,
            probabilidades,
        )

        ganancias_folds.append(
            ganancia_fold
        )

        auc_target_folds.append(
            float(auc_target)
        )

        auc_baja2_folds.append(
            float(auc_baja2)
        )

        iteraciones_folds.append(
            int(modelo.best_iteration)
        )


        # Guardar detalle dentro del trial
        mes_validacion = fold["validacion"]

        trial.set_user_attr(
            f"ganancia_{mes_validacion}",
            ganancia_fold,
        )

        trial.set_user_attr(
            f"auc_target_{mes_validacion}",
            float(auc_target),
        )

        trial.set_user_attr(
            f"auc_baja2_{mes_validacion}",
            float(auc_baja2),
        )

        trial.set_user_attr(
            f"iteraciones_{mes_validacion}",
            int(modelo.best_iteration),
        )


        del modelo
        del dataset_train
        del dataset_validacion
        del X_train
        del X_validacion

        gc.collect()


    # Promedio con igual peso para ambos meses
    ganancia_media = float(
        np.mean(ganancias_folds)
    )

    trial.set_user_attr(
        "ganancia_media",
        ganancia_media,
    )

    trial.set_user_attr(
        "auc_baja2_medio",
        float(np.mean(auc_baja2_folds)),
    )

    trial.set_user_attr(
        "auc_target_medio",
        float(np.mean(auc_target_folds)),
    )

    trial.set_user_attr(
        "iteraciones_mediana",
        float(np.median(iteraciones_folds)),
    )

    trial.set_user_attr(
        "esquema",
        esquema,
    )

    trial.set_user_attr(
        "corte",
        corte,
    )

    return ganancia_media


# ------------------------------------------------------------
# 5. Crear o recuperar los estudios
# ------------------------------------------------------------

estudios_clases = {}

for esquema in CONFIGURACION_ESQUEMAS:

    sampler = optuna.samplers.TPESampler(
        seed=SEMILLA_OPTUNA,
        n_startup_trials=10,
    )

    nombre_estudio = (
        "competencia_v4_"
        + esquema.lower()
        + "_topn_v1"
    )

    estudio = optuna.create_study(
        direction="maximize",
        study_name=nombre_estudio,
        storage=STORAGE_OPTUNA,
        load_if_exists=True,
        sampler=sampler,
    )


    # Incluir los hiperparámetros base como primer trial
    if len(estudio.trials) == 0:

        estudio.enqueue_trial({
            "max_depth": 8,
            "num_leaves": 48,
            "learning_rate":
                0.011968361180129191,
            "min_data_in_leaf": 405,
            "feature_fraction":
                0.30152141273199784,
            "bagging_fraction":
                0.7006936995574339,
            "lambda_l1":
                8.320444953703719e-05,
            "lambda_l2":
                0.0006696837335733366,
            "min_gain_to_split":
                1.5614099380945257,
            "max_bin": 255,
        })


    cantidad_actual = len(
        estudio.trials
    )

    trials_faltantes = max(
        0,
        (
            TRIALS_TOTALES_POR_ESQUEMA
            - cantidad_actual
        ),
    )

    print(
        "\nEstudio:",
        esquema,
        "| existentes:",
        cantidad_actual,
        "| faltantes:",
        trials_faltantes,
    )

    if trials_faltantes > 0:

        estudio.optimize(
            lambda trial, esquema=esquema:
                objetivo_hiperparametros(
                    trial,
                    esquema,
                ),
            n_trials=trials_faltantes,
            n_jobs=1,
            gc_after_trial=True,
            show_progress_bar=True,
        )

    estudios_clases[esquema] = estudio


# ------------------------------------------------------------
# 6. Tabla con los cinco mejores trials
# ------------------------------------------------------------

def obtener_top_trials(
    estudio,
    esquema,
    cantidad=5,
):

    trials_completos = [
        trial
        for trial in estudio.trials
        if (
            trial.state
            == optuna.trial.TrialState.COMPLETE
        )
    ]

    trials_ordenados = sorted(
        trials_completos,
        key=lambda trial: trial.value,
        reverse=True,
    )[:cantidad]

    filas = []

    for posicion, trial in enumerate(
        trials_ordenados,
        start=1,
    ):

        fila = {
            "esquema": esquema,
            "posicion": posicion,
            "trial": trial.number,
            "ganancia_media": trial.value,

            "ganancia_202105":
                trial.user_attrs[
                    "ganancia_202105"
                ],

            "ganancia_202106":
                trial.user_attrs[
                    "ganancia_202106"
                ],

            "auc_baja2_medio":
                trial.user_attrs[
                    "auc_baja2_medio"
                ],

            "iteraciones_202105":
                trial.user_attrs[
                    "iteraciones_202105"
                ],

            "iteraciones_202106":
                trial.user_attrs[
                    "iteraciones_202106"
                ],
        }

        fila.update(trial.params)
        filas.append(fila)

    return pd.DataFrame(filas)


top_trials_clases = pd.concat(
    [
        obtener_top_trials(
            estudios_clases[esquema],
            esquema,
            cantidad=5,
        )
        for esquema
        in CONFIGURACION_ESQUEMAS
    ],
    ignore_index=True,
)

display(
    top_trials_clases.round(6)
)


# ------------------------------------------------------------
# 7. Guardar resultados
# ------------------------------------------------------------

top_trials_clases.to_csv(
    "top_trials_optuna_clases.csv",
    index=False,
)

for esquema, estudio in estudios_clases.items():

    estudio.trials_dataframe().to_csv(
        (
            "trials_optuna_"
            + esquema.lower()
            + ".csv"
        ),
        index=False,
    )

print("\nOptimización terminada.")
print(
    "Los estudios quedaron guardados en:",
    STORAGE_OPTUNA,
)

[I 2026-10-10 21:00:59,118] A new study created in RDB with name: competencia_v4_baja2_vs_resto_topn_v1



Estudio: BAJA2_VS_RESTO | existentes: 1 | faltantes: 29


Best trial: 0. Best value: 3.16525e+08:   3%|▎         | 1/29 [00:48<22:42, 48.68s/it]

[I 2026-10-10 21:01:47,729] Trial 0 finished with value: 316525000.0 and parameters: {'max_depth': 8, 'num_leaves': 48, 'learning_rate': 0.011968361180129191, 'min_data_in_leaf': 405, 'feature_fraction': 0.30152141273199784, 'bagging_fraction': 0.7006936995574339, 'lambda_l1': 8.320444953703719e-05, 'lambda_l2': 0.0006696837335733366, 'min_gain_to_split': 1.5614099380945257, 'max_bin': 255}. Best is trial 0 with value: 316525000.0.


Best trial: 0. Best value: 3.16525e+08:   7%|▋         | 2/29 [01:07<13:54, 30.92s/it]

...

[I 2026-10-10 21:51:30,565] Trial 27 finished with value: 324775000.0 and parameters: {'max_depth': 5, 'num_leaves': 39, 'learning_rate': 0.040377872549842556, 'min_data_in_leaf': 1497, 'feature_fraction': 0.45010249100825717, 'bagging_fraction': 0.8330885252767213, 'lambda_l1': 2.8623905981118608e-05, 'lambda_l2': 0.0195299073361392, 'min_gain_to_split': 1.1975467240190607, 'max_bin': 255}. Best is trial 21 with value: 339075000.0.


Best trial: 21. Best value: 3.39075e+08: 100%|██████████| 29/29 [29:17<00:00, 60.60s/it]

[I 2026-10-10 21:52:16,654] Trial 28 finished with value: 324225000.0 and parameters: {'max_depth': 6, 'num_leaves': 77, 'learning_rate': 0.04447140177494953, 'min_data_in_leaf': 106, 'feature_fraction': 0.4001330904446002, 'bagging_fraction': 0.7727241437111348, 'lambda_l1': 0.00011728234075847908, 'lambda_l2': 0.0001509567937480484, 'min_gain_to_split': 0.5755714945354518, 'max_bin': 255}. Best is trial 21 with value: 339075000.0.


,esquema,posicion,trial,ganancia_media,ganancia_202105,ganancia_202106,auc_baja2_medio,iteraciones_202105,iteraciones_202106,max_depth,num_leaves,learning_rate,min_data_in_leaf,feature_fraction,bagging_fraction,lambda_l1,lambda_l2,min_gain_to_split,max_bin
0,BAJA2_VS_RESTO,1,13,317075000.0,255475000.0,378675000.0,0.892389,412,12,8,83,0.012087,1465,0.304163,0.812702,0.002578,0.000125,3.034533,255
1,BAJA2_VS_RESTO,2,16,317075000.0,265375000.0,368775000.0,0.891756,459,8,9,73,0.008817,1043,0.312505,0.817204,0.002126,0.000153,2.669592,255
2,BAJA2_VS_RESTO,3,0,316525000.0,263175000.0,369875000.0,0.895461,212,460,8,48,0.011968,405,0.301521,0.700694,0.000083,0.000670,1.561410,255
3,BAJA2_VS_RESTO,4,14,313775000.0,254375000.0,373175000.0,0.891714,256,17,8,37,0.011733,1448,0.325252,0.602897,0.000687,0.000290,3.348198,127
4,BAJA2_VS_RESTO,5,18,312675000.0,255475000.0,369875000.0,0.888271,247,5,10,102,0.007820,1480,0.497007,0.818325,0.001500,0.000068,2.645483,127
5,BAJAS12_VS_CONTINUA,1,21,339075000.0,279675000.0,398475000.0,0.898832,198,282,6,53,0.049754,559,0.366211,0.721882,0.000054,0.000552,2.184697,255
6,BAJAS12_VS_CONTINUA,2,24,336325000.0,280775000.0,391875000.0,0.898328,123,259,5,35,0.049969,1429,0.378443,0.775964,0.000038,0.000236,1.139989,255
7,BAJAS12_VS_CONTINUA,3,22,333575000.0,277475000.0,389675000.0,0.898913,263,249,6,60,0.049695,584,0.374490,0.718788,0.000038,0.000047,2.051292,255
8,BAJAS12_VS_CONTINUA,4,26,333025000.0,274175000.0,391875000.0,0.898610,132,613,7,34,0.032224,939,0.357622,0.651838,0.003121,0.000323,0.475239,255
9,BAJAS12_VS_CONTINUA,5,18,332475000.0,279675000.0,385275000.0,0.899144,203,391,5,55,0.047631,604,0.398920,0.718396,0.000040,0.000010,2.244281,255



Optimización terminada.
Los estudios quedaron guardados en: sqlite:///optuna_competencia_v4_clases.db


In [78]:
# ============================================================
# EVALUACIÓN CONTROLADA FINAL DE FEATURES
#
# 1. Candidato actual: BASE + B1 + B4 + B6 + temporales B6
# 2. Candidato actual + rangos mensuales
# 3. Candidato actual + B5 (flujos y canales)
#
# Target: BAJA+1 + BAJA+2 vs CONTINUA
# Métrica de negocio: ganancia real sobre BAJA+2
# Corte fijo: 8750
# Early stopping: AUC
# ============================================================

import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Configuración
# ------------------------------------------------------------

CORTE_CONTROLADO = 8750

FOLDS_CONTROLADOS = [
    {
        "validacion": 202105,
        "entrenamiento": [202103],
    },
    {
        "validacion": 202106,
        "entrenamiento": [202103, 202104],
    },
]


# ------------------------------------------------------------
# 2. Variables para transformar en rangos mensuales
#
# Se usan principalmente montos y niveles sensibles a inflación.
# El rango se calcula separadamente dentro de cada foto_mes.
# ------------------------------------------------------------

variables_rank_solicitadas = [
    # Actividad e ingreso
    "ctrx_quarter",
    "mpayroll",
    "payroll_referencia_2m",

    # Saldos
    "mcuentas_saldo",
    "mcuenta_corriente",
    "mcaja_ahorro",
    "saldo_promedio_por_cuenta",

    # Rentabilidad y márgenes
    "mrentabilidad",
    "mrentabilidad_annual",
    "mactivos_margen",
    "mpasivos_margen",
    "desvio_rentabilidad_run_rate",

    # Comisiones
    "mcomisiones",
    "mcomisiones_mantenimiento",
    "comisiones_monto_observado",

    # Préstamos
    "mprestamos_personales",
    "mprestamos_prendarios",
    "mprestamos_hipotecarios",
    "prestamos_promedio_por_credito",

    # Tarjetas
    "Master_msaldopesos",
    "Visa_msaldopesos",
    "Master_mlimitecompra",
    "Visa_mlimitecompra",
    "Master_mconsumospesos",
    "Visa_mconsumospesos",
    "Master_mpagominimo",
    "Visa_mpagominimo",

    # Inversiones y recurrentes
    "inversiones_pesos_confirmadas",
    "inversiones_dolares_confirmadas",
    "monto_recurrente_total",
]


variables_rank_mes = [
    variable
    for variable in variables_rank_solicitadas
    if variable in base_modelo.columns
]

variables_rank_faltantes = sorted(
    set(variables_rank_solicitadas)
    - set(variables_rank_mes)
)

print(
    "Variables disponibles para rangos:",
    len(variables_rank_mes),
)

if variables_rank_faltantes:
    print(
        "Variables no encontradas y omitidas:",
        variables_rank_faltantes,
    )

assert len(variables_rank_mes) >= 20, (
    "Se encontraron muy pocas variables para construir rangos. "
    "Revisar los nombres de las columnas."
)


# ------------------------------------------------------------
# 3. Construir rangos dentro de cada mes
#
# Los NULL permanecen como NULL.
# Cada rango queda entre 0 y 1.
# ------------------------------------------------------------

columnas_rank_mes = [
    f"{variable}_rank_mes"
    for variable in variables_rank_mes
]

# Permite volver a ejecutar la celda sin duplicar columnas.
base_modelo = base_modelo.drop(
    columns=columnas_rank_mes,
    errors="ignore",
)

rangos_mensuales = (
    base_modelo
    .groupby(
        "foto_mes",
        sort=False,
    )[variables_rank_mes]
    .rank(
        method="average",
        pct=True,
        na_option="keep",
    )
)

rangos_mensuales.columns = columnas_rank_mes

base_modelo = pd.concat(
    [
        base_modelo,
        rangos_mensuales,
    ],
    axis=1,
)

del rangos_mensuales
gc.collect()


# ------------------------------------------------------------
# 4. Auditoría de los rangos
# ------------------------------------------------------------

minimo_rangos = (
    base_modelo[columnas_rank_mes]
    .min()
    .min()
)

maximo_rangos = (
    base_modelo[columnas_rank_mes]
    .max()
    .max()
)

cantidad_infinitos = np.isinf(
    base_modelo[columnas_rank_mes]
    .to_numpy(dtype="float64")
).sum()

print("Mínimo de los rangos:", minimo_rangos)
print("Máximo de los rangos:", maximo_rangos)
print("Valores infinitos:", cantidad_infinitos)

assert cantidad_infinitos == 0
assert minimo_rangos >= 0
assert maximo_rangos <= 1


# ------------------------------------------------------------
# 5. Armar los tres candidatos
# ------------------------------------------------------------

features_actuales = list(
    dict.fromkeys(
        features_temporales_b6_candidato
    )
)

features_b5 = list(
    features_por_bloque[
        "B5_FLUJOS_CANALES"
    ]
)

experimentos_features_controlados = {
    "ACTUAL_B1_B4_B6_TEMP_B6":
        features_actuales,

    "ACTUAL_MAS_RANKS_MENSUALES":
        list(
            dict.fromkeys(
                features_actuales
                + columnas_rank_mes
            )
        ),

    "ACTUAL_MAS_B5":
        list(
            dict.fromkeys(
                features_actuales
                + features_b5
            )
        ),
}

for nombre, features in (
    experimentos_features_controlados.items()
):
    faltantes = sorted(
        set(features)
        - set(base_modelo.columns)
    )

    assert not faltantes, (
        f"{nombre}: faltan columnas: {faltantes}"
    )

    print(
        nombre,
        "| features:",
        len(features),
        "| agregadas:",
        len(features) - len(features_actuales),
    )


# ------------------------------------------------------------
# 6. Función de evaluación
#
# Importante:
# - El target de entrenamiento incluye BAJA+1 y BAJA+2.
# - La ganancia sólo premia BAJA+2.
# - Early stopping usa AUC, no ganancia top-N.
# - Se crea un Dataset nuevo en cada semilla.
# ------------------------------------------------------------

def evaluar_experimento_features(
    nombre_experimento,
    features,
):
    resultados = []

    for semilla in SEMILLAS:
        for fold in FOLDS_CONTROLADOS:

            mes_validacion = fold["validacion"]
            meses_entrenamiento = fold["entrenamiento"]

            mascara_train = (
                base_modelo["foto_mes"]
                .isin(meses_entrenamiento)
            )

            mascara_valid = (
                base_modelo["foto_mes"]
                .eq(mes_validacion)
            )

            X_train = base_modelo.loc[
                mascara_train,
                features,
            ]

            X_valid = base_modelo.loc[
                mascara_valid,
                features,
            ]

            y_train = construir_target_optuna(
                base_modelo.loc[
                    mascara_train,
                    "clase_ternaria",
                ],
                "bajas12",
            )

            y_valid = construir_target_optuna(
                base_modelo.loc[
                    mascara_valid,
                    "clase_ternaria",
                ],
                "bajas12",
            )

            y_valid_baja2 = (
                base_modelo.loc[
                    mascara_valid,
                    "clase_ternaria",
                ]
                .eq("BAJA+2")
                .astype("int8")
                .to_numpy()
            )

            ganancias_reales = np.where(
                y_valid_baja2 == 1,
                GANANCIA_ACIERTO,
                -COSTO_ESTIMULO,
            )

            parametros = {
                **PARAMETROS_BASE,

                # Early stopping estable
                "objective": "binary",
                "metric": "auc",
                "first_metric_only": True,

                # Semillas
                "seed": semilla,
                "feature_fraction_seed": semilla,
                "bagging_seed": semilla,
                "data_random_seed": semilla,

                "verbosity": -1,
            }

            dataset_train = lgb.Dataset(
                X_train,
                label=y_train,
                free_raw_data=False,
            )

            dataset_valid = lgb.Dataset(
                X_valid,
                label=y_valid,
                reference=dataset_train,
                free_raw_data=False,
            )

            modelo = lgb.train(
                params=parametros,
                train_set=dataset_train,
                valid_sets=[dataset_valid],
                valid_names=["validacion"],
                num_boost_round=3000,
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        first_metric_only=True,
                        verbose=False,
                    )
                ],
            )

            probabilidades = modelo.predict(
                X_valid,
                num_iteration=modelo.best_iteration,
            )

            ganancia = calcular_ganancia_top_n(
                probabilidades=probabilidades,
                ganancias_reales=ganancias_reales,
                corte=CORTE_CONTROLADO,
            )

            auc_target = roc_auc_score(
                y_valid,
                probabilidades,
            )

            auc_baja2 = roc_auc_score(
                y_valid_baja2,
                probabilidades,
            )

            resultados.append({
                "experimento": nombre_experimento,
                "semilla": semilla,
                "validacion": mes_validacion,
                "ganancia": ganancia,
                "auc_target_bajas12": auc_target,
                "auc_baja2": auc_baja2,
                "iteraciones": modelo.best_iteration,
                "cantidad_features": len(features),
                "features_agregadas": (
                    len(features)
                    - len(features_actuales)
                ),
                "corte": CORTE_CONTROLADO,
            })

            print(
                nombre_experimento,
                "| semilla",
                semilla,
                "| validación",
                mes_validacion,
                "| ganancia",
                f"{ganancia:,.0f}",
                "| AUC BAJA+2",
                f"{auc_baja2:.5f}",
                "| iteraciones",
                modelo.best_iteration,
            )

            del (
                X_train,
                X_valid,
                y_train,
                y_valid,
                y_valid_baja2,
                ganancias_reales,
                probabilidades,
                dataset_train,
                dataset_valid,
                modelo,
            )

            gc.collect()

    return resultados


# ------------------------------------------------------------
# 7. Ejecutar los tres experimentos
# ------------------------------------------------------------

resultados_features_controlados = []

for nombre, features in (
    experimentos_features_controlados.items()
):
    resultados_features_controlados.extend(
        evaluar_experimento_features(
            nombre_experimento=nombre,
            features=features,
        )
    )

resultados_features_controlados = pd.DataFrame(
    resultados_features_controlados
)

print(
    "\nResultados generados:",
    resultados_features_controlados.shape,
)


# ------------------------------------------------------------
# 8. Comparación pareada contra el candidato actual
# ------------------------------------------------------------

nombre_referencia = (
    "ACTUAL_B1_B4_B6_TEMP_B6"
)

referencia = (
    resultados_features_controlados
    .loc[
        resultados_features_controlados[
            "experimento"
        ].eq(nombre_referencia),
        [
            "semilla",
            "validacion",
            "ganancia",
            "auc_baja2",
        ],
    ]
    .rename(columns={
        "ganancia": "ganancia_referencia",
        "auc_baja2": "auc_baja2_referencia",
    })
)

comparacion_features_controlados = (
    resultados_features_controlados
    .loc[
        ~resultados_features_controlados[
            "experimento"
        ].eq(nombre_referencia)
    ]
    .merge(
        referencia,
        on=[
            "semilla",
            "validacion",
        ],
        how="inner",
        validate="many_to_one",
    )
)

comparacion_features_controlados[
    "delta_ganancia"
] = (
    comparacion_features_controlados["ganancia"]
    - comparacion_features_controlados[
        "ganancia_referencia"
    ]
)

comparacion_features_controlados[
    "delta_auc_baja2"
] = (
    comparacion_features_controlados["auc_baja2"]
    - comparacion_features_controlados[
        "auc_baja2_referencia"
    ]
)


# ------------------------------------------------------------
# 9. Resumen por mes
# ------------------------------------------------------------

resumen_por_mes = (
    comparacion_features_controlados
    .groupby(
        [
            "experimento",
            "validacion",
        ],
        as_index=False,
    )
    .agg(
        ganancia_referencia=(
            "ganancia_referencia",
            "mean",
        ),
        ganancia_candidato=(
            "ganancia",
            "mean",
        ),
        delta_ganancia_media=(
            "delta_ganancia",
            "mean",
        ),
        delta_ganancia_minima=(
            "delta_ganancia",
            "min",
        ),
        delta_ganancia_maxima=(
            "delta_ganancia",
            "max",
        ),
        semillas_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum()),
        ),
        delta_auc_baja2_medio=(
            "delta_auc_baja2",
            "mean",
        ),
        iteraciones_mediana=(
            "iteraciones",
            "median",
        ),
    )
    .sort_values(
        [
            "experimento",
            "validacion",
        ]
    )
)

display(resumen_por_mes)


# ------------------------------------------------------------
# 10. Resumen general y decisión automática
# ------------------------------------------------------------

resumen_general = (
    comparacion_features_controlados
    .groupby(
        "experimento",
        as_index=False,
    )
    .agg(
        ganancia_referencia=(
            "ganancia_referencia",
            "mean",
        ),
        ganancia_candidato=(
            "ganancia",
            "mean",
        ),
        delta_ganancia_media=(
            "delta_ganancia",
            "mean",
        ),
        delta_ganancia_minima=(
            "delta_ganancia",
            "min",
        ),
        delta_ganancia_maxima=(
            "delta_ganancia",
            "max",
        ),
        combinaciones_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum()),
        ),
        delta_auc_baja2_medio=(
            "delta_auc_baja2",
            "mean",
        ),
    )
)

criterios_por_mes = (
    resumen_por_mes
    .groupby(
        "experimento",
        as_index=False,
    )
    .agg(
        meses_con_delta_positivo=(
            "delta_ganancia_media",
            lambda x: int((x > 0).sum()),
        ),
        meses_con_3_semillas_favorables=(
            "semillas_favorables",
            lambda x: int((x >= 3).sum()),
        ),
    )
)

resumen_general = resumen_general.merge(
    criterios_por_mes,
    on="experimento",
    how="left",
)

condicion_aceptar = (
    resumen_general[
        "meses_con_delta_positivo"
    ].eq(2)
    &
    resumen_general[
        "meses_con_3_semillas_favorables"
    ].eq(2)
)

condicion_observar = (
    resumen_general[
        "delta_ganancia_media"
    ].gt(0)
)

resumen_general["decision"] = np.select(
    [
        condicion_aceptar,
        condicion_observar,
    ],
    [
        "ACEPTAR",
        "POSITIVO_PERO_INESTABLE",
    ],
    default="DESCARTAR",
)

resumen_general = resumen_general.sort_values(
    "delta_ganancia_media",
    ascending=False,
)

display(resumen_general)


# ------------------------------------------------------------
# 11. Guardar resultados
# ------------------------------------------------------------

resultados_features_controlados.to_csv(
    "resultados_features_finales_controlados.csv",
    index=False,
)

comparacion_features_controlados.to_csv(
    "comparacion_features_finales_controlados.csv",
    index=False,
)

resumen_por_mes.to_csv(
    "resumen_features_finales_por_mes.csv",
    index=False,
)

resumen_general.to_csv(
    "resumen_features_finales_general.csv",
    index=False,
)

print("\nEvaluación controlada finalizada.")

Variables disponibles para rangos: 30
Mínimo de los rangos: 6.093325371388181e-06
Máximo de los rangos: 1.0
Valores infinitos: 0
ACTUAL_B1_B4_B6_TEMP_B6 | features: 295 | agregadas: 0
ACTUAL_MAS_RANKS_MENSUALES | features: 325 | agregadas: 30
ACTUAL_MAS_B5 | features: 329 | agregadas: 34
ACTUAL_B1_B4_B6_TEMP_B6 | semilla 100129 | validación 202105 | ganancia 281,875,000 | AUC BAJA+2 0.89723 | iteraciones 603
ACTUAL_B1_B4_B6_TEMP_B6 | semilla 100129 | validación 202106 | ganancia 377,575,000 | AUC BAJA+2 0.90263 | iteraciones 833
ACTUAL_B1_B4_B6_TEMP_B6 | semilla 100183 | validación 202105 | ganancia 287,375,000 | AUC BAJA+2 0.89695 | iteraciones 758
ACTUAL_B1_B4_B6_TEMP_B6 | semilla 100183 | validación 202106 | ganancia 369,875,000 | AUC BAJA+2 0.90364 | iteraciones 953
ACTUAL_B1_B4_B6_TEMP_B6 | semilla 100151 | validación 202105 | ganancia 274,175,000 | AUC BAJA+2 0.89723 | iteraciones 738
ACTUAL_B1_B4_B6_TEMP_B6 | semilla 100151 | validación 202106 | ganancia 364,375,000 | AUC BAJA+2

,experimento,validacion,ganancia_referencia,ganancia_candidato,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,semillas_favorables,delta_auc_baja2_medio,iteraciones_mediana
0,ACTUAL_MAS_B5,202105,280335000.0,268675000.0,-11660000.0,-16500000.0,-5500000.0,0,0.000145,664.0
1,ACTUAL_MAS_B5,202106,368335000.0,375155000.0,6820000.0,-8800000.0,18700000.0,3,0.000483,816.0
2,ACTUAL_MAS_RANKS_MENSUALES,202105,280335000.0,275715000.0,-4620000.0,-8800000.0,-2200000.0,0,-0.000104,707.0
3,ACTUAL_MAS_RANKS_MENSUALES,202106,368335000.0,368775000.0,440000.0,-8800000.0,8800000.0,3,-0.001027,883.0


,experimento,ganancia_referencia,ganancia_candidato,delta_ganancia_media,delta_ganancia_minima,delta_ganancia_maxima,combinaciones_favorables,delta_auc_baja2_medio,meses_con_delta_positivo,meses_con_3_semillas_favorables,decision
1,ACTUAL_MAS_RANKS_MENSUALES,324335000.0,322245000.0,-2090000.0,-8800000.0,8800000.0,3,-0.000565,1,1,DESCARTAR
0,ACTUAL_MAS_B5,324335000.0,321915000.0,-2420000.0,-16500000.0,18700000.0,3,0.000314,1,1,DESCARTAR



Evaluación controlada finalizada.


In [79]:
# ============================================================
# COMPARACIÓN CONTROLADA DE ENSAMBLES POR SEMILLAS
#
# Features: 295 congeladas
# Target: BAJA+1 + BAJA+2 vs CONTINUA
# Ganancia real: solamente BAJA+2
# Corte: 8750
#
# Configuraciones:
# - Parámetros base
# - Trial 24: robusto
# - Trial 21: mayor ganancia media
# ============================================================

import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Configuración
# ------------------------------------------------------------

FEATURES_ENSEMBLE_CONTROL = list(
    features_temporales_b6_candidato
)

assert len(FEATURES_ENSEMBLE_CONTROL) == 295

SEMILLAS_ENSEMBLE_CONTROL = [
    100129,
    100183,
    100151,
    500041,
    900007,
]

CORTE_ENSEMBLE_CONTROL = 8750

FOLDS_ENSEMBLE_CONTROL = [
    {
        "entrenamiento": [202103],
        "validacion": 202105,
    },
    {
        "entrenamiento": [202103, 202104],
        "validacion": 202106,
    },
]


# ------------------------------------------------------------
# 2. Recuperar los parámetros exactos de Optuna
# ------------------------------------------------------------

NOMBRE_ESTUDIO_INDIVIDUAL = (
    "competencia_v4_bajas12_vs_continua_topn_v1"
)

estudio_individual_bajas12 = optuna.load_study(
    study_name=NOMBRE_ESTUDIO_INDIVIDUAL,
    storage=STORAGE_OPTUNA,
)


def obtener_parametros_trial(
    estudio,
    numero_trial,
):
    encontrados = [
        trial
        for trial in estudio.trials
        if (
            trial.number == numero_trial
            and trial.state
            == optuna.trial.TrialState.COMPLETE
        )
    ]

    if len(encontrados) != 1:
        raise ValueError(
            f"No se encontró completo el trial "
            f"{numero_trial}"
        )

    return dict(encontrados[0].params)


parametros_trial_24 = obtener_parametros_trial(
    estudio_individual_bajas12,
    24,
)

parametros_trial_21 = obtener_parametros_trial(
    estudio_individual_bajas12,
    21,
)


# Sólo conservar los hiperparámetros del modelo base.
CLAVES_HIPERPARAMETROS = [
    "max_depth",
    "num_leaves",
    "learning_rate",
    "min_data_in_leaf",
    "feature_fraction",
    "bagging_fraction",
    "lambda_l1",
    "lambda_l2",
    "min_gain_to_split",
    "max_bin",
]

parametros_base_control = {
    clave: PARAMETROS_BASE[clave]
    for clave in CLAVES_HIPERPARAMETROS
}


CONFIGURACIONES_ENSEMBLE = {
    "PARAMETROS_BASE":
        parametros_base_control,

    "TRIAL_24_ROBUSTO":
        parametros_trial_24,

    "TRIAL_21_MAYOR_MEDIA":
        parametros_trial_21,
}


for nombre, parametros in (
    CONFIGURACIONES_ENSEMBLE.items()
):
    print(f"\n{nombre}")
    print(parametros)


# ------------------------------------------------------------
# 3. Evaluar una configuración como ensamble
#
# Dataset nuevo para cada semilla:
# evita el error de data_random_seed.
# ------------------------------------------------------------

def evaluar_configuracion_ensemble(
    nombre_configuracion,
    hiperparametros,
):
    resultados_ensemble = []
    resultados_individuales = []

    for fold in FOLDS_ENSEMBLE_CONTROL:

        mes_validacion = fold["validacion"]

        mascara_train = (
            base_modelo["foto_mes"]
            .isin(fold["entrenamiento"])
        )

        mascara_validacion = (
            base_modelo["foto_mes"]
            .eq(mes_validacion)
        )

        X_train = base_modelo.loc[
            mascara_train,
            FEATURES_ENSEMBLE_CONTROL,
        ]

        X_validacion = base_modelo.loc[
            mascara_validacion,
            FEATURES_ENSEMBLE_CONTROL,
        ]

        clases_train = base_modelo.loc[
            mascara_train,
            "clase_ternaria",
        ]

        clases_validacion = base_modelo.loc[
            mascara_validacion,
            "clase_ternaria",
        ]

        # Target utilizado para entrenar.
        y_train = construir_target_optuna(
            clases_train,
            "bajas12",
        )

        y_validacion = construir_target_optuna(
            clases_validacion,
            "bajas12",
        )

        # Target real de la competencia.
        y_baja2_validacion = (
            clases_validacion
            .eq("BAJA+2")
            .astype("int8")
            .to_numpy()
        )

        ganancias_validacion = np.where(
            y_baja2_validacion == 1,
            GANANCIA_ACIERTO,
            -COSTO_ESTIMULO,
        )

        probabilidades_semillas = []
        ganancias_semillas = []
        iteraciones_semillas = []

        for semilla in SEMILLAS_ENSEMBLE_CONTROL:

            parametros_modelo = {
                "objective": "binary",
                "metric": "auc",
                "boosting_type": "gbdt",
                "boost_from_average": True,
                "feature_pre_filter": False,
                "first_metric_only": True,
                "verbosity": -1,
                "bagging_freq": 1,

                **hiperparametros,

                "seed": semilla,
                "feature_fraction_seed": semilla,
                "bagging_seed": semilla,
                "data_random_seed": semilla,
            }

            # IMPORTANTE:
            # estos Dataset se crean dentro del ciclo.
            dataset_train = lgb.Dataset(
                X_train,
                label=y_train,
                free_raw_data=False,
            )

            dataset_validacion = lgb.Dataset(
                X_validacion,
                label=y_validacion,
                reference=dataset_train,
                free_raw_data=False,
            )

            modelo = lgb.train(
                params=parametros_modelo,
                train_set=dataset_train,
                valid_sets=[dataset_validacion],
                valid_names=["validacion"],
                num_boost_round=4000,
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        first_metric_only=True,
                        verbose=False,
                    ),
                    lgb.log_evaluation(period=0),
                ],
            )

            probabilidades = modelo.predict(
                X_validacion,
                num_iteration=modelo.best_iteration,
            )

            ganancia_individual = (
                calcular_ganancia_top_n(
                    probabilidades=probabilidades,
                    ganancias_reales=ganancias_validacion,
                    corte=CORTE_ENSEMBLE_CONTROL,
                )
            )

            auc_baja2_individual = roc_auc_score(
                y_baja2_validacion,
                probabilidades,
            )

            probabilidades_semillas.append(
                probabilidades
            )

            ganancias_semillas.append(
                ganancia_individual
            )

            iteraciones_semillas.append(
                modelo.best_iteration
            )

            resultados_individuales.append({
                "configuracion":
                    nombre_configuracion,
                "semilla":
                    semilla,
                "validacion":
                    mes_validacion,
                "ganancia":
                    ganancia_individual,
                "auc_baja2":
                    auc_baja2_individual,
                "iteraciones":
                    modelo.best_iteration,
            })

            print(
                nombre_configuracion,
                "| semilla",
                semilla,
                "| validación",
                mes_validacion,
                "| ganancia individual",
                f"{ganancia_individual:,.0f}",
            )

            del modelo
            del dataset_train
            del dataset_validacion
            del probabilidades

            gc.collect()

        # ----------------------------------------------------
        # Promedio de probabilidades de las cinco semillas
        # ----------------------------------------------------

        probabilidades_ensemble = np.mean(
            np.vstack(probabilidades_semillas),
            axis=0,
        )

        ganancia_ensemble = calcular_ganancia_top_n(
            probabilidades=probabilidades_ensemble,
            ganancias_reales=ganancias_validacion,
            corte=CORTE_ENSEMBLE_CONTROL,
        )

        auc_target_ensemble = roc_auc_score(
            y_validacion,
            probabilidades_ensemble,
        )

        auc_baja2_ensemble = roc_auc_score(
            y_baja2_validacion,
            probabilidades_ensemble,
        )

        ganancia_individual_media = float(
            np.mean(ganancias_semillas)
        )

        mejora_por_ensamblar = (
            ganancia_ensemble
            - ganancia_individual_media
        )

        resultados_ensemble.append({
            "configuracion":
                nombre_configuracion,
            "validacion":
                mes_validacion,
            "ganancia_ensemble":
                ganancia_ensemble,
            "ganancia_individual_media":
                ganancia_individual_media,
            "ganancia_individual_minima":
                np.min(ganancias_semillas),
            "ganancia_individual_maxima":
                np.max(ganancias_semillas),
            "desvio_ganancia_individual":
                np.std(
                    ganancias_semillas,
                    ddof=1,
                ),
            "mejora_por_ensamblar":
                mejora_por_ensamblar,
            "auc_target_ensemble":
                auc_target_ensemble,
            "auc_baja2_ensemble":
                auc_baja2_ensemble,
            "iteraciones_mediana":
                np.median(iteraciones_semillas),
            "cantidad_semillas":
                len(SEMILLAS_ENSEMBLE_CONTROL),
            "corte":
                CORTE_ENSEMBLE_CONTROL,
        })

        print(
            "\n",
            nombre_configuracion,
            "| validación",
            mes_validacion,
            "| GANANCIA ENSEMBLE",
            f"{ganancia_ensemble:,.0f}",
            "| mejora sobre media individual",
            f"{mejora_por_ensamblar:,.0f}",
            "\n",
        )

        del X_train
        del X_validacion
        del probabilidades_semillas
        del probabilidades_ensemble

        gc.collect()

    return (
        resultados_ensemble,
        resultados_individuales,
    )


# ------------------------------------------------------------
# 4. Ejecutar las tres configuraciones
# ------------------------------------------------------------

filas_ensemble = []
filas_individuales = []

for nombre, parametros in (
    CONFIGURACIONES_ENSEMBLE.items()
):
    resultado_ensemble, resultado_individual = (
        evaluar_configuracion_ensemble(
            nombre_configuracion=nombre,
            hiperparametros=parametros,
        )
    )

    filas_ensemble.extend(resultado_ensemble)
    filas_individuales.extend(resultado_individual)


resultados_ensemble_control = pd.DataFrame(
    filas_ensemble
)

resultados_individuales_control = pd.DataFrame(
    filas_individuales
)


# ------------------------------------------------------------
# 5. Comparación contra el ensamble base
# ------------------------------------------------------------

referencia_base_ensemble = (
    resultados_ensemble_control
    .loc[
        resultados_ensemble_control[
            "configuracion"
        ].eq("PARAMETROS_BASE"),
        [
            "validacion",
            "ganancia_ensemble",
            "auc_baja2_ensemble",
        ],
    ]
    .rename(columns={
        "ganancia_ensemble":
            "ganancia_ensemble_base",
        "auc_baja2_ensemble":
            "auc_baja2_ensemble_base",
    })
)

comparacion_ensemble = (
    resultados_ensemble_control
    .merge(
        referencia_base_ensemble,
        on="validacion",
        how="left",
        validate="many_to_one",
    )
)

comparacion_ensemble[
    "delta_ganancia_vs_base"
] = (
    comparacion_ensemble["ganancia_ensemble"]
    - comparacion_ensemble[
        "ganancia_ensemble_base"
    ]
)

comparacion_ensemble[
    "delta_auc_baja2_vs_base"
] = (
    comparacion_ensemble["auc_baja2_ensemble"]
    - comparacion_ensemble[
        "auc_baja2_ensemble_base"
    ]
)


# ------------------------------------------------------------
# 6. Resúmenes
# ------------------------------------------------------------

resumen_ensemble = (
    comparacion_ensemble
    .groupby(
        "configuracion",
        as_index=False,
    )
    .agg(
        ganancia_ensemble_media=(
            "ganancia_ensemble",
            "mean",
        ),
        ganancia_ensemble_minima=(
            "ganancia_ensemble",
            "min",
        ),
        ganancia_individual_media=(
            "ganancia_individual_media",
            "mean",
        ),
        mejora_media_por_ensamblar=(
            "mejora_por_ensamblar",
            "mean",
        ),
        delta_ganancia_media_vs_base=(
            "delta_ganancia_vs_base",
            "mean",
        ),
        delta_ganancia_minima_vs_base=(
            "delta_ganancia_vs_base",
            "min",
        ),
        meses_favorables_vs_base=(
            "delta_ganancia_vs_base",
            lambda x: int((x > 0).sum()),
        ),
        auc_baja2_medio=(
            "auc_baja2_ensemble",
            "mean",
        ),
        delta_auc_baja2_medio_vs_base=(
            "delta_auc_baja2_vs_base",
            "mean",
        ),
        iteraciones_mediana=(
            "iteraciones_mediana",
            "median",
        ),
    )
    .sort_values(
        "ganancia_ensemble_media",
        ascending=False,
    )
)

print("\nRESULTADO POR MES")
display(
    comparacion_ensemble[
        [
            "configuracion",
            "validacion",
            "ganancia_individual_media",
            "ganancia_ensemble",
            "mejora_por_ensamblar",
            "delta_ganancia_vs_base",
            "auc_baja2_ensemble",
            "iteraciones_mediana",
        ]
    ].sort_values(
        [
            "validacion",
            "ganancia_ensemble",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

print("\nRESUMEN GENERAL")
display(resumen_ensemble)


# ------------------------------------------------------------
# 7. Guardar resultados
# ------------------------------------------------------------

resultados_ensemble_control.to_csv(
    "resultados_ensemble_control.csv",
    index=False,
)

resultados_individuales_control.to_csv(
    "resultados_individuales_ensemble_control.csv",
    index=False,
)

comparacion_ensemble.to_csv(
    "comparacion_ensemble_control.csv",
    index=False,
)

resumen_ensemble.to_csv(
    "resumen_ensemble_control.csv",
    index=False,
)

print("\nComparación de ensambles finalizada.")


PARAMETROS_BASE
{'max_depth': 8, 'num_leaves': 48, 'learning_rate': 0.011968361180129191, 'min_data_in_leaf': 405, 'feature_fraction': 0.30152141273199784, 'bagging_fraction': 0.7006936995574339, 'lambda_l1': 8.320444953703719e-05, 'lambda_l2': 0.0006696837335733366, 'min_gain_to_split': 1.5614099380945257, 'max_bin': 255}

TRIAL_24_ROBUSTO
{'max_depth': 5, 'num_leaves': 35, 'learning_rate': 0.04996867190287919, 'min_data_in_leaf': 1429, 'feature_fraction': 0.37844310349997656, 'bagging_fraction': 0.7759644671385546, 'lambda_l1': 3.758943946409675e-05, 'lambda_l2': 0.00023597942178946686, 'min_gain_to_split': 1.1399893154758223, 'max_bin': 255}

TRIAL_21_MAYOR_MEDIA
{'max_depth': 6, 'num_leaves': 53, 'learning_rate': 0.04975434210507218, 'min_data_in_leaf': 559, 'feature_fraction': 0.36621066744103115, 'bagging_fraction': 0.7218824933069297, 'lambda_l1': 5.3944530082667373e-05, 'lambda_l2': 0.0005517330594691839, 'min_gain_to_split': 2.1846969881737714, 'max_bin': 255}
PARAMETROS_BASE

,configuracion,validacion,ganancia_individual_media,ganancia_ensemble,mejora_por_ensamblar,delta_ganancia_vs_base,auc_baja2_ensemble,iteraciones_mediana
4,TRIAL_21_MAYOR_MEDIA,202105,269115000.0,281875000.0,12760000.0,1100000.0,0.896799,180.0
0,PARAMETROS_BASE,202105,280335000.0,280775000.0,440000.0,0.0,0.897663,738.0
2,TRIAL_24_ROBUSTO,202105,269555000.0,277475000.0,7920000.0,-3300000.0,0.896207,236.0
3,TRIAL_24_ROBUSTO,202106,372075000.0,374275000.0,2200000.0,6600000.0,0.903174,442.0
5,TRIAL_21_MAYOR_MEDIA,202106,368995000.0,369875000.0,880000.0,2200000.0,0.903361,248.0
1,PARAMETROS_BASE,202106,368335000.0,367675000.0,-660000.0,0.0,0.903585,953.0



RESUMEN GENERAL


,configuracion,ganancia_ensemble_media,ganancia_ensemble_minima,ganancia_individual_media,mejora_media_por_ensamblar,delta_ganancia_media_vs_base,delta_ganancia_minima_vs_base,meses_favorables_vs_base,auc_baja2_medio,delta_auc_baja2_medio_vs_base,iteraciones_mediana
1,TRIAL_21_MAYOR_MEDIA,325875000.0,281875000.0,319055000.0,6820000.0,1650000.0,1100000.0,2,0.900080,-0.000544,214.0
2,TRIAL_24_ROBUSTO,325875000.0,277475000.0,320815000.0,5060000.0,1650000.0,-3300000.0,1,0.899691,-0.000933,339.0
0,PARAMETROS_BASE,324225000.0,280775000.0,324335000.0,-110000.0,0.0,0.0,0,0.900624,0.000000,845.5



Comparación de ensambles finalizada.


In [80]:
# ============================================================
# OPTUNA SOBRE ENSAMBLE DE 3 SEMILLAS
#
# Features: 295 congeladas
# Target: BAJA+1 + BAJA+2 vs CONTINUA
# Corte: 8750
# Early stopping: AUC
#
# Objetivo robusto:
# prioriza la mejora del peor mes y luego la mejora promedio.
# ============================================================

import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Configuración
# ------------------------------------------------------------

FEATURES_OPTUNA_ENSEMBLE = list(
    features_temporales_b6_candidato
)

assert len(FEATURES_OPTUNA_ENSEMBLE) == 295

SEMILLAS_OPTUNA_ENSEMBLE = [
    100129,
    100183,
    100151,
]

FOLDS_OPTUNA_ENSEMBLE = [
    {
        "entrenamiento": [202103],
        "validacion": 202105,
    },
    {
        "entrenamiento": [202103, 202104],
        "validacion": 202106,
    },
]

CORTE_OPTUNA_ENSEMBLE = 8750
TRIALS_TOTALES_OPTUNA_ENSEMBLE = 15

NOMBRE_ESTUDIO_OPTUNA_ENSEMBLE = (
    "competencia_v4_"
    "bajas12_ensemble3_robusto_auc_v3"
)


# ------------------------------------------------------------
# 2. Recuperar Trial 21 exacto
# ------------------------------------------------------------

estudio_individual_bajas12 = optuna.load_study(
    study_name=(
        "competencia_v4_"
        "bajas12_vs_continua_topn_v1"
    ),
    storage=STORAGE_OPTUNA,
)


def obtener_parametros_trial(
    estudio,
    numero_trial,
):
    encontrados = [
        trial
        for trial in estudio.trials
        if (
            trial.number == numero_trial
            and trial.state
            == optuna.trial.TrialState.COMPLETE
        )
    ]

    if len(encontrados) != 1:
        raise ValueError(
            f"No se encontró completo el trial "
            f"{numero_trial}"
        )

    return dict(encontrados[0].params)


PARAMETROS_TRIAL_21 = obtener_parametros_trial(
    estudio_individual_bajas12,
    21,
)

print("Parámetros exactos del Trial 21:")
print(PARAMETROS_TRIAL_21)


# ------------------------------------------------------------
# 3. Hiperparámetros base comparables
# ------------------------------------------------------------

CLAVES_HIPERPARAMETROS = [
    "max_depth",
    "num_leaves",
    "learning_rate",
    "min_data_in_leaf",
    "feature_fraction",
    "bagging_fraction",
    "lambda_l1",
    "lambda_l2",
    "min_gain_to_split",
    "max_bin",
]

PARAMETROS_BASE_ENSEMBLE = {
    clave: PARAMETROS_BASE[clave]
    for clave in CLAVES_HIPERPARAMETROS
}


# ------------------------------------------------------------
# 4. Función general para evaluar un ensamble
#
# IMPORTANTE:
# se crea un Dataset nuevo dentro de cada semilla.
# ------------------------------------------------------------

def evaluar_ensemble_tres_semillas(
    hiperparametros,
    mostrar_progreso=False,
):
    resultados_folds = []

    for fold in FOLDS_OPTUNA_ENSEMBLE:

        mes_validacion = fold["validacion"]

        mascara_train = (
            base_modelo["foto_mes"]
            .isin(fold["entrenamiento"])
        )

        mascara_validacion = (
            base_modelo["foto_mes"]
            .eq(mes_validacion)
        )

        X_train = base_modelo.loc[
            mascara_train,
            FEATURES_OPTUNA_ENSEMBLE,
        ]

        X_validacion = base_modelo.loc[
            mascara_validacion,
            FEATURES_OPTUNA_ENSEMBLE,
        ]

        clases_train = base_modelo.loc[
            mascara_train,
            "clase_ternaria",
        ]

        clases_validacion = base_modelo.loc[
            mascara_validacion,
            "clase_ternaria",
        ]

        # Target de entrenamiento.
        y_train = construir_target_optuna(
            clases_train,
            "bajas12",
        )

        y_validacion = construir_target_optuna(
            clases_validacion,
            "bajas12",
        )

        # Target real de negocio.
        y_baja2_validacion = (
            clases_validacion
            .eq("BAJA+2")
            .astype("int8")
            .to_numpy()
        )

        ganancias_validacion = np.where(
            y_baja2_validacion == 1,
            GANANCIA_ACIERTO,
            -COSTO_ESTIMULO,
        )

        probabilidades_semillas = []
        ganancias_individuales = []
        iteraciones_semillas = []

        for semilla in SEMILLAS_OPTUNA_ENSEMBLE:

            parametros_modelo = {
                "objective": "binary",
                "metric": "auc",
                "boosting_type": "gbdt",
                "boost_from_average": True,
                "feature_pre_filter": False,
                "first_metric_only": True,
                "verbosity": -1,
                "bagging_freq": 1,

                **hiperparametros,

                "seed": semilla,
                "feature_fraction_seed": semilla,
                "bagging_seed": semilla,
                "data_random_seed": semilla,
            }

            # Dataset independiente para esta semilla.
            dataset_train = lgb.Dataset(
                X_train,
                label=y_train,
                free_raw_data=False,
            )

            dataset_validacion = lgb.Dataset(
                X_validacion,
                label=y_validacion,
                reference=dataset_train,
                free_raw_data=False,
            )

            modelo = lgb.train(
                params=parametros_modelo,
                train_set=dataset_train,
                valid_sets=[dataset_validacion],
                valid_names=["validacion"],
                num_boost_round=4000,
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        first_metric_only=True,
                        verbose=False,
                    ),
                    lgb.log_evaluation(period=0),
                ],
            )

            probabilidades = modelo.predict(
                X_validacion,
                num_iteration=modelo.best_iteration,
            )

            ganancia_individual = (
                calcular_ganancia_top_n(
                    probabilidades=probabilidades,
                    ganancias_reales=ganancias_validacion,
                    corte=CORTE_OPTUNA_ENSEMBLE,
                )
            )

            probabilidades_semillas.append(
                probabilidades
            )

            ganancias_individuales.append(
                ganancia_individual
            )

            iteraciones_semillas.append(
                modelo.best_iteration
            )

            if mostrar_progreso:
                print(
                    "semilla",
                    semilla,
                    "| validación",
                    mes_validacion,
                    "| ganancia",
                    f"{ganancia_individual:,.0f}",
                )

            del modelo
            del dataset_train
            del dataset_validacion
            del probabilidades

            gc.collect()

        # Promedio de probabilidades.
        probabilidades_ensemble = np.mean(
            np.vstack(probabilidades_semillas),
            axis=0,
        )

        ganancia_ensemble = calcular_ganancia_top_n(
            probabilidades=probabilidades_ensemble,
            ganancias_reales=ganancias_validacion,
            corte=CORTE_OPTUNA_ENSEMBLE,
        )

        auc_target_ensemble = roc_auc_score(
            y_validacion,
            probabilidades_ensemble,
        )

        auc_baja2_ensemble = roc_auc_score(
            y_baja2_validacion,
            probabilidades_ensemble,
        )

        ganancia_individual_media = float(
            np.mean(ganancias_individuales)
        )

        resultados_folds.append({
            "validacion":
                mes_validacion,
            "ganancia_ensemble":
                float(ganancia_ensemble),
            "ganancia_individual_media":
                ganancia_individual_media,
            "mejora_por_ensamblar":
                float(
                    ganancia_ensemble
                    - ganancia_individual_media
                ),
            "dispersion_semillas":
                float(
                    np.std(
                        ganancias_individuales,
                        ddof=1,
                    )
                ),
            "auc_target_ensemble":
                float(auc_target_ensemble),
            "auc_baja2_ensemble":
                float(auc_baja2_ensemble),
            "iteraciones_mediana":
                float(
                    np.median(iteraciones_semillas)
                ),
        })

        del X_train
        del X_validacion
        del probabilidades_semillas
        del probabilidades_ensemble

        gc.collect()

    return pd.DataFrame(resultados_folds)


# ------------------------------------------------------------
# 5. Referencia base con las mismas tres semillas
# ------------------------------------------------------------

print("\nCalculando referencia base de tres semillas...")

referencia_base_ensemble3 = (
    evaluar_ensemble_tres_semillas(
        hiperparametros=PARAMETROS_BASE_ENSEMBLE,
        mostrar_progreso=True,
    )
)

GANANCIA_BASE_ENSEMBLE3 = (
    referencia_base_ensemble3
    .set_index("validacion")[
        "ganancia_ensemble"
    ]
    .to_dict()
)

assert set(GANANCIA_BASE_ENSEMBLE3) == {
    202105,
    202106,
}

print("\nReferencia base:")
display(referencia_base_ensemble3)


# ------------------------------------------------------------
# 6. Objetivo Optuna
#
# score = peor delta mensual
#         + 10% del delta promedio
#
# De este modo, una gran mejora en junio no compensa
# fácilmente una pérdida en mayo.
# ------------------------------------------------------------

def objetivo_optuna_ensemble_robusto(trial):

    hiperparametros = {
        "max_depth":
            trial.suggest_int(
                "max_depth",
                4,
                9,
            ),

        "num_leaves":
            trial.suggest_int(
                "num_leaves",
                20,
                100,
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.02,
                0.08,
                log=True,
            ),

        "min_data_in_leaf":
            trial.suggest_int(
                "min_data_in_leaf",
                300,
                1600,
                log=True,
            ),

        "feature_fraction":
            trial.suggest_float(
                "feature_fraction",
                0.25,
                0.50,
            ),

        "bagging_fraction":
            trial.suggest_float(
                "bagging_fraction",
                0.60,
                0.90,
            ),

        "lambda_l1":
            trial.suggest_float(
                "lambda_l1",
                1e-6,
                0.10,
                log=True,
            ),

        "lambda_l2":
            trial.suggest_float(
                "lambda_l2",
                1e-6,
                0.10,
                log=True,
            ),

        "min_gain_to_split":
            trial.suggest_float(
                "min_gain_to_split",
                0.20,
                3.50,
            ),

        "max_bin":
            trial.suggest_categorical(
                "max_bin",
                [127, 255, 511],
            ),
    }

    resultados_trial = (
        evaluar_ensemble_tres_semillas(
            hiperparametros=hiperparametros,
            mostrar_progreso=False,
        )
    )

    resultados_trial["ganancia_base"] = (
        resultados_trial["validacion"]
        .map(GANANCIA_BASE_ENSEMBLE3)
    )

    resultados_trial["delta_ganancia"] = (
        resultados_trial["ganancia_ensemble"]
        - resultados_trial["ganancia_base"]
    )

    deltas = (
        resultados_trial["delta_ganancia"]
        .to_numpy(dtype="float64")
    )

    delta_medio = float(np.mean(deltas))
    delta_minimo = float(np.min(deltas))

    # El peor mes es el componente principal.
    score_robusto = float(
        delta_minimo
        + 0.10 * delta_medio
    )

    ganancia_media = float(
        resultados_trial[
            "ganancia_ensemble"
        ].mean()
    )

    # Guardar detalle de cada mes.
    for fila in resultados_trial.itertuples():

        mes = int(fila.validacion)

        trial.set_user_attr(
            f"ganancia_ensemble_{mes}",
            float(fila.ganancia_ensemble),
        )

        trial.set_user_attr(
            f"delta_ganancia_{mes}",
            float(fila.delta_ganancia),
        )

        trial.set_user_attr(
            f"ganancia_individual_media_{mes}",
            float(fila.ganancia_individual_media),
        )

        trial.set_user_attr(
            f"mejora_por_ensamblar_{mes}",
            float(fila.mejora_por_ensamblar),
        )

        trial.set_user_attr(
            f"auc_baja2_{mes}",
            float(fila.auc_baja2_ensemble),
        )

        trial.set_user_attr(
            f"auc_target_{mes}",
            float(fila.auc_target_ensemble),
        )

        trial.set_user_attr(
            f"iteraciones_mediana_{mes}",
            float(fila.iteraciones_mediana),
        )

    trial.set_user_attr(
        "ganancia_media",
        ganancia_media,
    )

    trial.set_user_attr(
        "delta_ganancia_medio",
        delta_medio,
    )

    trial.set_user_attr(
        "delta_ganancia_minimo",
        delta_minimo,
    )

    trial.set_user_attr(
        "score_robusto",
        score_robusto,
    )

    trial.set_user_attr(
        "semillas",
        list(SEMILLAS_OPTUNA_ENSEMBLE),
    )

    trial.set_user_attr(
        "corte",
        CORTE_OPTUNA_ENSEMBLE,
    )

    return score_robusto


# ------------------------------------------------------------
# 7. Crear estudio NUEVO
# ------------------------------------------------------------

sampler_ensemble_robusto = (
    optuna.samplers.TPESampler(
        seed=100130,
        n_startup_trials=5,
    )
)

estudio_ensemble_robusto = optuna.create_study(
    direction="maximize",
    study_name=NOMBRE_ESTUDIO_OPTUNA_ENSEMBLE,
    storage=STORAGE_OPTUNA,
    load_if_exists=True,
    sampler=sampler_ensemble_robusto,
)


# Trial 21 como primer candidato conocido.
if len(estudio_ensemble_robusto.trials) == 0:
    estudio_ensemble_robusto.enqueue_trial(
        PARAMETROS_TRIAL_21
    )


trials_completos = sum(
    trial.state
    == optuna.trial.TrialState.COMPLETE
    for trial in estudio_ensemble_robusto.trials
)

trials_por_ejecutar = max(
    0,
    (
        TRIALS_TOTALES_OPTUNA_ENSEMBLE
        - trials_completos
    ),
)

print(
    "\nEstudio:",
    NOMBRE_ESTUDIO_OPTUNA_ENSEMBLE,
)

print(
    "Trials completos:",
    trials_completos,
)

print(
    "Trials por ejecutar:",
    trials_por_ejecutar,
)


# ------------------------------------------------------------
# 8. Ejecutar optimización
# ------------------------------------------------------------

if trials_por_ejecutar > 0:

    estudio_ensemble_robusto.optimize(
        objetivo_optuna_ensemble_robusto,
        n_trials=trials_por_ejecutar,
        n_jobs=1,
        gc_after_trial=True,
        show_progress_bar=True,
    )


# ------------------------------------------------------------
# 9. Construir ranking
# ------------------------------------------------------------

filas_ranking = []

for trial in estudio_ensemble_robusto.trials:

    if (
        trial.state
        != optuna.trial.TrialState.COMPLETE
    ):
        continue

    fila = {
        "trial":
            trial.number,
        "score_robusto":
            trial.value,
        "ganancia_media":
            trial.user_attrs.get(
                "ganancia_media"
            ),
        "ganancia_202105":
            trial.user_attrs.get(
                "ganancia_ensemble_202105"
            ),
        "ganancia_202106":
            trial.user_attrs.get(
                "ganancia_ensemble_202106"
            ),
        "delta_202105":
            trial.user_attrs.get(
                "delta_ganancia_202105"
            ),
        "delta_202106":
            trial.user_attrs.get(
                "delta_ganancia_202106"
            ),
        "delta_medio":
            trial.user_attrs.get(
                "delta_ganancia_medio"
            ),
        "delta_minimo":
            trial.user_attrs.get(
                "delta_ganancia_minimo"
            ),
        "auc_baja2_202105":
            trial.user_attrs.get(
                "auc_baja2_202105"
            ),
        "auc_baja2_202106":
            trial.user_attrs.get(
                "auc_baja2_202106"
            ),
        "iteraciones_202105":
            trial.user_attrs.get(
                "iteraciones_mediana_202105"
            ),
        "iteraciones_202106":
            trial.user_attrs.get(
                "iteraciones_mediana_202106"
            ),
    }

    fila.update(trial.params)
    filas_ranking.append(fila)


ranking_optuna_ensemble = (
    pd.DataFrame(filas_ranking)
    .sort_values(
        "score_robusto",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\nMEJORES TRIALS DEL ENSAMBLE")
display(
    ranking_optuna_ensemble.head(10)
)


# Tres candidatos para confirmar con cinco semillas.
TRIALS_FINALISTAS_ENSEMBLE = (
    ranking_optuna_ensemble
    .head(3)["trial"]
    .astype(int)
    .tolist()
)

print(
    "Trials para confirmar con 5 semillas:",
    TRIALS_FINALISTAS_ENSEMBLE,
)


# ------------------------------------------------------------
# 10. Guardar ranking
# ------------------------------------------------------------

ranking_optuna_ensemble.to_csv(
    "ranking_optuna_ensemble_robusto_v3.csv",
    index=False,
)

referencia_base_ensemble3.to_csv(
    "referencia_base_ensemble3.csv",
    index=False,
)

print("\nOptimización del ensamble finalizada.")

Parámetros exactos del Trial 21:
{'max_depth': 6, 'num_leaves': 53, 'learning_rate': 0.04975434210507218, 'min_data_in_leaf': 559, 'feature_fraction': 0.36621066744103115, 'bagging_fraction': 0.7218824933069297, 'lambda_l1': 5.3944530082667373e-05, 'lambda_l2': 0.0005517330594691839, 'min_gain_to_split': 2.1846969881737714, 'max_bin': 255}

Calculando referencia base de tres semillas...
semilla 100129 | validación 202105 | ganancia 281,875,000
semilla 100183 | validación 202105 | ganancia 287,375,000
semilla 100151 | validación 202105 | ganancia 274,175,000
semilla 100129 | validación 202106 | ganancia 377,575,000
semilla 100183 | validación 202106 | ganancia 369,875,000
semilla 100151 | validación 202106 | ganancia 364,375,000

Referencia base:


,validacion,ganancia_ensemble,ganancia_individual_media,mejora_por_ensamblar,dispersion_semillas,auc_target_ensemble,auc_baja2_ensemble,iteraciones_mediana
0,202105,279675000.0,2.811417e+08,-1.466667e+06,6.630485e+06,0.924532,0.897655,738.0
1,202106,365475000.0,3.706083e+08,-5.133333e+06,6.630485e+06,0.918847,0.903622,861.0


[I 2026-10-10 22:51:51,943] A new study created in RDB with name: competencia_v4_bajas12_ensemble3_robusto_auc_v3



Estudio: competencia_v4_bajas12_ensemble3_robusto_auc_v3
Trials completos: 0
Trials por ejecutar: 15


Best trial: 0. Best value: -5.445e+06:   7%|▋         | 1/15 [01:35<22:21, 95.83s/it]

...

[I 2026-10-10 23:18:36,541] Trial 13 finished with value: 3685000.0 and parameters: {'max_depth': 8, 'num_leaves': 65, 'learning_rate': 0.03925081600763232, 'min_data_in_leaf': 423, 'feature_fraction': 0.29228346021081286, 'bagging_fraction': 0.750556632801188, 'lambda_l1': 0.0004943033756098246, 'lambda_l2': 1.245870074453164e-05, 'min_gain_to_split': 2.8445859841628707, 'max_bin': 127}. Best is trial 13 with value: 3685000.0.


Best trial: 13. Best value: 3.685e+06: 100%|██████████| 15/15 [28:23<00:00, 113.56s/it]

[I 2026-10-10 23:20:15,332] Trial 14 finished with value: 385000.0 and parameters: {'max_depth': 9, 'num_leaves': 66, 'learning_rate': 0.04081180101561441, 'min_data_in_leaf': 417, 'feature_fraction': 0.2768083734942281, 'bagging_fraction': 0.7713781949686263, 'lambda_l1': 6.481224388007292e-06, 'lambda_l2': 1.4397738934390947e-05, 'min_gain_to_split': 2.865142861169944, 'max_bin': 511}. Best is trial 13 with value: 3685000.0.

MEJORES TRIALS DEL ENSAMBLE


,trial,score_robusto,ganancia_media,ganancia_202105,ganancia_202106,delta_202105,delta_202106,delta_medio,delta_minimo,auc_baja2_202105,...,max_depth,num_leaves,learning_rate,min_data_in_leaf,feature_fraction,bagging_fraction,lambda_l1,lambda_l2,min_gain_to_split,max_bin
0,13,3685000.0,326425000.0,284075000.0,368775000.0,4400000.0,3300000.0,3850000.0,3300000.0,0.897150,...,8,65,0.039251,423,0.292283,0.750557,0.000494,0.000012,2.844586,127
1,3,1265000.0,324225000.0,281875000.0,366575000.0,2200000.0,1100000.0,1650000.0,1100000.0,0.897107,...,9,43,0.054932,1468,0.273826,0.770894,0.000002,0.000093,2.039981,255
2,14,385000.0,326425000.0,279675000.0,373175000.0,0.0,7700000.0,3850000.0,0.0,0.896636,...,9,66,0.040812,417,0.276808,0.771378,0.000006,0.000014,2.865143,511
3,2,-550000.0,328075000.0,278575000.0,377575000.0,-1100000.0,12100000.0,5500000.0,-1100000.0,0.897161,...,8,54,0.028087,307,0.351431,0.690639,0.001062,0.000001,1.438304,127
4,10,-825000.0,325325000.0,278575000.0,372075000.0,-1100000.0,6600000.0,2750000.0,-1100000.0,0.895482,...,5,98,0.062310,965,0.325750,0.807726,0.008374,0.000081,1.188549,127
5,12,-2090000.0,323675000.0,277475000.0,369875000.0,-2200000.0,4400000.0,1100000.0,-2200000.0,0.897380,...,8,37,0.032042,737,0.364566,0.668844,0.000029,0.000001,1.743894,127
6,8,-2530000.0,330275000.0,276375000.0,384175000.0,-3300000.0,18700000.0,7700000.0,-3300000.0,0.896920,...,9,76,0.048995,991,0.302954,0.603187,0.000675,0.000022,1.105741,255
7,4,-3630000.0,319275000.0,276375000.0,362175000.0,-3300000.0,-3300000.0,-3300000.0,-3300000.0,0.897458,...,9,28,0.050989,915,0.474346,0.850268,0.000009,0.000831,2.259823,255
8,6,-4675000.0,319825000.0,275275000.0,364375000.0,-4400000.0,-1100000.0,-2750000.0,-4400000.0,0.896097,...,4,21,0.020604,321,0.250481,0.796674,0.099378,0.000082,3.499724,255
9,1,-5280000.0,324775000.0,274175000.0,375375000.0,-5500000.0,9900000.0,2200000.0,-5500000.0,0.894618,...,4,76,0.030401,1524,0.407558,0.670644,0.053354,0.000004,2.521064,127


Trials para confirmar con 5 semillas: [13, 3, 14]

Optimización del ensamble finalizada.


In [81]:
# ============================================================
# DECISIÓN FINAL DEL MODELO
#
# Confirma trials 13, 3 y 14 con cinco semillas.
# Los compara contra:
# - parámetros base
# - Trial 21 anterior
#
# Al terminar deja definidos:
# - GANADOR_FINAL
# - PARAMETROS_MODELO_FINAL
# - iteraciones_finales_por_semilla
# ============================================================

import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from sklearn.metrics import roc_auc_score


FEATURES_MODELO_FINAL = list(
    features_temporales_b6_candidato
)

assert len(FEATURES_MODELO_FINAL) == 295

SEMILLAS_FINALES = [
    100129,
    100183,
    100151,
    500041,
    900007,
]

FOLDS_CONFIRMACION = [
    {
        "entrenamiento": [202103],
        "validacion": 202105,
    },
    {
        "entrenamiento": [202103, 202104],
        "validacion": 202106,
    },
]

CORTE_FINAL = 8750


# ------------------------------------------------------------
# Recuperar parámetros exactos
# ------------------------------------------------------------

estudio_ensemble_robusto = optuna.load_study(
    study_name=(
        "competencia_v4_"
        "bajas12_ensemble3_robusto_auc_v3"
    ),
    storage=STORAGE_OPTUNA,
)

estudio_individual_bajas12 = optuna.load_study(
    study_name=(
        "competencia_v4_"
        "bajas12_vs_continua_topn_v1"
    ),
    storage=STORAGE_OPTUNA,
)


def parametros_de_trial(
    estudio,
    numero_trial,
):
    coincidencias = [
        trial
        for trial in estudio.trials
        if (
            trial.number == numero_trial
            and trial.state
            == optuna.trial.TrialState.COMPLETE
        )
    ]

    if len(coincidencias) != 1:
        raise ValueError(
            f"No se encontró completo el trial "
            f"{numero_trial}"
        )

    return dict(coincidencias[0].params)


parametros_trial_13 = parametros_de_trial(
    estudio_ensemble_robusto,
    13,
)

parametros_trial_3 = parametros_de_trial(
    estudio_ensemble_robusto,
    3,
)

parametros_trial_14 = parametros_de_trial(
    estudio_ensemble_robusto,
    14,
)

parametros_trial_21 = parametros_de_trial(
    estudio_individual_bajas12,
    21,
)

claves_hiperparametros = [
    "max_depth",
    "num_leaves",
    "learning_rate",
    "min_data_in_leaf",
    "feature_fraction",
    "bagging_fraction",
    "lambda_l1",
    "lambda_l2",
    "min_gain_to_split",
    "max_bin",
]

parametros_base_final = {
    clave: PARAMETROS_BASE[clave]
    for clave in claves_hiperparametros
}


PARAMETROS_CANDIDATOS = {
    "PARAMETROS_BASE":
        parametros_base_final,

    "TRIAL_21_MAYOR_MEDIA":
        parametros_trial_21,

    "OPTUNA_ENSEMBLE_TRIAL_13":
        parametros_trial_13,

    "OPTUNA_ENSEMBLE_TRIAL_3":
        parametros_trial_3,

    "OPTUNA_ENSEMBLE_TRIAL_14":
        parametros_trial_14,
}


# ------------------------------------------------------------
# Evaluar un finalista con las cinco semillas
# ------------------------------------------------------------

def confirmar_finalista(
    nombre_configuracion,
    hiperparametros,
):
    filas_ensemble = []
    filas_individuales = []

    for fold in FOLDS_CONFIRMACION:

        mes_validacion = fold["validacion"]

        mascara_train = (
            base_modelo["foto_mes"]
            .isin(fold["entrenamiento"])
        )

        mascara_validacion = (
            base_modelo["foto_mes"]
            .eq(mes_validacion)
        )

        X_train = base_modelo.loc[
            mascara_train,
            FEATURES_MODELO_FINAL,
        ]

        X_validacion = base_modelo.loc[
            mascara_validacion,
            FEATURES_MODELO_FINAL,
        ]

        clases_train = base_modelo.loc[
            mascara_train,
            "clase_ternaria",
        ]

        clases_validacion = base_modelo.loc[
            mascara_validacion,
            "clase_ternaria",
        ]

        y_train = construir_target_optuna(
            clases_train,
            "bajas12",
        )

        y_validacion = construir_target_optuna(
            clases_validacion,
            "bajas12",
        )

        y_baja2 = (
            clases_validacion
            .eq("BAJA+2")
            .astype("int8")
            .to_numpy()
        )

        ganancias_reales = np.where(
            y_baja2 == 1,
            GANANCIA_ACIERTO,
            -COSTO_ESTIMULO,
        )

        predicciones_semillas = []
        ganancias_semillas = []
        iteraciones_semillas = []

        for semilla in SEMILLAS_FINALES:

            parametros = {
                "objective": "binary",
                "metric": "auc",
                "boosting_type": "gbdt",
                "boost_from_average": True,
                "feature_pre_filter": False,
                "first_metric_only": True,
                "verbosity": -1,
                "bagging_freq": 1,

                **hiperparametros,

                "seed": semilla,
                "feature_fraction_seed": semilla,
                "bagging_seed": semilla,
                "data_random_seed": semilla,
            }

            # Dataset nuevo para cada semilla.
            dataset_train = lgb.Dataset(
                X_train,
                label=y_train,
                free_raw_data=False,
            )

            dataset_validacion = lgb.Dataset(
                X_validacion,
                label=y_validacion,
                reference=dataset_train,
                free_raw_data=False,
            )

            modelo = lgb.train(
                params=parametros,
                train_set=dataset_train,
                valid_sets=[dataset_validacion],
                valid_names=["validacion"],
                num_boost_round=4000,
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        first_metric_only=True,
                        verbose=False,
                    ),
                    lgb.log_evaluation(period=0),
                ],
            )

            probabilidades = modelo.predict(
                X_validacion,
                num_iteration=modelo.best_iteration,
            )

            ganancia_individual = (
                calcular_ganancia_top_n(
                    probabilidades=probabilidades,
                    ganancias_reales=ganancias_reales,
                    corte=CORTE_FINAL,
                )
            )

            predicciones_semillas.append(
                probabilidades
            )

            ganancias_semillas.append(
                ganancia_individual
            )

            iteraciones_semillas.append(
                modelo.best_iteration
            )

            filas_individuales.append({
                "configuracion":
                    nombre_configuracion,
                "semilla":
                    semilla,
                "validacion":
                    mes_validacion,
                "ganancia":
                    ganancia_individual,
                "auc_baja2":
                    roc_auc_score(
                        y_baja2,
                        probabilidades,
                    ),
                "iteraciones":
                    modelo.best_iteration,
            })

            del modelo
            del dataset_train
            del dataset_validacion
            del probabilidades

            gc.collect()

        probabilidad_ensemble = np.mean(
            np.vstack(predicciones_semillas),
            axis=0,
        )

        ganancia_ensemble = (
            calcular_ganancia_top_n(
                probabilidades=probabilidad_ensemble,
                ganancias_reales=ganancias_reales,
                corte=CORTE_FINAL,
            )
        )

        ganancia_individual_media = float(
            np.mean(ganancias_semillas)
        )

        filas_ensemble.append({
            "configuracion":
                nombre_configuracion,
            "validacion":
                mes_validacion,
            "ganancia_ensemble":
                ganancia_ensemble,
            "ganancia_individual_media":
                ganancia_individual_media,
            "mejora_por_ensamblar":
                (
                    ganancia_ensemble
                    - ganancia_individual_media
                ),
            "auc_baja2_ensemble":
                roc_auc_score(
                    y_baja2,
                    probabilidad_ensemble,
                ),
            "iteraciones_mediana":
                np.median(
                    iteraciones_semillas
                ),
        })

        print(
            nombre_configuracion,
            "| validación",
            mes_validacion,
            "| ganancia ensemble",
            f"{ganancia_ensemble:,.0f}",
        )

        del X_train
        del X_validacion
        del predicciones_semillas
        del probabilidad_ensemble

        gc.collect()

    return filas_ensemble, filas_individuales


# ------------------------------------------------------------
# Confirmar solamente los tres finalistas nuevos
#
# Base y Trial 21 ya fueron evaluados con cinco semillas.
# No los entrenamos de nuevo.
# ------------------------------------------------------------

filas_confirmacion_ensemble = []
filas_confirmacion_individual = []

for nombre in [
    "OPTUNA_ENSEMBLE_TRIAL_13",
    "OPTUNA_ENSEMBLE_TRIAL_3",
    "OPTUNA_ENSEMBLE_TRIAL_14",
]:
    filas_e, filas_i = confirmar_finalista(
        nombre_configuracion=nombre,
        hiperparametros=(
            PARAMETROS_CANDIDATOS[nombre]
        ),
    )

    filas_confirmacion_ensemble.extend(
        filas_e
    )

    filas_confirmacion_individual.extend(
        filas_i
    )


confirmacion_ensemble_nuevos = pd.DataFrame(
    filas_confirmacion_ensemble
)

confirmacion_individual_nuevos = pd.DataFrame(
    filas_confirmacion_individual
)


# ------------------------------------------------------------
# Incorporar evaluaciones anteriores de Base y Trial 21
# ------------------------------------------------------------

configuraciones_anteriores = [
    "PARAMETROS_BASE",
    "TRIAL_21_MAYOR_MEDIA",
]

ensemble_anteriores = (
    resultados_ensemble_control
    .loc[
        resultados_ensemble_control[
            "configuracion"
        ].isin(configuraciones_anteriores),
        [
            "configuracion",
            "validacion",
            "ganancia_ensemble",
            "ganancia_individual_media",
            "mejora_por_ensamblar",
            "auc_baja2_ensemble",
            "iteraciones_mediana",
        ],
    ]
    .copy()
)

individuales_anteriores = (
    resultados_individuales_control
    .loc[
        resultados_individuales_control[
            "configuracion"
        ].isin(configuraciones_anteriores),
        [
            "configuracion",
            "semilla",
            "validacion",
            "ganancia",
            "auc_baja2",
            "iteraciones",
        ],
    ]
    .copy()
)


evaluacion_final_ensemble = pd.concat(
    [
        ensemble_anteriores,
        confirmacion_ensemble_nuevos,
    ],
    ignore_index=True,
)

evaluacion_final_individual = pd.concat(
    [
        individuales_anteriores,
        confirmacion_individual_nuevos,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Comparar contra parámetros base
# ------------------------------------------------------------

referencia_base = (
    evaluacion_final_ensemble
    .loc[
        evaluacion_final_ensemble[
            "configuracion"
        ].eq("PARAMETROS_BASE"),
        [
            "validacion",
            "ganancia_ensemble",
        ],
    ]
    .rename(columns={
        "ganancia_ensemble":
            "ganancia_base",
    })
)

evaluacion_final_ensemble = (
    evaluacion_final_ensemble
    .merge(
        referencia_base,
        on="validacion",
        how="left",
        validate="many_to_one",
    )
)

evaluacion_final_ensemble[
    "delta_vs_base"
] = (
    evaluacion_final_ensemble[
        "ganancia_ensemble"
    ]
    - evaluacion_final_ensemble[
        "ganancia_base"
    ]
)


resumen_decision_final = (
    evaluacion_final_ensemble
    .groupby(
        "configuracion",
        as_index=False,
    )
    .agg(
        ganancia_media=(
            "ganancia_ensemble",
            "mean",
        ),
        ganancia_minima=(
            "ganancia_ensemble",
            "min",
        ),
        delta_medio_vs_base=(
            "delta_vs_base",
            "mean",
        ),
        delta_minimo_vs_base=(
            "delta_vs_base",
            "min",
        ),
        meses_favorables=(
            "delta_vs_base",
            lambda x: int((x > 0).sum()),
        ),
        mejora_media_por_ensamblar=(
            "mejora_por_ensamblar",
            "mean",
        ),
        auc_baja2_medio=(
            "auc_baja2_ensemble",
            "mean",
        ),
    )
)

# Mismo criterio robusto usado por Optuna.
resumen_decision_final[
    "score_robusto"
] = (
    resumen_decision_final[
        "delta_minimo_vs_base"
    ]
    + 0.10
    * resumen_decision_final[
        "delta_medio_vs_base"
    ]
)

resumen_decision_final = (
    resumen_decision_final
    .sort_values(
        [
            "score_robusto",
            "ganancia_media",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

GANADOR_FINAL = (
    resumen_decision_final
    .iloc[0]["configuracion"]
)

PARAMETROS_MODELO_FINAL = dict(
    PARAMETROS_CANDIDATOS[
        GANADOR_FINAL
    ]
)


# ------------------------------------------------------------
# Iteraciones finales por semilla
# ------------------------------------------------------------

iteraciones_finales_por_semilla = (
    evaluacion_final_individual
    .loc[
        evaluacion_final_individual[
            "configuracion"
        ].eq(GANADOR_FINAL)
    ]
    .groupby("semilla")["iteraciones"]
    .median()
    .round()
    .astype(int)
    .to_dict()
)

assert set(
    iteraciones_finales_por_semilla
) == set(SEMILLAS_FINALES)


print("\nDECISIÓN FINAL")
display(resumen_decision_final)

print("Ganador:", GANADOR_FINAL)
print(
    "Iteraciones:",
    iteraciones_finales_por_semilla,
)

resumen_decision_final.to_csv(
    "decision_modelo_final_v4.csv",
    index=False,
)

evaluacion_final_ensemble.to_csv(
    "confirmacion_modelo_final_v4.csv",
    index=False,
)

OPTUNA_ENSEMBLE_TRIAL_13 | validación 202105 | ganancia ensemble 281,875,000
OPTUNA_ENSEMBLE_TRIAL_13 | validación 202106 | ganancia ensemble 370,975,000
OPTUNA_ENSEMBLE_TRIAL_3 | validación 202105 | ganancia ensemble 285,175,000
OPTUNA_ENSEMBLE_TRIAL_3 | validación 202106 | ganancia ensemble 368,775,000
OPTUNA_ENSEMBLE_TRIAL_14 | validación 202105 | ganancia ensemble 278,575,000
OPTUNA_ENSEMBLE_TRIAL_14 | validación 202106 | ganancia ensemble 368,775,000

DECISIÓN FINAL


,configuracion,ganancia_media,ganancia_minima,delta_medio_vs_base,delta_minimo_vs_base,meses_favorables,mejora_media_por_ensamblar,auc_baja2_medio,score_robusto
0,OPTUNA_ENSEMBLE_TRIAL_3,326975000.0,285175000.0,2750000.0,1100000.0,2,5280000.0,0.900251,1375000.0
1,OPTUNA_ENSEMBLE_TRIAL_13,326425000.0,281875000.0,2200000.0,1100000.0,2,4510000.0,0.900014,1320000.0
2,TRIAL_21_MAYOR_MEDIA,325875000.0,281875000.0,1650000.0,1100000.0,2,6820000.0,0.900080,1265000.0
3,PARAMETROS_BASE,324225000.0,280775000.0,0.0,0.0,0,-110000.0,0.900624,0.0
4,OPTUNA_ENSEMBLE_TRIAL_14,323675000.0,278575000.0,-550000.0,-2200000.0,1,5720000.0,0.899994,-2255000.0


Ganador: OPTUNA_ENSEMBLE_TRIAL_3
Iteraciones: {100129: 175, 100151: 155, 100183: 156, 500041: 126, 900007: 183}


In [83]:

# ------------------------------------------------------------
# 1. Verificar decisión anterior
# ------------------------------------------------------------

assert (
    GANADOR_FINAL
    == "OPTUNA_ENSEMBLE_TRIAL_3"
), (
    f"El ganador esperado era "
    f"OPTUNA_ENSEMBLE_TRIAL_3, "
    f"pero GANADOR_FINAL vale {GANADOR_FINAL}"
)

FEATURES_MODELO_FINAL = list(
    features_temporales_b6_candidato
)

VARIABLES_TEMPORALES_B6_FINAL = list(
    variables_temporales_b6
)

SEMILLAS_MODELO_FINAL = [
    100129,
    100183,
    100151,
    500041,
    900007,
]

CORTE_MODELO_FINAL = 8750

assert len(FEATURES_MODELO_FINAL) == 295

assert set(
    iteraciones_finales_por_semilla
) == set(SEMILLAS_MODELO_FINAL)

print("Modelo:", GANADOR_FINAL)
print("Features:", len(FEATURES_MODELO_FINAL))
print(
    "Iteraciones por semilla:",
    iteraciones_finales_por_semilla,
)


# ------------------------------------------------------------
# 2. Liberar objetos grandes de validación
# ------------------------------------------------------------

objetos_para_liberar = [
    "base_modelo",
    "confirmacion_ensemble_nuevos",
    "confirmacion_individual_nuevos",
    "evaluacion_final_ensemble",
    "evaluacion_final_individual",
]

for nombre_objeto in objetos_para_liberar:
    globals().pop(nombre_objeto, None)

gc.collect()


# ------------------------------------------------------------
# 3. Leer los seis meses
# ------------------------------------------------------------

base_final_6m = %sql SELECT * FROM competencia_07_recurrentes_rentabilidad_challenger ORDER BY numero_de_cliente, foto_mes

base_final_6m = (
    base_final_6m
    .sort_values(
        [
            "numero_de_cliente",
            "foto_mes",
        ]
    )
    .reset_index(drop=True)
)

assert len(base_final_6m) == 983_061

print(
    "Base completa:",
    base_final_6m.shape,
)


# ------------------------------------------------------------
# 4. Reconstruir los temporales B6 en la base completa
#
# Es necesario hacerlo con los seis meses para que agosto
# utilice correctamente julio como mes anterior.
# ------------------------------------------------------------

historia_valida_final = (
    base_final_6m[
        "tiene_historia_lag1"
    ].eq(1)
)

for variable in VARIABLES_TEMPORALES_B6_FINAL:

    assert variable in base_final_6m.columns, (
        f"No existe la variable {variable}"
    )

    lag = (
        base_final_6m
        .groupby(
            "numero_de_cliente",
            sort=False,
        )[variable]
        .shift(1)
        .where(historia_valida_final)
    )

    nombre_lag = (
        f"{variable}_lag1_extra"
    )

    nombre_delta = (
        f"{variable}_delta1_extra"
    )

    base_final_6m[nombre_lag] = lag

    base_final_6m[nombre_delta] = (
        base_final_6m[variable]
        - lag
    )


# ------------------------------------------------------------
# 5. Auditoría de las features finales
# ------------------------------------------------------------

features_faltantes = sorted(
    set(FEATURES_MODELO_FINAL)
    - set(base_final_6m.columns)
)

assert not features_faltantes, (
    f"Faltan features finales: "
    f"{features_faltantes}"
)

base_final_6m[
    FEATURES_MODELO_FINAL
] = (
    base_final_6m[
        FEATURES_MODELO_FINAL
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

columnas_no_numericas = (
    base_final_6m[
        FEATURES_MODELO_FINAL
    ]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

assert columnas_no_numericas == [], (
    f"Columnas no numéricas: "
    f"{columnas_no_numericas}"
)

cantidad_infinitos = np.isinf(
    base_final_6m[
        FEATURES_MODELO_FINAL
    ]
    .to_numpy(dtype="float64")
).sum()

assert cantidad_infinitos == 0

print(
    "Auditoría de features correcta."
)


# ------------------------------------------------------------
# 6. Preparar entrenamiento y agosto
# ------------------------------------------------------------

MESES_ENTRENAMIENTO_FINAL = [
    202103,
    202104,
    202105,
    202106,
]

MES_SCORING_FINAL = 202108

mascara_entrenamiento = (
    base_final_6m["foto_mes"]
    .isin(MESES_ENTRENAMIENTO_FINAL)
    & base_final_6m[
        "clase_ternaria"
    ].notna()
)

mascara_agosto = (
    base_final_6m["foto_mes"]
    .eq(MES_SCORING_FINAL)
)

X_entrenamiento_final = (
    base_final_6m.loc[
        mascara_entrenamiento,
        FEATURES_MODELO_FINAL,
    ]
    .copy()
)

# Target elegido:
# BAJA+1 y BAJA+2 son positivos.
y_entrenamiento_final = (
    base_final_6m.loc[
        mascara_entrenamiento,
        "clase_ternaria",
    ]
    .isin([
        "BAJA+1",
        "BAJA+2",
    ])
    .astype("int8")
    .to_numpy()
)

X_agosto = (
    base_final_6m.loc[
        mascara_agosto,
        FEATURES_MODELO_FINAL,
    ]
    .copy()
)

clientes_agosto = (
    base_final_6m.loc[
        mascara_agosto,
        "numero_de_cliente",
    ]
    .astype("int64")
    .to_numpy()
)

assert len(X_entrenamiento_final) == 654_066
assert len(X_agosto) == 164_647
assert len(clientes_agosto) == 164_647
assert len(np.unique(clientes_agosto)) == 164_647
assert y_entrenamiento_final.sum() > 0

print(
    "Filas de entrenamiento:",
    len(X_entrenamiento_final),
)

print(
    "Casos BAJA+1/BAJA+2:",
    int(y_entrenamiento_final.sum()),
)

print(
    "Clientes de agosto:",
    len(X_agosto),
)


# Liberar la tabla completa.
del base_final_6m
del mascara_entrenamiento
del mascara_agosto
gc.collect()


# ------------------------------------------------------------
# 7. Entrenar los cinco modelos finales
# ------------------------------------------------------------

directorio_modelos = Path(
    "modelos_competencia_v4_final"
)

directorio_modelos.mkdir(
    exist_ok=True
)

predicciones_agosto = {}

for semilla in SEMILLAS_MODELO_FINAL:

    iteraciones = int(
        iteraciones_finales_por_semilla[
            semilla
        ]
    )

    parametros_semilla = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "bagging_freq": 1,

        **PARAMETROS_MODELO_FINAL,

        "seed": semilla,
        "feature_fraction_seed": semilla,
        "bagging_seed": semilla,
        "data_random_seed": semilla,
    }

    # Dataset nuevo para cada semilla.
    dataset_entrenamiento = lgb.Dataset(
        X_entrenamiento_final,
        label=y_entrenamiento_final,
        free_raw_data=False,
    )

    modelo_final = lgb.train(
        params=parametros_semilla,
        train_set=dataset_entrenamiento,
        num_boost_round=iteraciones,
        callbacks=[
            lgb.log_evaluation(period=0)
        ],
    )

    probabilidades = modelo_final.predict(
        X_agosto,
        num_iteration=iteraciones,
    )

    predicciones_agosto[semilla] = (
        probabilidades
    )

    ruta_modelo = (
        directorio_modelos
        / (
            f"modelo_{GANADOR_FINAL}_"
            f"semilla_{semilla}.txt"
        )
    )

    modelo_final.save_model(
        str(ruta_modelo)
    )

    print(
        "Semilla",
        semilla,
        "| iteraciones",
        iteraciones,
        "| modelo guardado",
    )

    del modelo_final
    del dataset_entrenamiento
    del probabilidades

    gc.collect()


# ------------------------------------------------------------
# 8. Ensamble de probabilidades
# ------------------------------------------------------------

matriz_predicciones = np.vstack([
    predicciones_agosto[semilla]
    for semilla in SEMILLAS_MODELO_FINAL
])

probabilidad_ensemble = (
    matriz_predicciones.mean(axis=0)
)

desvio_entre_semillas = (
    matriz_predicciones.std(
        axis=0,
        ddof=1,
    )
)

assert np.isfinite(
    probabilidad_ensemble
).all()


# ------------------------------------------------------------
# 9. Seleccionar exactamente los 8750 primeros
#
# El cliente se utiliza como desempate determinístico.
# ------------------------------------------------------------

orden_probabilidad = np.lexsort((
    clientes_agosto,
    -probabilidad_ensemble,
))

indices_seleccionados = (
    orden_probabilidad[
        :CORTE_MODELO_FINAL
    ]
)

predicted = np.zeros(
    len(clientes_agosto),
    dtype="int8",
)

predicted[
    indices_seleccionados
] = 1

ranking = np.empty(
    len(clientes_agosto),
    dtype="int32",
)

ranking[
    orden_probabilidad
] = np.arange(
    1,
    len(clientes_agosto) + 1,
)


# ------------------------------------------------------------
# 10. Archivo para entregar
# ------------------------------------------------------------

entrega_final = pd.DataFrame({
    "numero_de_cliente":
        clientes_agosto,
    "Predicted":
        predicted,
})

entrega_final = (
    entrega_final
    .sort_values("numero_de_cliente")
    .reset_index(drop=True)
)

assert len(entrega_final) == 164_647
assert entrega_final[
    "numero_de_cliente"
].is_unique
assert int(
    entrega_final["Predicted"].sum()
) == CORTE_MODELO_FINAL

entrega_final.to_csv(
    "entrega_competencia_v4_final.csv",
    index=False,
)


# ------------------------------------------------------------
# 11. Archivo de detalle
# ------------------------------------------------------------

detalle_final = pd.DataFrame({
    "numero_de_cliente":
        clientes_agosto,
    "ranking":
        ranking,
    "probabilidad_ensemble":
        probabilidad_ensemble,
    "desvio_entre_semillas":
        desvio_entre_semillas,
    "Predicted":
        predicted,
})

for semilla in SEMILLAS_MODELO_FINAL:
    detalle_final[
        f"probabilidad_{semilla}"
    ] = predicciones_agosto[semilla]

detalle_final = (
    detalle_final
    .sort_values("ranking")
    .reset_index(drop=True)
)

detalle_final.to_csv(
    "detalle_competencia_v4_final.csv",
    index=False,
)


# ------------------------------------------------------------
# 12. Resultado final
# ------------------------------------------------------------

print("\nMODELO FINAL ENTRENADO")
print("Ganador:", GANADOR_FINAL)
print("Features:", len(FEATURES_MODELO_FINAL))
print("Semillas:", len(SEMILLAS_MODELO_FINAL))
print("Clientes de agosto:", len(entrega_final))
print(
    "Clientes seleccionados:",
    int(entrega_final["Predicted"].sum()),
)
print("Corte:", CORTE_MODELO_FINAL)
print(
    "Entrega:",
    "entrega_competencia_v4_final.csv",
)
print(
    "Detalle:",
    "detalle_competencia_v4_final.csv",
)
print(
    "Modelos:",
    directorio_modelos.resolve(),
)

Modelo: OPTUNA_ENSEMBLE_TRIAL_3
Features: 295
Iteraciones por semilla: {100129: 175, 100151: 155, 100183: 156, 500041: 126, 900007: 183}
Base completa: (983061, 352)
Auditoría de features correcta.
Filas de entrenamiento: 654066
Casos BAJA+1/BAJA+2: 8067
Clientes de agosto: 164647
Semilla 100129 | iteraciones 175 | modelo guardado
Semilla 100183 | iteraciones 156 | modelo guardado
Semilla 100151 | iteraciones 155 | modelo guardado
Semilla 500041 | iteraciones 126 | modelo guardado
Semilla 900007 | iteraciones 183 | modelo guardado

MODELO FINAL ENTRENADO
Ganador: OPTUNA_ENSEMBLE_TRIAL_3
Features: 295
Semillas: 5
Clientes de agosto: 164647
Clientes seleccionados: 8750
Corte: 8750
Entrega: entrega_competencia_v4_final.csv
Detalle: detalle_competencia_v4_final.csv
Modelos: /Users/manuelmarasco/Desktop/dmeyf402/modelos_competencia_v4_final
